In [1]:
#initiate spark
import pyspark
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
c = pyspark.SparkConf().setAppName("test_app").setMaster("local")
sc = pyspark.SparkContext(conf = c)
spark = SparkSession(sc)

In [2]:
from pyspark.sql.functions import *

In [3]:
import pandas as pd
import numpy as np
import datetime
import warnings 
warnings.filterwarnings('ignore')

In [4]:
from pyspark.sql.window import Window

In [5]:
# Loading Data Sets with PySpark

s_boxf = spark.read.csv("D:\Data Engineer books\CORE TOPICS\PYSPARK\PySpark Assignment\Assignment 3\Boxoffice_Fact.csv",
                        header = True, inferSchema = True, escape = '"')
s_dir = spark.read.csv("D:\Data Engineer books\CORE TOPICS\PYSPARK\PySpark Assignment\Assignment 3\Director_dim.csv",
                        header = True, inferSchema = True, escape = '"')
s_gen = spark.read.csv("D:\Data Engineer books\CORE TOPICS\PYSPARK\PySpark Assignment\Assignment 3\Genere_dim.csv",
                        header = True, inferSchema = True, escape = '"')
s_lan = spark.read.csv("D:\Data Engineer books\CORE TOPICS\PYSPARK\PySpark Assignment\Assignment 3\Language_dim.csv",
                        header = True, inferSchema = True, escape = '"')

In [6]:
# Loading Data Sets with Pandas

p_boxf = pd.read_csv("D:\Data Engineer books\CORE TOPICS\PYSPARK\PySpark Assignment\Assignment 3\Boxoffice_Fact.csv")
p_dir = pd.read_csv("D:\Data Engineer books\CORE TOPICS\PYSPARK\PySpark Assignment\Assignment 3\Director_dim.csv")
p_gen = pd.read_csv("D:\Data Engineer books\CORE TOPICS\PYSPARK\PySpark Assignment\Assignment 3\Genere_dim.csv")
p_lan = pd.read_csv("D:\Data Engineer books\CORE TOPICS\PYSPARK\PySpark Assignment\Assignment 3\Language_dim.csv")

In [7]:
# Checking Sample Data With Pandas
p_boxf.head(5)

,FilmID,Title,Release Date,DirectorID,Lead Actor/Actress,LanguageID,Industry,GenreID,Budget in Crores,First Day Collection Worldwide in Crores,Worldwide Collection in Crores,Overseas Collection in Crores,India Gross Collection in Crores,Verdict,IMDb Rating,Runtime (mins),OTT Platform
0,34,Sanju,29-Jun-18,101,Ranbir Kapoor,501,Bollywood,623,100,54.4,588.5,150,438.5,Blockbuster,7.6,155,Netflix
1,36,Simmba,28-Dec-18,105,Ranveer Singh,501,Bollywood,605,125,20.2,390,95,295,Blockbuster,5.5,158,ZEE5
2,225,Janatha Garage,01-Sep-16,140,N.T. Rama Rao Jr.,503,Kollywood,605,50,45,140,15,125,Blockbuster,7.2,162,Disney+ Hotstar
3,13,Kaththi,22-Oct-14,107,Joseph Vijay,503,Kollywood,606,70,25,117.5,29,88.5,Blockbuster,8.1,166,Amazon Prime Video
4,31,Maanikya,01-May-14,130,V. Ravichandran,507,Sandalwood,606,15,3.5,43,0,43,Blockbuster,6.5,165,ZEE5


In [8]:
# Checking Sample Data With PySpark
s_boxf.show(5,False)

+------+--------------+------------+----------+------------------+----------+----------+-------+----------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+-----------+-----------+--------------+------------------+
|FilmID|Title         |Release Date|DirectorID|Lead Actor/Actress|LanguageID|Industry  |GenreID|Budget in Crores|First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|Verdict    |IMDb Rating|Runtime (mins)|OTT Platform      |
+------+--------------+------------+----------+------------------+----------+----------+-------+----------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+-----------+-----------+--------------+------------------+
|34    |Sanju         |29-Jun-18   |101       |Ranbir Kapoor     |501    

In [9]:
# Checking Volume of Data With Pandas
p_boxf.shape

(604, 17)

In [10]:
# Checking Volume of Data With PySprak
s_boxf.count(),len(s_boxf.columns)

(604, 17)

In [11]:
# checking null Values with Pandas
p_boxf.isnull().sum()

FilmID                                       0
Title                                        0
Release Date                                 0
DirectorID                                   0
Lead Actor/Actress                           0
LanguageID                                   0
Industry                                     0
GenreID                                      0
Budget in Crores                             0
First Day Collection Worldwide  in Crores    0
Worldwide Collection  in Crores              0
Overseas Collection in Crores                9
India Gross Collection in Crores             0
Verdict                                      0
IMDb Rating                                  0
Runtime (mins)                               0
OTT Platform                                 0
dtype: int64

In [12]:
# Writing Function for checking missing values
def checking_missing_values(df,list_col):
    missing_values = {}
    for i in list_col:
        a = df.filter(col(i).isNull()).count()
        missing_values[i] = a
    return missing_values

In [13]:
# Checking Missing Values With PySpark
checking_missing_values(s_boxf.cache(),s_boxf.columns)

{'FilmID': 0,
 'Title': 0,
 'Release Date': 0,
 'DirectorID': 0,
 'Lead Actor/Actress': 0,
 'LanguageID': 0,
 'Industry': 0,
 'GenreID': 0,
 'Budget in Crores': 0,
 'First Day Collection Worldwide  in Crores': 0,
 'Worldwide Collection  in Crores': 0,
 'Overseas Collection in Crores': 9,
 'India Gross Collection in Crores': 0,
 'Verdict': 0,
 'IMDb Rating': 0,
 'Runtime (mins)': 0,
 'OTT Platform': 0}

In [14]:
# Checking data types with Pandas
p_boxf.dtypes

FilmID                                         int64
Title                                         object
Release Date                                  object
DirectorID                                     int64
Lead Actor/Actress                            object
LanguageID                                     int64
Industry                                      object
GenreID                                        int64
Budget in Crores                              object
First Day Collection Worldwide  in Crores     object
Worldwide Collection  in Crores               object
Overseas Collection in Crores                 object
India Gross Collection in Crores              object
Verdict                                       object
IMDb Rating                                  float64
Runtime (mins)                                 int64
OTT Platform                                  object
dtype: object

In [15]:
# Checking data Type with PySpak
s_boxf.printSchema()

root
 |-- FilmID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Release Date: string (nullable = true)
 |-- DirectorID: integer (nullable = true)
 |-- Lead Actor/Actress: string (nullable = true)
 |-- LanguageID: integer (nullable = true)
 |-- Industry: string (nullable = true)
 |-- GenreID: integer (nullable = true)
 |-- Budget in Crores: string (nullable = true)
 |-- First Day Collection Worldwide  in Crores: string (nullable = true)
 |-- Worldwide Collection  in Crores: string (nullable = true)
 |-- Overseas Collection in Crores: string (nullable = true)
 |-- India Gross Collection in Crores: string (nullable = true)
 |-- Verdict: string (nullable = true)
 |-- IMDb Rating: double (nullable = true)
 |-- Runtime (mins): integer (nullable = true)
 |-- OTT Platform: string (nullable = true)



### Data Cleaning With Pandas

#### Changing the Data Type with Pandas

In [16]:
# checking how many null values are present in the data frame with Pandas
p_boxf["Budget in Crores"].str.isnumeric().sum()

np.int64(587)

In [17]:
# Replacing the values in the data frame
p_boxf["Budget in Crores"] = p_boxf["Budget in Crores"].str.replace(":","",regex = True).astype(float)

In [18]:
p_boxf.dtypes

FilmID                                         int64
Title                                         object
Release Date                                  object
DirectorID                                     int64
Lead Actor/Actress                            object
LanguageID                                     int64
Industry                                      object
GenreID                                        int64
Budget in Crores                             float64
First Day Collection Worldwide  in Crores     object
Worldwide Collection  in Crores               object
Overseas Collection in Crores                 object
India Gross Collection in Crores              object
Verdict                                       object
IMDb Rating                                  float64
Runtime (mins)                                 int64
OTT Platform                                  object
dtype: object

In [19]:
p_boxf["First Day Collection Worldwide  in Crores"] = p_boxf["First Day Collection Worldwide  in Crores"]\
    .str.replace("[: ]","",regex = True).astype(float)

In [20]:
p_boxf.dtypes

FilmID                                         int64
Title                                         object
Release Date                                  object
DirectorID                                     int64
Lead Actor/Actress                            object
LanguageID                                     int64
Industry                                      object
GenreID                                        int64
Budget in Crores                             float64
First Day Collection Worldwide  in Crores    float64
Worldwide Collection  in Crores               object
Overseas Collection in Crores                 object
India Gross Collection in Crores              object
Verdict                                       object
IMDb Rating                                  float64
Runtime (mins)                                 int64
OTT Platform                                  object
dtype: object

In [21]:
p_boxf["Worldwide Collection  in Crores"] = p_boxf["Worldwide Collection  in Crores"]\
    .str.replace("[: ]","",regex = True).astype(float)
p_boxf["Overseas Collection in Crores"]=p_boxf["Overseas Collection in Crores"]\
    .str.replace("[: ]","",regex = True).astype(float)
p_boxf["India Gross Collection in Crores"]=p_boxf["India Gross Collection in Crores"]\
    .str.replace("[: ]","",regex=True).astype(float)

In [22]:
p_boxf.dtypes

FilmID                                         int64
Title                                         object
Release Date                                  object
DirectorID                                     int64
Lead Actor/Actress                            object
LanguageID                                     int64
Industry                                      object
GenreID                                        int64
Budget in Crores                             float64
First Day Collection Worldwide  in Crores    float64
Worldwide Collection  in Crores              float64
Overseas Collection in Crores                float64
India Gross Collection in Crores             float64
Verdict                                       object
IMDb Rating                                  float64
Runtime (mins)                                 int64
OTT Platform                                  object
dtype: object

In [23]:
list(p_boxf["Verdict"].unique())

['Blockbuster',
 'All Time Blockbuster',
 'Below Average',
 ': Disaster',
 ': Blockbuster',
 ': All Time Blockbuster',
 'Super Hit',
 ': Flop',
 'Above Average',
 'Flop',
 ': Hit',
 ':  Super Hit',
 ': Super Hit',
 ': Average',
 'Disaster',
 'Average',
 'Hit',
 ': Below Average',
 ': Above Average',
 ' Below Average',
 'Super Hitt']

In [24]:
p_boxf["Verdict"]= p_boxf["Verdict"].str.replace("[: ]","",regex = True)

In [25]:
list(p_boxf["Verdict"].unique())

['Blockbuster',
 'AllTimeBlockbuster',
 'BelowAverage',
 'Disaster',
 'SuperHit',
 'Flop',
 'AboveAverage',
 'Hit',
 'Average',
 'SuperHitt']

In [26]:
p_boxf["Verdict"]= p_boxf["Verdict"].replace("SuperHitt","SuperHit")

In [27]:
list(p_boxf["Verdict"].unique())

['Blockbuster',
 'AllTimeBlockbuster',
 'BelowAverage',
 'Disaster',
 'SuperHit',
 'Flop',
 'AboveAverage',
 'Hit',
 'Average']

In [28]:
list(p_boxf["OTT Platform"].unique())

['Netflix',
 'ZEE5',
 'Disney+ Hotstar',
 'Amazon Prime Video',
 'Aha  ',
 'Google Play Movies',
 'Amazon Prime Video ',
 'Disney+ Hotstar ',
 'Netflix ',
 'Sun NXT',
 'ZEE5  ',
 'Sun NXT  ',
 'Amazon Prime Video  ',
 'SonyLIV  ',
 'Netflix  ',
 'Voot  ',
 'Disney+ Hotstar  ',
 'aha  ',
 'ZEE5 ',
 'SonyLIV',
 'ZEE5      ',
 'JioCinema',
 'Aha']

In [29]:
p_boxf["OTT Platform"] = p_boxf["OTT Platform"].str.strip()

In [30]:
list(p_boxf["OTT Platform"].unique())

['Netflix',
 'ZEE5',
 'Disney+ Hotstar',
 'Amazon Prime Video',
 'Aha',
 'Google Play Movies',
 'Sun NXT',
 'SonyLIV',
 'Voot',
 'aha',
 'JioCinema']

In [31]:
p_boxf["OTT Platform"] = p_boxf["OTT Platform"].replace("aha","Aha")

In [32]:
list(p_boxf["OTT Platform"].unique())

['Netflix',
 'ZEE5',
 'Disney+ Hotstar',
 'Amazon Prime Video',
 'Aha',
 'Google Play Movies',
 'Sun NXT',
 'SonyLIV',
 'Voot',
 'JioCinema']

In [33]:
# checking the LanguageID columns
p_boxf["LanguageID"].unique()

array([501, 503, 507, 505, 509, 511, 515, 517, 513, 519])

In [34]:
# checking the language table how many languge in avlilbe in data frame
p_lan.head(10)

,Language,LanguageID
0,Hindi,501
1,Tamil,503
2,Telugu,505
3,Kannada,507
4,Malayalam,509


In [35]:
# grouping languageID with the Industry to get the wrong matching languageID
p_boxf[["LanguageID","Industry"]].groupby("LanguageID").head(20)

,LanguageID,Industry
0,501,Bollywood
1,501,Bollywood
2,503,Kollywood
3,503,Kollywood
4,507,Sandalwood
...,...,...
241,511,Tollywood
242,513,Mollywood
253,517,Kollywood
255,519,Sandalwood


In [36]:
# 511 --> 505
# 513 --> 509
# 515 --> 501
# 517 --> 503
# 519 --> 507

In [37]:
# replacing the languageID with the representative industry
p_boxf["LanguageID"] = p_boxf["LanguageID"].replace({511:505,513:509,515:501,517:503,519:507})

In [38]:
# checking the unique values in languageID
p_boxf["LanguageID"].unique()

array([501, 503, 507, 505, 509])

In [39]:
p_boxf.dtypes

FilmID                                         int64
Title                                         object
Release Date                                  object
DirectorID                                     int64
Lead Actor/Actress                            object
LanguageID                                     int64
Industry                                      object
GenreID                                        int64
Budget in Crores                             float64
First Day Collection Worldwide  in Crores    float64
Worldwide Collection  in Crores              float64
Overseas Collection in Crores                float64
India Gross Collection in Crores             float64
Verdict                                       object
IMDb Rating                                  float64
Runtime (mins)                                 int64
OTT Platform                                  object
dtype: object

In [40]:
# changing the realse Date  data type
p_boxf["Release Date"] = pd.to_datetime(p_boxf["Release Date"])

In [41]:
p_boxf.dtypes

FilmID                                                int64
Title                                                object
Release Date                                 datetime64[ns]
DirectorID                                            int64
Lead Actor/Actress                                   object
LanguageID                                            int64
Industry                                             object
GenreID                                               int64
Budget in Crores                                    float64
First Day Collection Worldwide  in Crores           float64
Worldwide Collection  in Crores                     float64
Overseas Collection in Crores                       float64
India Gross Collection in Crores                    float64
Verdict                                              object
IMDb Rating                                         float64
Runtime (mins)                                        int64
OTT Platform                            

### Data Cleaning With PySpark

#### Changing Data Types with PySpark

In [42]:
s_boxf.printSchema()

root
 |-- FilmID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Release Date: string (nullable = true)
 |-- DirectorID: integer (nullable = true)
 |-- Lead Actor/Actress: string (nullable = true)
 |-- LanguageID: integer (nullable = true)
 |-- Industry: string (nullable = true)
 |-- GenreID: integer (nullable = true)
 |-- Budget in Crores: string (nullable = true)
 |-- First Day Collection Worldwide  in Crores: string (nullable = true)
 |-- Worldwide Collection  in Crores: string (nullable = true)
 |-- Overseas Collection in Crores: string (nullable = true)
 |-- India Gross Collection in Crores: string (nullable = true)
 |-- Verdict: string (nullable = true)
 |-- IMDb Rating: double (nullable = true)
 |-- Runtime (mins): integer (nullable = true)
 |-- OTT Platform: string (nullable = true)



In [43]:
cols = ["Budget in Crores",\
     "First Day Collection Worldwide  in Crores",\
     "Worldwide Collection  in Crores",\
     "Overseas Collection in Crores",\
     "India Gross Collection in Crores"]
for c in cols:
    s_boxf = s_boxf.withColumn(c,regexp_replace(col(c),r"[: ]","").cast("double"))

In [44]:
s_boxf.printSchema()

root
 |-- FilmID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Release Date: string (nullable = true)
 |-- DirectorID: integer (nullable = true)
 |-- Lead Actor/Actress: string (nullable = true)
 |-- LanguageID: integer (nullable = true)
 |-- Industry: string (nullable = true)
 |-- GenreID: integer (nullable = true)
 |-- Budget in Crores: double (nullable = true)
 |-- First Day Collection Worldwide  in Crores: double (nullable = true)
 |-- Worldwide Collection  in Crores: double (nullable = true)
 |-- Overseas Collection in Crores: double (nullable = true)
 |-- India Gross Collection in Crores: double (nullable = true)
 |-- Verdict: string (nullable = true)
 |-- IMDb Rating: double (nullable = true)
 |-- Runtime (mins): integer (nullable = true)
 |-- OTT Platform: string (nullable = true)



In [45]:
s_boxf.select("Verdict").distinct().show()

+--------------------+
|             Verdict|
+--------------------+
|          Super Hitt|
|        :  Super Hit|
|          : Disaster|
|              : Flop|
|       Above Average|
|             Average|
|           : Average|
|       Below Average|
|     : Above Average|
|     : Below Average|
|       Below Average|
|            Disaster|
|               : Hit|
|All Time Blockbuster|
|       : Blockbuster|
|           Super Hit|
|                Flop|
|: All Time Blockb...|
|         Blockbuster|
|         : Super Hit|
+--------------------+
only showing top 20 rows


In [46]:
cols = ["Verdict","OTT Platform"]
for c in cols:
    s_boxf = s_boxf.withColumn(c,initcap(trim(regexp_replace(col(c),":",""))))

In [47]:
s_boxf.select("Verdict").distinct().show()

+--------------------+
|             Verdict|
+--------------------+
|          Super Hitt|
|       Above Average|
|             Average|
|       Below Average|
|            Disaster|
|All Time Blockbuster|
|           Super Hit|
|                Flop|
|         Blockbuster|
|                 Hit|
+--------------------+



In [48]:
s_boxf = s_boxf.withColumn("Verdict",when(col("Verdict")=="Super Hitt","Super Hit").otherwise(col("Verdict")))

In [49]:
s_boxf.select("Verdict").distinct().show()

+--------------------+
|             Verdict|
+--------------------+
|       Above Average|
|             Average|
|       Below Average|
|            Disaster|
|All Time Blockbuster|
|           Super Hit|
|                Flop|
|         Blockbuster|
|                 Hit|
+--------------------+



In [50]:
s_boxf.select("OTT Platform").distinct().show()

+------------------+
|      OTT Platform|
+------------------+
|           Sun Nxt|
|           Sonyliv|
|              Voot|
|Google Play Movies|
|         Jiocinema|
|              Zee5|
|Amazon Prime Video|
|   Disney+ Hotstar|
|               Aha|
|           Netflix|
+------------------+



In [51]:
s_boxf.select("LanguageID").distinct().show()

+----------+
|LanguageID|
+----------+
|       513|
|       501|
|       519|
|       507|
|       511|
|       505|
|       503|
|       509|
|       515|
|       517|
+----------+



In [52]:
s_boxf = s_boxf.replace({511:505,513:509,515:501,517:503,519:507},subset=["LanguageID"])

In [53]:
s_boxf.select("LanguageID").distinct().show()

+----------+
|LanguageID|
+----------+
|       501|
|       507|
|       505|
|       503|
|       509|
+----------+



In [54]:
s_boxf.printSchema()

root
 |-- FilmID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Release Date: string (nullable = true)
 |-- DirectorID: integer (nullable = true)
 |-- Lead Actor/Actress: string (nullable = true)
 |-- LanguageID: integer (nullable = true)
 |-- Industry: string (nullable = true)
 |-- GenreID: integer (nullable = true)
 |-- Budget in Crores: double (nullable = true)
 |-- First Day Collection Worldwide  in Crores: double (nullable = true)
 |-- Worldwide Collection  in Crores: double (nullable = true)
 |-- Overseas Collection in Crores: double (nullable = true)
 |-- India Gross Collection in Crores: double (nullable = true)
 |-- Verdict: string (nullable = true)
 |-- IMDb Rating: double (nullable = true)
 |-- Runtime (mins): integer (nullable = true)
 |-- OTT Platform: string (nullable = true)



In [55]:
# Converting PySpark datasets to Spark SQL datasets 
s_boxf.createOrReplaceTempView("boxf_tb")
s_dir.createOrReplaceTempView("dir_tb")
s_gen.createOrReplaceTempView("gen_tb")
s_lan.createOrReplaceTempView("lan_tb")

#### 1 Write a query to get Total films released ? 

In [56]:
# With Pandas
p_boxf["Title"].nunique()

600

In [57]:
# With PySpark
s_boxf.select("Title").distinct().count()

600

In [58]:
# With Spark SQL
spark.sql("select count(Title) as Total_Films from boxf_tb").show()

+-----------+
|Total_Films|
+-----------+
|        604|
+-----------+



#### 2 Write a query to get Total budget ?

In [59]:
# With Pandas
Total_Budget = p_boxf["Budget in Crores"].sum()
print("Total_Budget: ",Total_Budget)

Total_Budget:  34653.0


In [60]:
# With PySpark
Total_Budget = s_boxf.select("Budget in Crores").agg(sum("Budget in Crores").alias("Total_Budget")).show()
print("Total_Budget: ",Total_Budget)

+------------+
|Total_Budget|
+------------+
|     34653.0|
+------------+

Total_Budget:  None


In [61]:
# With Spark SQL
spark.sql("""select sum(`Budget in Crores`) as Total_Budget from boxf_tb""").show()

+------------+
|Total_Budget|
+------------+
|     34653.0|
+------------+



#### 3 Write a query to get Total worldwide collection ?

In [62]:
# With Pandas
Total_WWC = p_boxf["Worldwide Collection  in Crores"].sum()
print("Total worldwide collection: ", Total_WWC)

Total worldwide collection:  79878.79


In [63]:
# WIith PySpark
s_boxf.select("Worldwide Collection  in Crores")\
    .agg(sum("Worldwide Collection  in Crores")\
        .alias("Total worldwide collection")).show()

+--------------------------+
|Total worldwide collection|
+--------------------------+
|         79878.78999999998|
+--------------------------+



In [64]:
# With Spark SQL
spark.sql("select sum(`Worldwide Collection  in Crores`)\
as Total_worldwide_collection from boxf_tb").show()

+--------------------------+
|Total_worldwide_collection|
+--------------------------+
|         79878.78999999998|
+--------------------------+



#### 4 Write a query to get Total First day collection worldwide ?

In [65]:
# With Pandas
Total_FDCW = p_boxf["First Day Collection Worldwide  in Crores"].sum()
print("Total First day collection worldwide: ",Total_FDCW)

Total First day collection worldwide:  11070.27


In [66]:
# With PySpark
s_boxf.select("First Day Collection Worldwide  in Crores")\
    .agg(sum("First Day Collection Worldwide  in Crores")\
        .alias("Total First day collection worldwide")).show()

+------------------------------------+
|Total First day collection worldwide|
+------------------------------------+
|                  11070.270000000006|
+------------------------------------+



In [67]:
# With Spark SQL
spark.sql("select sum(`First Day Collection Worldwide  in Crores`)\
as Total_First_day_collection_worldwide from boxf_tb").show()

+------------------------------------+
|Total_First_day_collection_worldwide|
+------------------------------------+
|                  11070.270000000006|
+------------------------------------+



#### 5 Write a query to get Total Overseas collection ?

In [68]:
# With Pandas
Total_OC = p_boxf["Overseas Collection in Crores"].sum()
print("Total Overseas collection:",Total_OC)

Total Overseas collection: 20953.07


In [69]:
# With PySpark
s_boxf.select("Overseas Collection in Crores")\
    .agg(sum("Overseas Collection in Crores")
        .alias("Total_Overseas_collection")).show()

+-------------------------+
|Total_Overseas_collection|
+-------------------------+
|        20953.07000000001|
+-------------------------+



In [70]:
# With Spark SQL
spark.sql("select sum(`Overseas Collection in Crores`)\
as Total_OVerseas_Collection from boxf_tb").show()

+-------------------------+
|Total_OVerseas_Collection|
+-------------------------+
|        20953.07000000001|
+-------------------------+



#### 6 Write a query to get Total India Gross collection ?

In [71]:
# With Pandas
Total_IGC = p_boxf["India Gross Collection in Crores"].sum()
print("Total India Gross collection: ",Total_IGC)

Total India Gross collection:  60064.1


In [72]:
# With PySpark
s_boxf.select("India Gross Collection in Crores")\
    .agg(sum("India Gross Collection in Crores")\
        .alias("Total India Gross collection")).show()

+----------------------------+
|Total India Gross collection|
+----------------------------+
|           60064.10000000004|
+----------------------------+



In [73]:
# With Spark SQL
spark.sql("select sum(`India Gross Collection in Crores`)as \
Total_India_Gross_collection from boxf_tb").show()

+----------------------------+
|Total_India_Gross_collection|
+----------------------------+
|           60064.10000000004|
+----------------------------+



#### 7. Top 10 filmsbased on world wide collections. Display films,collections

In [74]:
# With Pandas
p_boxf[['Title','First Day Collection Worldwide  in Crores',\
        'Worldwide Collection  in Crores',\
        'Overseas Collection in Crores',\
        'India Gross Collection in Crores']]\
    .sort_values("Worldwide Collection  in Crores",ascending = False)\
    .reset_index().head(10)

,index,Title,First Day Collection Worldwide in Crores,Worldwide Collection in Crores,Overseas Collection in Crores,India Gross Collection in Crores
0,524,Dangal,58.0,2122.30,1535.30,587.00
1,396,Bãhubali 2: The Conclusion,217.0,1788.00,371.00,1417.00
2,45,RRR (Rise Roar Revolt),223.0,1230.00,314.15,915.85
3,149,K.G.F: Chapter 2,159.0,1215.00,214.15,1000.85
4,572,Jawan,129.1,1160.00,400.00,760.00
5,573,Pathaan,104.8,1055.00,397.50,657.50
6,126,Kalki 2898-AD,177.7,1042.25,275.00,767.25
7,503,Bajrangi Bhaijaan,27.2,922.10,489.70,432.40
8,114,Animal,116.0,915.00,255.00,660.00
9,46,Secret Superstar,5.0,912.60,831.40,81.20


In [75]:
# With PySpark
s_boxf.select('Title','First Day Collection Worldwide  in Crores',\
        'Worldwide Collection  in Crores',\
        'Overseas Collection in Crores',\
        'India Gross Collection in Crores')\
    .sort(desc('Worldwide Collection  in Crores')).show(10,False)

+--------------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|Title                     |First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|
+--------------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|Dangal                    |58.0                                     |2122.3                         |1535.3                       |587.0                           |
|Bãhubali 2: The Conclusion|217.0                                    |1788.0                         |371.0                        |1417.0                          |
|RRR (Rise Roar Revolt)    |223.0                                    |1230.0                         |314.15                       |915.85                          |
|K.G

In [76]:
# With Spark SQL
spark.sql("select title,`First Day Collection Worldwide  in Crores`,\
`Worldwide Collection  in Crores`,`Overseas Collection in Crores`,\
`India Gross Collection in Crores` from boxf_tb \
order by `Worldwide Collection  in Crores` desc limit 10").show()

+--------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|               title|First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|
+--------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|              Dangal|                                     58.0|                         2122.3|                       1535.3|                           587.0|
|Bãhubali 2: The C...|                                    217.0|                         1788.0|                        371.0|                          1417.0|
|RRR (Rise Roar Re...|                                    223.0|                         1230.0|                       314.15|                          915.85|
|    K.G.F: Chapter 2|                  

#### 8 Total Number of films released by year

In [77]:
p_boxf["Year"] = p_boxf["Release Date"].dt.year

In [78]:
p_boxf.groupby("Year").agg(Total_Films = ("FilmID","count")).reset_index()

,Year,Total_Films
0,2014,32
1,2015,32
2,2016,41
3,2017,72
4,2018,71
5,2019,74
6,2020,27
7,2021,31
8,2022,80
9,2023,79


In [79]:
s_boxf = s_boxf.withColumn(
    "Release Date",
    expr("try_to_timestamp(`Release Date`, 'dd-MMM-yy')")
)

s_boxf.withColumn("Year", year(col("Release Date"))) \
      .groupBy("Year") \
      .agg(count("FilmID").alias("Total_Films")) \
      .show()

+----+-----------+
|Year|Total_Films|
+----+-----------+
|2018|         71|
|2015|         32|
|2023|         79|
|2022|         80|
|2014|         32|
|2019|         74|
|2020|         27|
|2016|         41|
|2024|         65|
|2017|         72|
|2021|         31|
+----+-----------+



In [80]:
# With Spark SQL
spark.sql("SELECT YEAR(try_to_timestamp\
(`Release Date`, 'dd-MMM-yy')) AS Yr,\
COUNT(FilmID) AS Film_count \
FROM boxf_tb \
GROUP BY Yr \
ORDER BY Yr").show()

+----+----------+
|  Yr|Film_count|
+----+----------+
|2014|        32|
|2015|        32|
|2016|        41|
|2017|        72|
|2018|        71|
|2019|        74|
|2020|        27|
|2021|        31|
|2022|        80|
|2023|        79|
|2024|        65|
+----+----------+



#### 9 Top 10 filmsbased on india collections.Display films,collections

In [81]:
# With Pandas
p_boxf[['Title','First Day Collection Worldwide  in Crores',\
        'Worldwide Collection  in Crores',\
        'Overseas Collection in Crores',\
        'India Gross Collection in Crores']]\
    .sort_values('India Gross Collection in Crores',ascending = False)\
    .reset_index().head(10)

,index,Title,First Day Collection Worldwide in Crores,Worldwide Collection in Crores,Overseas Collection in Crores,India Gross Collection in Crores
0,396,Bãhubali 2: The Conclusion,217.0,1788.00,371.00,1417.00
1,149,K.G.F: Chapter 2,159.0,1215.00,214.15,1000.85
2,45,RRR (Rise Roar Revolt),223.0,1230.00,314.15,915.85
3,126,Kalki 2898-AD,177.7,1042.25,275.00,767.25
4,572,Jawan,129.1,1160.00,400.00,760.00
5,127,Stree 2: Sarkate Ka Aatank,80.0,857.07,144.00,713.07
6,114,Animal,116.0,915.00,255.00,660.00
7,573,Pathaan,104.8,1055.00,397.50,657.50
8,115,Gadar 2,53.3,686.00,65.50,620.50
9,524,Dangal,58.0,2122.30,1535.30,587.00


In [82]:
# With PySpark
s_boxf.select('Title','First Day Collection Worldwide  in Crores',\
              'Worldwide Collection  in Crores',\
              'Overseas Collection in Crores',\
              'India Gross Collection in Crores')\
    .sort(desc('India Gross Collection in Crores')).show(10)

+--------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|               Title|First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|
+--------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|Bãhubali 2: The C...|                                    217.0|                         1788.0|                        371.0|                          1417.0|
|    K.G.F: Chapter 2|                                    159.0|                         1215.0|                       214.15|                         1000.85|
|RRR (Rise Roar Re...|                                    223.0|                         1230.0|                       314.15|                          915.85|
|       Kalki 2898-AD|                  

In [83]:
# With Spark SQL
spark.sql("select FilmID,`First Day Collection Worldwide  in Crores`,\
`Worldwide Collection  in Crores`,`Overseas Collection in Crores`,\
`India Gross Collection in Crores` from boxf_tb \
order by `India Gross Collection in Crores` desc limit 10").show()

+------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|FilmID|First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|
+------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|   104|                                    217.0|                         1788.0|                        371.0|                          1417.0|
|   287|                                    159.0|                         1215.0|                       214.15|                         1000.85|
|   286|                                    223.0|                         1230.0|                       314.15|                          915.85|
|   540|                                    177.7|                        1042.25|                        275.0|            

#### 10 Top 10 filmsbased on overses collections.Display films,collections

In [84]:
# With Pandas
p_boxf[['Title','First Day Collection Worldwide  in Crores',\
        'Worldwide Collection  in Crores',\
        'Overseas Collection in Crores',\
        'India Gross Collection in Crores']]\
    .sort_values('Overseas Collection in Crores',ascending = False)\
    .reset_index().head(10)

,index,Title,First Day Collection Worldwide in Crores,Worldwide Collection in Crores,Overseas Collection in Crores,India Gross Collection in Crores
0,524,Dangal,58.0,2122.30,1535.30,587.00
1,46,Secret Superstar,5.0,912.60,831.40,81.20
2,503,Bajrangi Bhaijaan,27.2,922.10,489.70,432.40
3,572,Jawan,129.1,1160.00,400.00,760.00
4,573,Pathaan,104.8,1055.00,397.50,657.50
5,396,Bãhubali 2: The Conclusion,217.0,1788.00,371.00,1417.00
6,45,RRR (Rise Roar Revolt),223.0,1230.00,314.15,915.85
7,460,PK,26.6,792.00,303.00,489.00
8,126,Kalki 2898-AD,177.7,1042.25,275.00,767.25
9,114,Animal,116.0,915.00,255.00,660.00


In [85]:
# With PySpark
s_boxf.select('Title','First Day Collection Worldwide  in Crores',\
              'Worldwide Collection  in Crores',\
              'Overseas Collection in Crores',\
              'India Gross Collection in Crores')\
    .sort(desc('Overseas Collection in Crores')).show(10)

+--------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|               Title|First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|
+--------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|              Dangal|                                     58.0|                         2122.3|                       1535.3|                           587.0|
|    Secret Superstar|                                      5.0|                          912.6|                        831.4|                            81.2|
|   Bajrangi Bhaijaan|                                     27.2|                          922.1|                        489.7|                           432.4|
|               Jawan|                  

In [86]:
# With Spark SQL
spark.sql("select FilmID,`First Day Collection Worldwide  in Crores`,\
`Worldwide Collection  in Crores`,`Overseas Collection in Crores`,\
`India Gross Collection in Crores` from boxf_tb \
order by `Overseas Collection in Crores` desc limit 10").show()

+------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|FilmID|First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|
+------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|   213|                                     58.0|                         2122.3|                       1535.3|                           587.0|
|   105|                                      5.0|                          912.6|                        831.4|                            81.2|
|   181|                                     27.2|                          922.1|                        489.7|                           432.4|
|   463|                                    129.1|                         1160.0|                        400.0|            

#### 11 Top 10 filmsbased on firstday collections.Display films,collections

In [87]:
# With Pandas
p_boxf[['Title','First Day Collection Worldwide  in Crores',\
        'Worldwide Collection  in Crores',\
        'Overseas Collection in Crores',\
        'India Gross Collection in Crores']]\
    .sort_values('First Day Collection Worldwide  in Crores',ascending = False)\
    .reset_index().head(10)

,index,Title,First Day Collection Worldwide in Crores,Worldwide Collection in Crores,Overseas Collection in Crores,India Gross Collection in Crores
0,45,RRR (Rise Roar Revolt),223.0,1230.00,314.15,915.85
1,396,Bãhubali 2: The Conclusion,217.0,1788.00,371.00,1417.00
2,126,Kalki 2898-AD,177.7,1042.25,275.00,767.25
3,149,K.G.F: Chapter 2,159.0,1215.00,214.15,1000.85
4,212,Salaar: Part 1 - Ceasefire,158.1,617.75,130.00,487.75
5,116,Leo,142.7,605.90,204.00,401.90
6,455,Devara: Part 1,142.0,421.60,78.85,342.75
7,572,Jawan,129.1,1160.00,400.00,760.00
8,213,Adipurush,127.5,393.00,50.00,343.00
9,114,Animal,116.0,915.00,255.00,660.00


In [88]:
# With PySpark
s_boxf.select('Title','First Day Collection Worldwide  in Crores',\
              'Worldwide Collection  in Crores',\
              'Overseas Collection in Crores',\
              'India Gross Collection in Crores')\
    .sort(desc('First Day Collection Worldwide  in Crores')).show(10)

+--------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|               Title|First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|
+--------------------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|RRR (Rise Roar Re...|                                    223.0|                         1230.0|                       314.15|                          915.85|
|Bãhubali 2: The C...|                                    217.0|                         1788.0|                        371.0|                          1417.0|
|       Kalki 2898-AD|                                    177.7|                        1042.25|                        275.0|                          767.25|
|    K.G.F: Chapter 2|                  

In [89]:
# With Spark SQL
spark.sql("select FilmID,`First Day Collection Worldwide  in Crores`,\
`Worldwide Collection  in Crores`,`Overseas Collection in Crores`,\
`India Gross Collection in Crores` from boxf_tb \
order by `First Day Collection Worldwide  in Crores` desc limit 10").show()

+------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|FilmID|First Day Collection Worldwide  in Crores|Worldwide Collection  in Crores|Overseas Collection in Crores|India Gross Collection in Crores|
+------+-----------------------------------------+-------------------------------+-----------------------------+--------------------------------+
|   286|                                    223.0|                         1230.0|                       314.15|                          915.85|
|   104|                                    217.0|                         1788.0|                        371.0|                          1417.0|
|   540|                                    177.7|                        1042.25|                        275.0|                          767.25|
|   287|                                    159.0|                         1215.0|                       214.15|            

#### 12 Weekday wise films released,Display week name and no of films released

In [90]:
# exracting the week days from date
p_boxf["WeekDay"] = p_boxf["Release Date"].dt.strftime("%A") 

In [91]:
# With Pandas
p_boxf.groupby("WeekDay").agg(Total_Film=("FilmID","count"))\
    .sort_values("Total_Film",ascending = False).reset_index()

,WeekDay,Total_Film
0,Friday,423
1,Thursday,124
2,Wednesday,35
3,Saturday,10
4,Sunday,7
5,Tuesday,5


In [92]:
# extracting the week days from date
s_boxf = s_boxf.withColumn("WeekDay", date_format("Release Date", "EEEE"))

In [93]:
# With PySpark
s_boxf.groupby("WeekDay").agg(count("FilmID")\
    .alias("Total_Films"))\
    .orderBy(desc("Total_Films")).show()

+---------+-----------+
|  WeekDay|Total_Films|
+---------+-----------+
|   Friday|        423|
| Thursday|        124|
|Wednesday|         35|
| Saturday|         10|
|   Sunday|          7|
|  Tuesday|          5|
+---------+-----------+



In [94]:
# With Spark SQL
spark.sql("""Select date_format(to_date(`Release Date`,'dd-MMM-yy'),\
'EEEE')as WeekDay, count(FilmID) as Total_Films from boxf_tb \
group by WeekDay \
order by 2 desc""").show()

+---------+-----------+
|  WeekDay|Total_Films|
+---------+-----------+
|   Friday|        423|
| Thursday|        124|
|Wednesday|         35|
| Saturday|         10|
|   Sunday|          7|
|  Tuesday|          5|
+---------+-----------+



#### 13 Write a query to get OTT platofrm wise movies count

In [95]:
# With Pandas
p_boxf.groupby("OTT Platform")[['FilmID']].count()\
    .sort_values("FilmID",ascending = False).reset_index()

,OTT Platform,FilmID
0,Amazon Prime Video,384
1,Netflix,99
2,Disney+ Hotstar,54
3,ZEE5,39
4,Sun NXT,11
5,Aha,8
6,SonyLIV,4
7,JioCinema,3
8,Google Play Movies,1
9,Voot,1


In [96]:
# With PySpark
s_boxf.groupby("OTT Platform")\
    .agg(count("FilmID")\
        .alias("Film_count"))\
    .sort(desc("Film_count")).show()

+------------------+----------+
|      OTT Platform|Film_count|
+------------------+----------+
|Amazon Prime Video|       384|
|           Netflix|        99|
|   Disney+ Hotstar|        54|
|              Zee5|        39|
|           Sun Nxt|        11|
|               Aha|         8|
|           Sonyliv|         4|
|         Jiocinema|         3|
|              Voot|         1|
|Google Play Movies|         1|
+------------------+----------+



In [97]:
# With Spark SQL
spark.sql("select `OTT platform`,\
count(FilmID) as Count_film from boxf_tb \
group by `OTT Platform`\
order by Count_film desc").show()

+------------------+----------+
|      OTT platform|Count_film|
+------------------+----------+
|Amazon Prime Video|       384|
|           Netflix|        99|
|   Disney+ Hotstar|        54|
|              Zee5|        39|
|           Sun Nxt|        11|
|               Aha|         8|
|           Sonyliv|         4|
|         Jiocinema|         3|
|              Voot|         1|
|Google Play Movies|         1|
+------------------+----------+



#### 14 Top 10 Directors by films released

In [98]:
# With Pandas
p_boxf.merge(p_dir,left_on ="DirectorID", right_on = "Director ID", how = "right")\
    .groupby("Director")[["FilmID"]].count().sort_values("FilmID",ascending = False)\
    .reset_index().head(10)

,Director,FilmID
0,Trivikram Srinivas,6
1,Rohit Shetty,6
2,A.R. Murugadoss,5
3,Siva,5
4,Anil Ravipudi,5
5,Boyapati Srinu,5
6,Koratala Siva,5
7,Remo D'Souza,4
8,Atlee,4
9,Siddharth Anand,4


In [99]:
# With PySpark
s_boxf.join(s_dir, s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director").agg(count("FilmID").alias("Film_Count"))\
    .sort(desc("Film_Count")).show(10)

+------------------+----------+
|          Director|Film_Count|
+------------------+----------+
|Trivikram Srinivas|         6|
|      Rohit Shetty|         6|
|     Anil Ravipudi|         5|
|     Koratala Siva|         5|
|   A.R. Murugadoss|         5|
|    Boyapati Srinu|         5|
|              Siva|         5|
|    Maruthi Dasari|         4|
|         Sundar C.|         4|
|   Siddharth Anand|         4|
+------------------+----------+
only showing top 10 rows


In [100]:
# With Spark SQL
spark.sql("select Director, count(FilmID) as Film_Count \
from boxf_tb boxf \
right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
group by Director \
order by Film_Count desc limit 10").show()

+------------------+----------+
|          Director|Film_Count|
+------------------+----------+
|Trivikram Srinivas|         6|
|      Rohit Shetty|         6|
|     Anil Ravipudi|         5|
|     Koratala Siva|         5|
|   A.R. Murugadoss|         5|
|    Boyapati Srinu|         5|
|              Siva|         5|
|    Maruthi Dasari|         4|
|  Lokesh Kanagaraj|         4|
|   Siddharth Anand|         4|
+------------------+----------+



#### 15 Top 10 directors by world wide collection

In [101]:
# With Pandas
p_boxf.merge(p_dir, left_on = "DirectorID", right_on = "Director ID",how= "right")\
    .groupby("Director").agg(Total_WWC=("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index().head(10)

,Director,Total_WWC
0,S.S. Rajamouli,3668.00
1,Siddharth Anand,2229.45
2,Nitesh Tiwari,2122.30
3,Prashanth Neel,2101.75
4,Atlee,1865.35
5,Rajkumar Hirani,1834.50
6,Rohit Shetty,1656.60
7,Lokesh Kanagaraj,1350.13
8,Sandeep Reddy Vanga,1342.00
9,Kabir Khan,1220.55


In [102]:
# With PySpark
s_boxf.join(s_dir, s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director").agg(sum("Worldwide Collection  in Crores")\
        .alias("Total_WWC")).sort(desc("Total_WWC")).show(10)

+-------------------+------------------+
|           Director|         Total_WWC|
+-------------------+------------------+
|     S.S. Rajamouli|            3668.0|
|    Siddharth Anand|           2229.45|
|      Nitesh Tiwari|            2122.3|
|     Prashanth Neel|           2101.75|
|              Atlee|           1865.35|
|    Rajkumar Hirani|            1834.5|
|       Rohit Shetty|            1656.6|
|   Lokesh Kanagaraj|1350.1299999999999|
|Sandeep Reddy Vanga|            1342.0|
|         Kabir Khan|           1220.55|
+-------------------+------------------+
only showing top 10 rows


In [103]:
# With Spark SQL
spark.sql("select Director,sum(`Worldwide Collection  in Crores`) as Total_WWC \
from boxf_tb boxf \
right join dir_tb dir on dir.`Director ID` = boxf.DirectorID \
group by Director \
order by Total_WWC desc limit 10 ").show()

+-------------------+------------------+
|           Director|         Total_WWC|
+-------------------+------------------+
|     S.S. Rajamouli|            3668.0|
|    Siddharth Anand|           2229.45|
|      Nitesh Tiwari|            2122.3|
|     Prashanth Neel|           2101.75|
|              Atlee|           1865.35|
|    Rajkumar Hirani|            1834.5|
|       Rohit Shetty|            1656.6|
|   Lokesh Kanagaraj|1350.1299999999999|
|Sandeep Reddy Vanga|            1342.0|
|         Kabir Khan|           1220.55|
+-------------------+------------------+



#### 16 Top10 lead actors by world wide collection

In [104]:
# With Pandas
p_boxf.groupby("Lead Actor/Actress").agg(Total_WWC=('Worldwide Collection  in Crores','sum'))\
    .sort_values("Total_WWC", ascending = False).reset_index().head(10)

,Lead Actor/Actress,Total_WWC
0,Prabhas,5091.50
1,Salman Khan,4515.20
2,Shah Rukh Khan,3800.10
3,Akshay Kumar,3168.22
4,Joseph Vijay,3076.11
5,Aamir Khan,2914.30
6,Rajinikanth,2695.55
7,Ajay Devgn,2571.21
8,N.T. Rama Rao Jr.,2242.60
9,Ranbir Kapoor,2111.25


In [105]:
# With PySpark
s_boxf.groupby("Lead Actor/Actress").agg(sum("Worldwide Collection  in Crores")\
    .alias("Total_WWC")).sort(desc("Total_WWC")).show(10)

+------------------+------------------+
|Lead Actor/Actress|         Total_WWC|
+------------------+------------------+
|           Prabhas|            5091.5|
|       Salman Khan|            4515.2|
|    Shah Rukh Khan|3800.1000000000004|
|      Akshay Kumar|           3168.22|
|      Joseph Vijay|3076.1099999999997|
|        Aamir Khan|            2914.3|
|       Rajinikanth|           2695.55|
|        Ajay Devgn|2571.2100000000005|
| N.T. Rama Rao Jr.|            2242.6|
|     Ranbir Kapoor|           2111.25|
+------------------+------------------+
only showing top 10 rows


In [106]:
# With Spark SQL
spark.sql("select `Lead Actor/Actress`,\
sum(`Worldwide Collection  in Crores`) as Total_WWC \
from boxf_tb \
group by `Lead Actor/Actress` \
order by Total_WWC desc limit 10").show()

+------------------+------------------+
|Lead Actor/Actress|         Total_WWC|
+------------------+------------------+
|           Prabhas|            5091.5|
|       Salman Khan|            4515.2|
|    Shah Rukh Khan|3800.1000000000004|
|      Akshay Kumar|           3168.22|
|      Joseph Vijay|3076.1099999999997|
|        Aamir Khan|            2914.3|
|       Rajinikanth|           2695.55|
|        Ajay Devgn|2571.2100000000005|
| N.T. Rama Rao Jr.|            2242.6|
|     Ranbir Kapoor|           2111.25|
+------------------+------------------+



#### 17 Top 10 movies by IMDb rating

In [107]:
# With Pandas 
p_boxf[["Title","IMDb Rating"]]\
    .sort_values("IMDb Rating",ascending=False)\
    .reset_index().head(10)

,index,Title,IMDb Rating
0,121,12th Fail,8.8
1,316,Rocketry: The Nambi Effect,8.7
2,83,777 Charlie,8.7
3,140,Kishkindha Kaandam,8.6
4,79,The Kashmir Files,8.6
5,437,Jersey,8.5
6,25,96,8.5
7,512,Sachin,8.5
8,350,Meiyazhagan,8.5
9,204,Kumbalangi Nights,8.5


In [108]:
# With PySpark
s_boxf.select("Title","IMDb Rating")\
    .sort(desc("IMDb Rating")).show(10)

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|           12th Fail|        8.8|
|         777 Charlie|        8.7|
|Rocketry: The Nam...|        8.7|
|   The Kashmir Files|        8.6|
|  Kishkindha Kaandam|        8.6|
|          Sita Ramam|        8.5|
|                  96|        8.5|
|            Maharaja|        8.5|
|   Kumbalangi Nights|        8.5|
|         Meiyazhagan|        8.5|
+--------------------+-----------+
only showing top 10 rows


In [109]:
# With Spark SQL
spark.sql("select Title, `IMDb Rating` from boxf_tb \
order by `IMDb Rating` desc limit 10").show()

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|           12th Fail|        8.8|
|         777 Charlie|        8.7|
|Rocketry: The Nam...|        8.7|
|   The Kashmir Files|        8.6|
|  Kishkindha Kaandam|        8.6|
|                  96|        8.5|
|          Sita Ramam|        8.5|
|   Kumbalangi Nights|        8.5|
|            Maharaja|        8.5|
|         Meiyazhagan|        8.5|
+--------------------+-----------+



#### 18 Bottom 10 movies by IMDb rating

In [110]:
# With Pandas
p_boxf[["Title","IMDb Rating"]]\
    .sort_values("IMDb Rating",ascending = True)\
    .reset_index().head(10)

,index,Title,IMDb Rating
0,468,Race 3,1.9
1,491,Baaghi 3,2.2
2,571,Student of the Year 2,2.2
3,496,Heropanti 2,2.3
4,338,Chandramukhi 2,2.6
5,417,Liger,2.6
6,478,Gunday,2.7
7,213,Adipurush,2.7
8,565,Dabangg 3,3.0
9,495,A Flying Jatt,3.1


In [111]:
# With PySpark
s_boxf.select("Title","IMDb Rating")\
    .sort(asc("IMDb Rating")).show(10)

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|              Race 3|        1.9|
|            Baaghi 3|        2.2|
|Student of the Ye...|        2.2|
|         Heropanti 2|        2.3|
|      Chandramukhi 2|        2.6|
|               Liger|        2.6|
|           Adipurush|        2.7|
|              Gunday|        2.7|
|           Dabangg 3|        3.0|
|       A Flying Jatt|        3.1|
+--------------------+-----------+
only showing top 10 rows


In [112]:
# With Spark SQl
spark.sql("Select Title, `IMDb Rating` from boxf_tb \
order by `IMDb Rating` asc limit 10").show()

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|              Race 3|        1.9|
|            Baaghi 3|        2.2|
|Student of the Ye...|        2.2|
|         Heropanti 2|        2.3|
|      Chandramukhi 2|        2.6|
|               Liger|        2.6|
|           Adipurush|        2.7|
|              Gunday|        2.7|
|           Dabangg 3|        3.0|
|       A Flying Jatt|        3.1|
+--------------------+-----------+



#### 19 Write a query to get 5 longest run time movies 

In [113]:
# With Pandas
p_boxf[["Title","Runtime (mins)"]]\
    .sort_values("Runtime (mins)",ascending = False)\
    .reset_index().head(5)

,index,Title,Runtime (mins)
0,114,Animal,204
1,39,I,188
2,45,RRR (Rise Roar Revolt),187
3,103,Avane Srimannarayana,186
4,261,Jilla,185


In [114]:
# With PySpark
s_boxf.select("Title","Runtime (mins)")\
    .sort(desc("Runtime (mins)")).show(5)

+--------------------+--------------+
|               Title|Runtime (mins)|
+--------------------+--------------+
|              Animal|           204|
|                   I|           188|
|RRR (Rise Roar Re...|           187|
|Avane Srimannarayana|           186|
|         Kurukshetra|           185|
+--------------------+--------------+
only showing top 5 rows


In [115]:
# With Spark SQL
spark.sql("select Title, `Runtime (mins)` from boxf_tb \
order by `Runtime (mins)` desc limit 5").show()

+--------------------+--------------+
|               Title|Runtime (mins)|
+--------------------+--------------+
|              Animal|           204|
|                   I|           188|
|RRR (Rise Roar Re...|           187|
|Avane Srimannarayana|           186|
|               Jilla|           185|
+--------------------+--------------+



#### 20 Write a query to get 5 shortest run time movies

In [116]:
# With Pandas
p_boxf[["Title","Runtime (mins)"]]\
    .sort_values("Runtime (mins)",ascending = True)\
    .reset_index().head(5)

,index,Title,Runtime (mins)
0,599,Kill,105
1,497,Raksha Bandhan,108
2,560,Bhoot: Part One - The Haunted Ship,114
3,477,Hichki,116
4,407,The Ghazi Attack,116


In [117]:
# With PySpark
s_boxf.select("Title","Runtime (mins)")\
    .sort(asc("Runtime (mins)")).show(5)

+--------------------+--------------+
|               Title|Runtime (mins)|
+--------------------+--------------+
|                Kill|           105|
|      Raksha Bandhan|           108|
|Bhoot: Part One -...|           114|
|    The Ghazi Attack|           116|
|                Guru|           116|
+--------------------+--------------+
only showing top 5 rows


In [118]:
# With Spark SQL
spark.sql("select Title,`Runtime (mins)` from boxf_tb \
order by `Runtime (mins)` asc limit 5").show()

+--------------------+--------------+
|               Title|Runtime (mins)|
+--------------------+--------------+
|                Kill|           105|
|      Raksha Bandhan|           108|
|Bhoot: Part One -...|           114|
|                Guru|           116|
|              Hichki|           116|
+--------------------+--------------+



#### 21 Top7 movies by world wide collection in Bollywood

In [119]:
# With Pandas 
p_boxf.query("Industry=='Bollywood'").groupby(["Industry","Title"])\
    .agg(Total_WWC = ("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index().head(7)

,Industry,Title,Total_WWC
0,Bollywood,Dangal,2122.30
1,Bollywood,Jawan,1160.00
2,Bollywood,Pathaan,1055.00
3,Bollywood,Bajrangi Bhaijaan,922.10
4,Bollywood,Animal,915.00
5,Bollywood,Secret Superstar,912.60
6,Bollywood,Stree 2: Sarkate Ka Aatank,857.07


In [120]:
# With PySpark
s_boxf.filter(col("Industry")=="Bollywood").groupby("Industry","Title")\
    .agg(sum("Worldwide Collection  in Crores")\
    .alias("Total_WWC")).sort(desc("Total_WWC")).show(7)

+---------+--------------------+---------+
| Industry|               Title|Total_WWC|
+---------+--------------------+---------+
|Bollywood|              Dangal|   2122.3|
|Bollywood|               Jawan|   1160.0|
|Bollywood|             Pathaan|   1055.0|
|Bollywood|   Bajrangi Bhaijaan|    922.1|
|Bollywood|              Animal|    915.0|
|Bollywood|    Secret Superstar|    912.6|
|Bollywood|Stree 2: Sarkate ...|   857.07|
+---------+--------------------+---------+
only showing top 7 rows


In [121]:
# With Spark SQL
spark.sql("select Industry,Title, sum(`Worldwide Collection  in Crores`) \
as Total_WWC from boxf_tb \
Where Industry = 'Bollywood' \
group by Industry,Title \
order by Total_WWC desc limit 7").show()

+---------+--------------------+---------+
| Industry|               Title|Total_WWC|
+---------+--------------------+---------+
|Bollywood|              Dangal|   2122.3|
|Bollywood|               Jawan|   1160.0|
|Bollywood|             Pathaan|   1055.0|
|Bollywood|   Bajrangi Bhaijaan|    922.1|
|Bollywood|              Animal|    915.0|
|Bollywood|    Secret Superstar|    912.6|
|Bollywood|Stree 2: Sarkate ...|   857.07|
+---------+--------------------+---------+



#### 22 Top7 movies by world wide collection in Tollywood

In [122]:
# With Pandas 
p_boxf.query("Industry=='Tollywood'").groupby(["Industry","Title"])\
    .agg(Total_WWC = ("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index().head(7)

,Industry,Title,Total_WWC
0,Tollywood,Bãhubali 2: The Conclusion,1788.00
1,Tollywood,RRR (Rise Roar Revolt),1230.00
2,Tollywood,Kalki 2898-AD,1042.25
3,Tollywood,Bãhubali: The Beginning,650.00
4,Tollywood,Salaar: Part 1 - Ceasefire,617.75
5,Tollywood,Saaho,451.00
6,Tollywood,Devara: Part 1,421.60


In [123]:
# With PySpark
s_boxf.filter(col("Industry")=="Tollywood").groupby("Industry","Title")\
    .agg(sum("Worldwide Collection  in Crores")\
    .alias("Total_WWC")).sort(desc("Total_WWC")).show(7)

+---------+--------------------+---------+
| Industry|               Title|Total_WWC|
+---------+--------------------+---------+
|Tollywood|Bãhubali 2: The C...|   1788.0|
|Tollywood|RRR (Rise Roar Re...|   1230.0|
|Tollywood|       Kalki 2898-AD|  1042.25|
|Tollywood|Bãhubali: The Beg...|    650.0|
|Tollywood|Salaar: Part 1 - ...|   617.75|
|Tollywood|               Saaho|    451.0|
|Tollywood|      Devara: Part 1|    421.6|
+---------+--------------------+---------+
only showing top 7 rows


In [124]:
# With Spark SQL
spark.sql("select Industry,Title, sum(`Worldwide Collection  in Crores`) \
as Total_WWC from boxf_tb \
Where Industry = 'Tollywood' \
group by Industry,Title \
order by Total_WWC desc limit 7").show()

+---------+--------------------+---------+
| Industry|               Title|Total_WWC|
+---------+--------------------+---------+
|Tollywood|Bãhubali 2: The C...|   1788.0|
|Tollywood|RRR (Rise Roar Re...|   1230.0|
|Tollywood|       Kalki 2898-AD|  1042.25|
|Tollywood|Bãhubali: The Beg...|    650.0|
|Tollywood|Salaar: Part 1 - ...|   617.75|
|Tollywood|               Saaho|    451.0|
|Tollywood|      Devara: Part 1|    421.6|
+---------+--------------------+---------+



#### 23 Top7 movies by world wide collection in Kollywood

In [125]:
# With Pandas 
p_boxf.query("Industry=='Kollywood'").groupby(["Industry","Title"])\
    .agg(Total_WWC = ("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index().head(7)

,Industry,Title,Total_WWC
0,Kollywood,2,701.00
1,Kollywood,Leo,605.90
2,Kollywood,Jailer,604.50
3,Kollywood,Ponniyin Selvan: Part One,488.36
4,Kollywood,The GOAT,451.23
5,Kollywood,Vikram,414.43
6,Kollywood,Ponniyin Selvan: Part Two,344.63


In [126]:
# With PySpark
s_boxf.filter(col("Industry")=="Kollywood").groupby("Industry","Title")\
    .agg(sum("Worldwide Collection  in Crores")\
    .alias("Total_WWC")).sort(desc("Total_WWC")).show(7)

+---------+--------------------+---------+
| Industry|               Title|Total_WWC|
+---------+--------------------+---------+
|Kollywood|                   2|    701.0|
|Kollywood|                 Leo|    605.9|
|Kollywood|              Jailer|    604.5|
|Kollywood|Ponniyin Selvan: ...|   488.36|
|Kollywood|            The GOAT|   451.23|
|Kollywood|              Vikram|   414.43|
|Kollywood|Ponniyin Selvan: ...|   344.63|
+---------+--------------------+---------+
only showing top 7 rows


In [127]:
# With Spark SQL
spark.sql("select Industry,Title, sum(`Worldwide Collection  in Crores`) \
as Total_WWC from boxf_tb \
Where Industry = 'Kollywood' \
group by Industry,Title \
order by Total_WWC desc limit 7").show()

+---------+--------------------+---------+
| Industry|               Title|Total_WWC|
+---------+--------------------+---------+
|Kollywood|                   2|    701.0|
|Kollywood|                 Leo|    605.9|
|Kollywood|              Jailer|    604.5|
|Kollywood|Ponniyin Selvan: ...|   488.36|
|Kollywood|            The GOAT|   451.23|
|Kollywood|              Vikram|   414.43|
|Kollywood|Ponniyin Selvan: ...|   344.63|
+---------+--------------------+---------+



#### 24 Top7 movies by world wide collection in Sandalwood

In [128]:
# With Pandas 
p_boxf.query("Industry=='Sandalwood'").groupby(["Industry","Title"])\
    .agg(Total_WWC = ("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index().head(7)

,Industry,Title,Total_WWC
0,Sandalwood,K.G.F: Chapter 2,1215.00
1,Sandalwood,Kantara,407.82
2,Sandalwood,K.G.F: Chapter 1,238.00
3,Sandalwood,777 Charlie,102.75
4,Sandalwood,VR (Vikrant Rona),100.35
5,Sandalwood,James,94.20
6,Sandalwood,Kurukshetra,81.00


In [129]:
# With PySpark
s_boxf.filter(col("Industry")=="Sandalwood").groupby("Industry","Title")\
    .agg(sum("Worldwide Collection  in Crores")\
    .alias("Total_WWC")).sort(desc("Total_WWC")).show(7)

+----------+-----------------+---------+
|  Industry|            Title|Total_WWC|
+----------+-----------------+---------+
|Sandalwood| K.G.F: Chapter 2|   1215.0|
|Sandalwood|          Kantara|   407.82|
|Sandalwood| K.G.F: Chapter 1|    238.0|
|Sandalwood|      777 Charlie|   102.75|
|Sandalwood|VR (Vikrant Rona)|   100.35|
|Sandalwood|            James|     94.2|
|Sandalwood|      Kurukshetra|     81.0|
+----------+-----------------+---------+
only showing top 7 rows


In [130]:
# With Spark SQL
spark.sql("select Industry,Title, sum(`Worldwide Collection  in Crores`) \
as Total_WWC from boxf_tb \
Where Industry = 'Sandalwood' \
group by Industry,Title \
order by Total_WWC desc limit 7").show()

+----------+-----------------+---------+
|  Industry|            Title|Total_WWC|
+----------+-----------------+---------+
|Sandalwood| K.G.F: Chapter 2|   1215.0|
|Sandalwood|          Kantara|   407.82|
|Sandalwood| K.G.F: Chapter 1|    238.0|
|Sandalwood|      777 Charlie|   102.75|
|Sandalwood|VR (Vikrant Rona)|   100.35|
|Sandalwood|            James|     94.2|
|Sandalwood|      Kurukshetra|     81.0|
+----------+-----------------+---------+



#### 25 Top7 movies by world wide collection in Mollywood

In [131]:
# With Pandas 
p_boxf.query("Industry=='Mollywood'").groupby(["Industry","Title"])\
    .agg(Total_WWC = ("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index().head(7)

,Industry,Title,Total_WWC
0,Mollywood,Manjummel Boys,241.10
1,Mollywood,2018,180.03
2,Mollywood,The Goat Life,157.35
3,Mollywood,Aavesham,154.79
4,Mollywood,Pulimurugan,150.00
5,Mollywood,Premalu,131.18
6,Mollywood,Lucifer,129.20


In [132]:
# With PySpark
s_boxf.filter(col("Industry")=="Mollywood").groupby("Industry","Title")\
    .agg(sum("Worldwide Collection  in Crores")\
    .alias("Total_WWC")).sort(desc("Total_WWC")).show(7)

+---------+--------------+---------+
| Industry|         Title|Total_WWC|
+---------+--------------+---------+
|Mollywood|Manjummel Boys|    241.1|
|Mollywood|          2018|   180.03|
|Mollywood| The Goat Life|   157.35|
|Mollywood|      Aavesham|   154.79|
|Mollywood|   Pulimurugan|    150.0|
|Mollywood|       Premalu|   131.18|
|Mollywood|       Lucifer|    129.2|
+---------+--------------+---------+
only showing top 7 rows


In [133]:
# With Spark SQL
spark.sql("select Industry,Title, sum(`Worldwide Collection  in Crores`) \
as Total_WWC from boxf_tb \
Where Industry = 'Mollywood' \
group by Industry,Title \
order by Total_WWC desc limit 7").show()

+---------+--------------+---------+
| Industry|         Title|Total_WWC|
+---------+--------------+---------+
|Mollywood|Manjummel Boys|    241.1|
|Mollywood|          2018|   180.03|
|Mollywood| The Goat Life|   157.35|
|Mollywood|      Aavesham|   154.79|
|Mollywood|   Pulimurugan|    150.0|
|Mollywood|       Premalu|   131.18|
|Mollywood|       Lucifer|    129.2|
+---------+--------------+---------+



#### 26 Write query to display industry and verdict wise films count

In [134]:
# With Pandas
p_boxf.groupby(["Industry","Verdict"])\
    .agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False)\
    .reset_index().head(10)

,Industry,Verdict,Total_Films
0,Mollywood,Blockbuster,46
1,Kollywood,Blockbuster,44
2,Bollywood,Hit,39
3,Tollywood,Blockbuster,39
4,Bollywood,Blockbuster,30
5,Tollywood,Hit,25
6,Bollywood,Disaster,25
7,Tollywood,Disaster,24
8,Kollywood,SuperHit,23
9,Tollywood,SuperHit,23


In [135]:
# With PySpark
s_boxf.groupby(["Industry","Verdict"])\
    .agg(count("FilmID").alias("Total_Films"))\
    .sort(desc("Total_Films")).show(10)

+---------+-----------+-----------+
| Industry|    Verdict|Total_Films|
+---------+-----------+-----------+
|Mollywood|Blockbuster|         46|
|Kollywood|Blockbuster|         44|
|Tollywood|Blockbuster|         39|
|Bollywood|        Hit|         39|
|Bollywood|Blockbuster|         30|
|Bollywood|   Disaster|         25|
|Tollywood|        Hit|         25|
|Tollywood|   Disaster|         24|
|Kollywood|  Super Hit|         23|
|Tollywood|  Super Hit|         23|
+---------+-----------+-----------+
only showing top 10 rows


In [136]:
# With Spark SQL
spark.sql("select Industry, Verdict, count(FilmID)as \
Total_Film from boxf_tb \
group by Industry, Verdict \
order by Total_Film desc limit 10").show()

+---------+-----------+----------+
| Industry|    Verdict|Total_Film|
+---------+-----------+----------+
|Mollywood|Blockbuster|        46|
|Kollywood|Blockbuster|        44|
|Bollywood|        Hit|        39|
|Tollywood|Blockbuster|        39|
|Bollywood|Blockbuster|        30|
|Bollywood|   Disaster|        25|
|Tollywood|        Hit|        25|
|Tollywood|   Disaster|        24|
|Kollywood|  Super Hit|        23|
|Tollywood|  Super Hit|        23|
+---------+-----------+----------+



#### 27 Write query to get films based on budget in Bollowood

In [137]:
# With Pandas 
p_boxf.query("Industry =='Bollywood'")\
    .groupby(["Industry","Title"])\
    .agg(Budget = ( "Budget in Crores","sum"))\
    .sort_values("Budget", ascending = False)\
    .reset_index().head(10)

,Industry,Title,Budget
0,Bollywood,Brahmastra Part One: Shiva,400.0
1,Bollywood,Jawan,300.0
2,Bollywood,Thugs of Hindostan,275.0
3,Bollywood,Pathaan,250.0
4,Bollywood,Tiger 3,250.0
5,Bollywood,Bade Miyan Chote Miyan,250.0
6,Bollywood,Fighter,250.0
7,Bollywood,Samrat Prithviraj,220.0
8,Bollywood,Tiger Zinda Hai,200.0
9,Bollywood,83,200.0


In [138]:
# With PySpark
s_boxf.filter(col("Industry")=="Bollywood")\
    .groupby("Industry","Title")\
    .agg(sum("Budget in Crores").alias("Budget"))\
    .sort(desc("Budget")).show(10)

+---------+--------------------+------+
| Industry|               Title|Budget|
+---------+--------------------+------+
|Bollywood|Brahmastra Part O...| 400.0|
|Bollywood|               Jawan| 300.0|
|Bollywood|  Thugs of Hindostan| 275.0|
|Bollywood|             Pathaan| 250.0|
|Bollywood|Bade Miyan Chote ...| 250.0|
|Bollywood|             Fighter| 250.0|
|Bollywood|             Tiger 3| 250.0|
|Bollywood|   Samrat Prithviraj| 220.0|
|Bollywood|                Zero| 200.0|
|Bollywood|     Tiger Zinda Hai| 200.0|
+---------+--------------------+------+
only showing top 10 rows


In [139]:
# With Spark SQL
spark.sql("select Industry, Title,`Budget in Crores` as Budget \
from boxf_tb \
where Industry = 'Bollywood' \
order by Budget desc limit 10").show()

+---------+--------------------+------+
| Industry|               Title|Budget|
+---------+--------------------+------+
|Bollywood|Brahmastra Part O...| 400.0|
|Bollywood|               Jawan| 300.0|
|Bollywood|  Thugs of Hindostan| 275.0|
|Bollywood|             Pathaan| 250.0|
|Bollywood|             Tiger 3| 250.0|
|Bollywood|             Fighter| 250.0|
|Bollywood|Bade Miyan Chote ...| 250.0|
|Bollywood|   Samrat Prithviraj| 220.0|
|Bollywood|                Zero| 200.0|
|Bollywood|                  83| 200.0|
+---------+--------------------+------+



#### 28 Write query to get films based on budget in Tollywood 

In [140]:
# With Pandas 
p_boxf.query("Industry =='Tollywood'")\
    .groupby(["Industry","Title"])\
    .agg(Budget = ( "Budget in Crores","sum"))\
    .sort_values("Budget", ascending = False)\
    .reset_index().head(10)

,Industry,Title,Budget
0,Tollywood,RRR (Rise Roar Revolt),550.0
1,Tollywood,Adipurush,500.0
2,Tollywood,Kalki 2898-AD,500.0
3,Tollywood,Radhe Shyam,300.0
4,Tollywood,Salaar: Part 1 - Ceasefire,280.0
5,Tollywood,Bãhubali 2: The Conclusion,250.0
6,Tollywood,Devara: Part 1,250.0
7,Tollywood,Bãhubali: The Beginning,180.0
8,Tollywood,Guntur Kaaram,150.0
9,Tollywood,Pushpa: The Rise - Part 1,150.0


In [141]:
# With PySpark
s_boxf.filter(col("Industry")=="Tollywood")\
    .groupby("Industry","Title")\
    .agg(sum("Budget in Crores").alias("Budget"))\
    .sort(desc("Budget")).show(10)

+---------+--------------------+------+
| Industry|               Title|Budget|
+---------+--------------------+------+
|Tollywood|RRR (Rise Roar Re...| 550.0|
|Tollywood|           Adipurush| 500.0|
|Tollywood|       Kalki 2898-AD| 500.0|
|Tollywood|         Radhe Shyam| 300.0|
|Tollywood|Salaar: Part 1 - ...| 280.0|
|Tollywood|      Devara: Part 1| 250.0|
|Tollywood|Bãhubali 2: The C...| 250.0|
|Tollywood|Bãhubali: The Beg...| 180.0|
|Tollywood|Pushpa: The Rise ...| 150.0|
|Tollywood|       Guntur Kaaram| 150.0|
+---------+--------------------+------+
only showing top 10 rows


In [142]:
# With Spark SQL
spark.sql("select Industry, Title,`Budget in Crores` as Budget \
from boxf_tb \
where Industry = 'Tollywood' \
order by Budget desc limit 10").show()

+---------+--------------------+------+
| Industry|               Title|Budget|
+---------+--------------------+------+
|Tollywood|RRR (Rise Roar Re...| 550.0|
|Tollywood|       Kalki 2898-AD| 500.0|
|Tollywood|           Adipurush| 500.0|
|Tollywood|         Radhe Shyam| 300.0|
|Tollywood|Salaar: Part 1 - ...| 280.0|
|Tollywood|Bãhubali 2: The C...| 250.0|
|Tollywood|      Devara: Part 1| 250.0|
|Tollywood|Bãhubali: The Beg...| 180.0|
|Tollywood|Pushpa: The Rise ...| 150.0|
|Tollywood|       Guntur Kaaram| 150.0|
+---------+--------------------+------+



#### 29 Write query to get films based on budget in Kollywood 

In [143]:
# With Pandas 
p_boxf.query("Industry =='Kollywood'")\
    .groupby(["Industry","Title"])\
    .agg(Budget = ( "Budget in Crores","sum"))\
    .sort_values("Budget", ascending = False)\
    .reset_index().head(10)

,Industry,Title,Budget
0,Kollywood,2,540.0
1,Kollywood,The GOAT,320.0
2,Kollywood,Leo,250.0
3,Kollywood,Ponniyin Selvan: Part One,250.0
4,Kollywood,Indian 2,250.0
5,Kollywood,Ponniyin Selvan: Part Two,250.0
6,Kollywood,Jailer,200.0
7,Kollywood,Vettaiyan,200.0
8,Kollywood,Varisu,200.0
9,Kollywood,Darbar,200.0


In [144]:
# With PySpark
s_boxf.filter(col("Industry")=="Kollywood")\
    .groupby("Industry","Title")\
    .agg(sum("Budget in Crores").alias("Budget"))\
    .sort(desc("Budget")).show(10)

+---------+--------------------+------+
| Industry|               Title|Budget|
+---------+--------------------+------+
|Kollywood|                   2| 540.0|
|Kollywood|            The GOAT| 320.0|
|Kollywood|Ponniyin Selvan: ...| 250.0|
|Kollywood|                 Leo| 250.0|
|Kollywood|Ponniyin Selvan: ...| 250.0|
|Kollywood|            Indian 2| 250.0|
|Kollywood|              Varisu| 200.0|
|Kollywood|              Darbar| 200.0|
|Kollywood|           Vettaiyan| 200.0|
|Kollywood|              Jailer| 200.0|
+---------+--------------------+------+
only showing top 10 rows


In [145]:
# With Spark SQL
spark.sql("select Industry, Title,`Budget in Crores` as Budget \
from boxf_tb \
where Industry = 'Kollywood' \
order by Budget desc limit 10").show()

+---------+--------------------+------+
| Industry|               Title|Budget|
+---------+--------------------+------+
|Kollywood|                   2| 540.0|
|Kollywood|            The GOAT| 320.0|
|Kollywood|Ponniyin Selvan: ...| 250.0|
|Kollywood|                 Leo| 250.0|
|Kollywood|Ponniyin Selvan: ...| 250.0|
|Kollywood|            Indian 2| 250.0|
|Kollywood|              Jailer| 200.0|
|Kollywood|              Varisu| 200.0|
|Kollywood|              Darbar| 200.0|
|Kollywood|           Vettaiyan| 200.0|
+---------+--------------------+------+



#### 30 Write query to get films based on budget in Sandalwood

In [146]:
# With Pandas 
p_boxf.query("Industry =='Sandalwood'")\
    .groupby(["Industry","Title"])\
    .agg(Budget = ( "Budget in Crores","sum"))\
    .sort_values("Budget", ascending = False)\
    .reset_index().head(10)

,Industry,Title,Budget
0,Sandalwood,K.G.F: Chapter 2,100.0
1,Sandalwood,VR (Vikrant Rona),80.0
2,Sandalwood,Kabzaa,70.0
3,Sandalwood,Kurukshetra,60.0
4,Sandalwood,K.G.F: Chapter 1,50.0
5,Sandalwood,The Villain,45.0
6,Sandalwood,Kaatera,45.0
7,Sandalwood,James,40.0
8,Sandalwood,Pailwaan,35.0
9,Sandalwood,Kranti,35.0


In [147]:
# With PySpark
s_boxf.filter(col("Industry")=="Sandalwood")\
    .groupby("Industry","Title")\
    .agg(sum("Budget in Crores").alias("Budget"))\
    .sort(desc("Budget")).show(10)

+----------+-----------------+------+
|  Industry|            Title|Budget|
+----------+-----------------+------+
|Sandalwood| K.G.F: Chapter 2| 100.0|
|Sandalwood|VR (Vikrant Rona)|  80.0|
|Sandalwood|           Kabzaa|  70.0|
|Sandalwood|      Kurukshetra|  60.0|
|Sandalwood| K.G.F: Chapter 1|  50.0|
|Sandalwood|      The Villain|  45.0|
|Sandalwood|          Kaatera|  45.0|
|Sandalwood|            James|  40.0|
|Sandalwood|         Pailwaan|  35.0|
|Sandalwood|           Kranti|  35.0|
+----------+-----------------+------+
only showing top 10 rows


In [148]:
# With Spark SQL
spark.sql("select Industry, Title,`Budget in Crores` as Budget \
from boxf_tb \
where Industry = 'Sandalwood' \
order by Budget desc limit 10").show()

+----------+-----------------+------+
|  Industry|            Title|Budget|
+----------+-----------------+------+
|Sandalwood| K.G.F: Chapter 2| 100.0|
|Sandalwood|VR (Vikrant Rona)|  80.0|
|Sandalwood|           Kabzaa|  70.0|
|Sandalwood|      Kurukshetra|  60.0|
|Sandalwood| K.G.F: Chapter 1|  50.0|
|Sandalwood|      The Villain|  45.0|
|Sandalwood|          Kaatera|  45.0|
|Sandalwood|            James|  40.0|
|Sandalwood|         Pailwaan|  35.0|
|Sandalwood|           Kranti|  35.0|
+----------+-----------------+------+



#### 31 Write query to get films based on budget in Mollywood 

In [149]:
# With Pandas 
p_boxf.query("Industry =='Mollywood'")\
    .groupby(["Industry","Title"])\
    .agg(Budget = ( "Budget in Crores","sum"))\
    .sort_values("Budget", ascending = False)\
    .reset_index().head(10)

,Industry,Title,Budget
0,Mollywood,Marakkar: Lion of the Arabian Sea,100.0
1,Mollywood,Malaikottai Vaaliban,60.0
2,Mollywood,The Goat Life,50.0
3,Mollywood,Odiyan,50.0
4,Mollywood,Mamangam: History of the Brave,45.0
5,Mollywood,King of Kotha,45.0
6,Mollywood,Kayamkulam Kochunni,40.0
7,Mollywood,Lucifer,35.0
8,Mollywood,Kurup,35.0
9,Mollywood,Turbo,35.0


In [150]:
# With PySpark
s_boxf.filter(col("Industry")=="Mollywood")\
    .groupby("Industry","Title")\
    .agg(sum("Budget in Crores").alias("Budget"))\
    .sort(desc("Budget")).show(10)

+---------+--------------------+------+
| Industry|               Title|Budget|
+---------+--------------------+------+
|Mollywood|Marakkar: Lion of...| 100.0|
|Mollywood|Malaikottai Vaaliban|  60.0|
|Mollywood|              Odiyan|  50.0|
|Mollywood|       The Goat Life|  50.0|
|Mollywood|       King of Kotha|  45.0|
|Mollywood|Mamangam: History...|  45.0|
|Mollywood| Kayamkulam Kochunni|  40.0|
|Mollywood|               Turbo|  35.0|
|Mollywood|             Lucifer|  35.0|
|Mollywood|              Trance|  35.0|
+---------+--------------------+------+
only showing top 10 rows


In [151]:
# With Spark SQL
spark.sql("select Industry, Title,`Budget in Crores` as Budget \
from boxf_tb \
where Industry = 'Mollywood' \
order by Budget desc limit 10").show()

+---------+--------------------+------+
| Industry|               Title|Budget|
+---------+--------------------+------+
|Mollywood|Marakkar: Lion of...| 100.0|
|Mollywood|Malaikottai Vaaliban|  60.0|
|Mollywood|       The Goat Life|  50.0|
|Mollywood|              Odiyan|  50.0|
|Mollywood|Mamangam: History...|  45.0|
|Mollywood|       King of Kotha|  45.0|
|Mollywood| Kayamkulam Kochunni|  40.0|
|Mollywood|             Lucifer|  35.0|
|Mollywood|               Kurup|  35.0|
|Mollywood|              Trance|  35.0|
+---------+--------------------+------+



#### 32 Top 5 movies by IMDb rating from Bollowood

In [152]:
# With Pandas
p_boxf.query("Industry == 'Bollywood'")[["Title","IMDb Rating"]]\
    .sort_values("IMDb Rating",ascending = False).head(7).head(5)

,Title,IMDb Rating
121,12th Fail,8.8
79,The Kashmir Files,8.6
512,Sachin,8.5
602,Laapataa Ladies,8.4
524,Dangal,8.3


In [153]:
# With PySpark
s_boxf.filter("Industry == 'Bollywood'")\
    .select("Title","IMDb Rating")\
    .sort(desc("IMDb Rating")).show(5)

+-----------------+-----------+
|            Title|IMDb Rating|
+-----------------+-----------+
|        12th Fail|        8.8|
|The Kashmir Files|        8.6|
|           Sachin|        8.5|
|  Laapataa Ladies|        8.4|
|           Dangal|        8.3|
+-----------------+-----------+
only showing top 5 rows


In [154]:
# With Spark SQL
spark.sql("Select Title,`IMDb Rating` from boxf_tb \
where Industry = 'Bollywood' \
order by `IMDb Rating` desc limit 5").show()

+-----------------+-----------+
|            Title|IMDb Rating|
+-----------------+-----------+
|        12th Fail|        8.8|
|The Kashmir Files|        8.6|
|           Sachin|        8.5|
|  Laapataa Ladies|        8.4|
|           Dangal|        8.3|
+-----------------+-----------+



#### 33 Top 5 movies by IMDb rating from Kollywood 

In [155]:
# With Pandas
p_boxf.query("Industry == 'Kollywood'")[["Title","IMDb Rating"]]\
    .sort_values("IMDb Rating",ascending = False).head(7).head(5)

,Title,IMDb Rating
316,Rocketry: The Nambi Effect,8.7
350,Meiyazhagan,8.5
135,Maharaja,8.5
25,96,8.5
274,Vada Chennai,8.4


In [156]:
# With PySpark
s_boxf.filter("Industry == 'Kollywood'")\
    .select("Title","IMDb Rating")\
    .sort(desc("IMDb Rating")).show(5)

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|Rocketry: The Nam...|        8.7|
|                  96|        8.5|
|            Maharaja|        8.5|
|         Meiyazhagan|        8.5|
|              Asuran|        8.4|
+--------------------+-----------+
only showing top 5 rows


In [157]:
# With Spark SQL
spark.sql("Select Title,`IMDb Rating` from boxf_tb \
where Industry = 'Kollywood' \
order by `IMDb Rating` desc limit 5").show()

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|Rocketry: The Nam...|        8.7|
|                  96|        8.5|
|            Maharaja|        8.5|
|         Meiyazhagan|        8.5|
|              Asuran|        8.4|
+--------------------+-----------+



#### 34 Top 5 movies by IMDb rating from Tollywood

In [158]:
# With Pandas
p_boxf.query("Industry == 'Tollywood'")[["Title","IMDb Rating"]]\
    .sort_values("IMDb Rating",ascending = False).head(7).head(5)

,Title,IMDb Rating
437,Jersey,8.5
84,Sita Ramam,8.5
35,Mahanati,8.4
360,Rangasthalam,8.2
226,Hi Nanna,8.2


In [159]:
# With PySpark
s_boxf.filter("Industry == 'Tollywood'")\
    .select("Title","IMDb Rating")\
    .sort(desc("IMDb Rating")).show(5)

+------------+-----------+
|       Title|IMDb Rating|
+------------+-----------+
|  Sita Ramam|        8.5|
|      Jersey|        8.5|
|    Mahanati|        8.4|
|Rangasthalam|        8.2|
|    Hi Nanna|        8.2|
+------------+-----------+
only showing top 5 rows


In [160]:
# With Spark SQL
spark.sql("Select Title,`IMDb Rating` from boxf_tb \
where Industry = 'Tollywood' \
order by `IMDb Rating` desc limit 5").show()

+------------+-----------+
|       Title|IMDb Rating|
+------------+-----------+
|  Sita Ramam|        8.5|
|      Jersey|        8.5|
|    Mahanati|        8.4|
|Rangasthalam|        8.2|
|    Hi Nanna|        8.2|
+------------+-----------+



#### 35 Top 5 movies by IMDb rating from Sandalwood

In [161]:
# With Pandas
p_boxf.query("Industry == 'Sandalwood'")[["Title","IMDb Rating"]]\
    .sort_values("IMDb Rating",ascending = False).head(7).head(5)

,Title,IMDb Rating
83,777 Charlie,8.7
148,K.G.F: Chapter 1,8.2
149,K.G.F: Chapter 2,8.2
237,Sapta Sagaradaache Ello: Side A,8.2
186,Kantara,8.2


In [162]:
# With PySpark
s_boxf.filter("Industry == 'Sandalwood'")\
    .select("Title","IMDb Rating")\
    .sort(desc("IMDb Rating")).show(5)

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|         777 Charlie|        8.7|
|             Kantara|        8.2|
|    K.G.F: Chapter 1|        8.2|
|    K.G.F: Chapter 2|        8.2|
|Sapta Sagaradaach...|        8.2|
+--------------------+-----------+
only showing top 5 rows


In [163]:
# With Spark SQL
spark.sql("Select Title,`IMDb Rating` from boxf_tb \
where Industry = 'Sandalwood' \
order by `IMDb Rating` desc limit 5").show()

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|         777 Charlie|        8.7|
|    K.G.F: Chapter 2|        8.2|
|    K.G.F: Chapter 1|        8.2|
|             Kantara|        8.2|
|Sapta Sagaradaach...|        8.2|
+--------------------+-----------+



#### 36 Top 5 movies by IMDb rating from Mollywood 

In [164]:
# With Pandas
p_boxf.query("Industry == 'Mollywood'")[["Title","IMDb Rating"]]\
    .sort_values("IMDb Rating",ascending = False).head(7).head(5)

,Title,IMDb Rating
140,Kishkindha Kaandam,8.6
204,Kumbalangi Nights,8.5
189,Jana Gana Mana,8.3
217,2018,8.3
147,Bangalore Days,8.3


In [165]:
# With PySpark
s_boxf.filter("Industry == 'Mollywood'")\
    .select("Title","IMDb Rating")\
    .sort(desc("IMDb Rating")).show(5)

+------------------+-----------+
|             Title|IMDb Rating|
+------------------+-----------+
|Kishkindha Kaandam|        8.6|
| Kumbalangi Nights|        8.5|
|    Bangalore Days|        8.3|
|              2018|        8.3|
|            Premam|        8.3|
+------------------+-----------+
only showing top 5 rows


In [166]:
# With Spark SQL
spark.sql("Select Title,`IMDb Rating` from boxf_tb \
where Industry = 'Mollywood' \
order by `IMDb Rating` desc limit 5").show()

+------------------+-----------+
|             Title|IMDb Rating|
+------------------+-----------+
|Kishkindha Kaandam|        8.6|
| Kumbalangi Nights|        8.5|
|    Bangalore Days|        8.3|
|            Premam|        8.3|
|    Jana Gana Mana|        8.3|
+------------------+-----------+



#### 37 Write a query to get language wise budget? 

In [167]:
# With Pandas
p_boxf.merge(p_lan,on = "LanguageID",how="right")\
    .groupby("Language").agg(Total_Budget=("Budget in Crores","sum"))\
    .sort_values("Total_Budget",ascending = False).reset_index()

,Language,Total_Budget
0,Hindi,14535.0
1,Telugu,9144.5
2,Tamil,8682.0
3,Malayalam,1342.5
4,Kannada,949.0


In [168]:
# With PySpark
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"right")\
    .groupby("Language").agg(sum("Budget in Crores").alias("Total_Budget"))\
    .sort(desc("Total_Budget")).show()

+---------+------------+
| Language|Total_Budget|
+---------+------------+
|    Hindi|     14535.0|
|   Telugu|      9144.5|
|    Tamil|      8682.0|
|Malayalam|      1342.5|
|  Kannada|       949.0|
+---------+------------+



In [169]:
# With Spark SQL
spark.sql("Select Language, sum(`Budget in Crores`) \
as Total_Budget from boxf_tb boxf \
right join lan_tb lang on lang.LanguageID = boxf.LanguageID \
group by Language \
order by Total_Budget desc").show()

+---------+------------+
| Language|Total_Budget|
+---------+------------+
|    Hindi|     14535.0|
|   Telugu|      9144.5|
|    Tamil|      8682.0|
|Malayalam|      1342.5|
|  Kannada|       949.0|
+---------+------------+



#### 38 Write a query to get language wise how many directors are there? 

In [170]:
# With Pandas
p_boxf.merge(p_lan,on="LanguageID",how="inner")\
    .merge(p_dir,left_on = "DirectorID",\
           right_on = "Director ID",how="inner")\
    .groupby("Language").agg(Total_Directors=("DirectorID","count"))\
    .sort_values("Total_Directors",ascending = False)\
    .reset_index().head(10)

,Language,Total_Directors
0,Hindi,182
1,Telugu,166
2,Tamil,140
3,Malayalam,84
4,Kannada,32


In [171]:
# With PySpark
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .join(s_dir, s_boxf["DirectorID"]==s_dir["Director ID"],"inner")\
    .groupby("Language").agg(count("DirectorID").alias("Total_Directors"))\
    .sort(desc("Total_Directors")).show()

+---------+---------------+
| Language|Total_Directors|
+---------+---------------+
|    Hindi|            182|
|   Telugu|            166|
|    Tamil|            140|
|Malayalam|             84|
|  Kannada|             32|
+---------+---------------+



In [172]:
# With Spark SQL
spark.sql("select Language, count(`Director ID`) \
as Total_Directors from boxf_tb boxf \
inner join lan_tb lan on lan.LanguageID = boxf.LanguageID \
inner join dir_tb dir on dir.`Director ID` = boxf.DirectorID \
group by Language \
order by Total_Directors desc").show()

+---------+---------------+
| Language|Total_Directors|
+---------+---------------+
|    Hindi|            182|
|   Telugu|            166|
|    Tamil|            140|
|Malayalam|             84|
|  Kannada|             32|
+---------+---------------+



#### 39 Write a query to get language wise worldwide total collection ? 

In [173]:
# With Pandas
p_boxf.merge(p_lan,on="LanguageID",how="inner")\
    .groupby("Language")\
    .agg(Total_WWC=("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index()

,Language,Total_WWC
0,Hindi,38610.79
1,Telugu,18065.27
2,Tamil,15235.99
3,Malayalam,4597.32
4,Kannada,3369.42


In [174]:
# With PySpark
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"right")\
    .groupby("Language").agg(sum("Worldwide Collection  in Crores")\
        .alias("Total_WWC")).sort(desc("Total_WWC")).show()

+---------+------------------+
| Language|         Total_WWC|
+---------+------------------+
|    Hindi|          38610.79|
|   Telugu|18065.270000000004|
|    Tamil|          15235.99|
|Malayalam|           4597.32|
|  Kannada|           3369.42|
+---------+------------------+



In [175]:
# With Spark SQL
spark.sql("select Language, sum(`Worldwide Collection  in Crores`) \
as Total_WWC from boxf_tb boxf \
right join lan_tb lan on lan.LanguageID =boxf.LanguageID \
group by Language \
order by 2 desc").show()

+---------+------------------+
| Language|         Total_WWC|
+---------+------------------+
|    Hindi|          38610.79|
|   Telugu|18065.270000000004|
|    Tamil|          15235.99|
|Malayalam|           4597.32|
|  Kannada|           3369.42|
+---------+------------------+



#### 40 Write a query to get language, lead actor/actress wise films they acted? 

In [176]:
# With Pandas
p_boxf.merge(p_lan,on="LanguageID",how="right")\
    .groupby(["Language","Lead Actor/Actress"])\
    .agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False)\
    .reset_index().head(15)

,Language,Lead Actor/Actress,Total_Films
0,Hindi,Akshay Kumar,20
1,Hindi,Ajay Devgn,14
2,Tamil,Dhanush,13
3,Malayalam,Mammootty,13
4,Telugu,Nani,12
5,Tamil,Joseph Vijay,12
6,Hindi,Salman Khan,12
7,Tamil,Sivakarthikeyan,11
8,Malayalam,Mohanlal,11
9,Telugu,Ravi Teja,10


In [177]:
# With PySpark
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"right")\
    .groupby("Language","Lead Actor/Actress").agg(count("FilmID")\
        .alias("Total_Films")).sort(desc("Total_Films")).show(15)

+---------+--------------------+-----------+
| Language|  Lead Actor/Actress|Total_Films|
+---------+--------------------+-----------+
|    Hindi|        Akshay Kumar|         20|
|    Hindi|          Ajay Devgn|         14|
|Malayalam|           Mammootty|         13|
|    Tamil|             Dhanush|         13|
|    Tamil|        Joseph Vijay|         12|
|    Hindi|         Salman Khan|         12|
|   Telugu|                Nani|         12|
|    Tamil|     Sivakarthikeyan|         11|
|Malayalam|            Mohanlal|         11|
|   Telugu|         Mahesh Babu|         10|
|   Telugu|           Ravi Teja|         10|
|    Tamil|         Rajinikanth|         10|
|    Tamil|              Karthi|         10|
|Malayalam|Prithviraj Sukumaran|          9|
|    Tamil|              Vikram|          9|
+---------+--------------------+-----------+
only showing top 15 rows


In [178]:
# With Spark SQL
spark.sql("select Language, `Lead Actor/Actress`,\
count(FilmID) as Total_Films from boxf_tb boxf \
right join lan_tb lan on lan.LanguageID = boxf.LanguageID \
group by Language, `Lead Actor/Actress` \
order by 3 desc limit 15").show()

+---------+--------------------+-----------+
| Language|  Lead Actor/Actress|Total_Films|
+---------+--------------------+-----------+
|    Hindi|        Akshay Kumar|         20|
|    Hindi|          Ajay Devgn|         14|
|Malayalam|           Mammootty|         13|
|    Tamil|             Dhanush|         13|
|    Tamil|        Joseph Vijay|         12|
|    Hindi|         Salman Khan|         12|
|   Telugu|                Nani|         12|
|    Tamil|     Sivakarthikeyan|         11|
|Malayalam|            Mohanlal|         11|
|   Telugu|         Mahesh Babu|         10|
|   Telugu|           Ravi Teja|         10|
|    Tamil|         Rajinikanth|         10|
|    Tamil|              Karthi|         10|
|Malayalam|Prithviraj Sukumaran|          9|
|    Tamil|              Vikram|          9|
+---------+--------------------+-----------+



#### 41 Write a query to get language, year wise films released? 

In [179]:
# With Pandas
p_boxf.merge(p_lan, on = "LanguageID", how="inner")\
    .groupby(["Language","Year"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False).reset_index().head(10)

,Language,Year,Total_Films
0,Telugu,2023,25
1,Telugu,2017,24
2,Hindi,2017,24
3,Hindi,2022,24
4,Hindi,2023,23
5,Telugu,2018,23
6,Telugu,2022,22
7,Tamil,2018,22
8,Hindi,2024,20
9,Telugu,2019,19


In [180]:
s_boxf = s_boxf.withColumn("Year", year(col("Release Date")))

In [181]:
# With PySpark
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .groupby("Language","Year").agg(count("FilmID").alias("Total_Films"))\
    .sort(desc("Total_Films")).show(10)

+--------+----+-----------+
|Language|Year|Total_Films|
+--------+----+-----------+
|  Telugu|2023|         25|
|   Hindi|2022|         24|
|   Hindi|2017|         24|
|  Telugu|2017|         24|
|   Hindi|2023|         23|
|  Telugu|2018|         23|
|  Telugu|2022|         22|
|   Tamil|2018|         22|
|   Hindi|2024|         20|
|   Tamil|2019|         19|
+--------+----+-----------+
only showing top 10 rows


In [182]:
# With Spark SQL
spark.sql("select Language,YEAR(try_to_timestamp(`Release Date`, 'dd-MMM-yy'))\
as Yr , count(FilmID) as Total_Films from boxf_tb boxf \
inner join lan_tb lan on lan.LanguageID = Boxf.LanguageID \
group by 1,2 \
order by 3 desc limit 10").show()

+--------+----+-----------+
|Language|  Yr|Total_Films|
+--------+----+-----------+
|  Telugu|2023|         25|
|   Hindi|2022|         24|
|   Hindi|2017|         24|
|  Telugu|2017|         24|
|   Hindi|2023|         23|
|  Telugu|2018|         23|
|   Tamil|2018|         22|
|  Telugu|2022|         22|
|   Hindi|2024|         20|
|   Tamil|2019|         19|
+--------+----+-----------+



#### 42 Write a query to get films which was not released on overseas? 

In [183]:
# With PySpark
p_boxf.loc[p_boxf["Overseas Collection in Crores"]==0,\
    ["Title","Overseas Collection in Crores"]].reset_index()

,index,Title,Overseas Collection in Crores
0,4,Maanikya,0.0
1,13,Ugramm,0.0
2,36,Tagaru,0.0
3,57,Hebbuli,0.0
4,169,Anjaniputra,0.0
5,171,Chakravarthy,0.0
6,446,118,0.0


In [184]:
# With PySpark
s_boxf.select("Title","Overseas Collection in Crores")\
    .filter(col("Overseas Collection in Crores")==0).show()

+------------+-----------------------------+
|       Title|Overseas Collection in Crores|
+------------+-----------------------------+
|    Maanikya|                          0.0|
|      Ugramm|                          0.0|
|      Tagaru|                          0.0|
|     Hebbuli|                          0.0|
| Anjaniputra|                          0.0|
|Chakravarthy|                          0.0|
|         118|                          0.0|
+------------+-----------------------------+



In [185]:
# With Spark SQL
spark.sql("select Title,`Overseas Collection in Crores` \
from boxf_tb where `Overseas Collection in Crores` = 0 ").show()

+------------+-----------------------------+
|       Title|Overseas Collection in Crores|
+------------+-----------------------------+
|    Maanikya|                          0.0|
|      Ugramm|                          0.0|
|      Tagaru|                          0.0|
|     Hebbuli|                          0.0|
| Anjaniputra|                          0.0|
|Chakravarthy|                          0.0|
|         118|                          0.0|
+------------+-----------------------------+



#### 43 Write a query to get language wise top 3 longest runtime moves? 

In [186]:
p_boxf.merge(p_lan,on = "LanguageID",how="inner")[["Title","Language","Runtime (mins)"]]\
    .sort_values(["Language","Runtime (mins)"],ascending=[True,False])\
    .assign(row_number=lambda x:x.groupby("Language").cumcount()+1)\
    .query("row_number <=3")

,Title,Language,Runtime (mins),row_number
114,Animal,Hindi,204,1
176,M.S. Dhoni: The Untold Story,Hindi,184,2
594,Maidaan,Hindi,181,3
103,Avane Srimannarayana,Kannada,186,1
201,Kurukshetra,Kannada,185,2
225,Kaatera,Kannada,183,3
178,Marakkar: Lion of the Arabian Sea,Malayalam,181,1
92,Ayyappanum Koshiyum,Malayalam,177,2
98,Lucifer,Malayalam,175,3
39,I,Tamil,188,1


In [187]:
# With PySpark
window_spec = Window.partitionBy("Language").orderBy(desc("Runtime (mins)"))
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .withColumn("Rnk",row_number().over(window_spec))\
    .filter(col("Rnk")<=3).select("Title","Language","Runtime (mins)","Rnk")\
    .sort("Language",desc("Runtime (mins)")).show()

+--------------------+---------+--------------+---+
|               Title| Language|Runtime (mins)|Rnk|
+--------------------+---------+--------------+---+
|              Animal|    Hindi|           204|  1|
|M.S. Dhoni: The U...|    Hindi|           184|  2|
|             Maidaan|    Hindi|           181|  3|
|Avane Srimannarayana|  Kannada|           186|  1|
|         Kurukshetra|  Kannada|           185|  2|
|             Kaatera|  Kannada|           183|  3|
|Marakkar: Lion of...|Malayalam|           181|  1|
| Ayyappanum Koshiyum|Malayalam|           177|  2|
|             Lucifer|Malayalam|           175|  3|
|                   I|    Tamil|           188|  1|
|               Jilla|    Tamil|           185|  2|
|               Cobra|    Tamil|           183|  3|
|RRR (Rise Roar Re...|   Telugu|           187|  1|
|         Arjun Reddy|   Telugu|           182|  2|
|       Kalki 2898-AD|   Telugu|           180|  3|
+--------------------+---------+--------------+---+



In [188]:
# With Spark SQL
spark.sql("select * from (select Language,Title,`Runtime (mins)`,\
row_number() over(partition by Language order by `Runtime (mins)` desc)as rnk \
from boxf_tb boxf \
inner join lan_tb lan on lan.LanguageID = boxf.LanguageID) a \
where rnk <= 3 \
order by Language, `Runtime (mins)` desc").show()

+---------+--------------------+--------------+---+
| Language|               Title|Runtime (mins)|rnk|
+---------+--------------------+--------------+---+
|    Hindi|              Animal|           204|  1|
|    Hindi|M.S. Dhoni: The U...|           184|  2|
|    Hindi|             Maidaan|           181|  3|
|  Kannada|Avane Srimannarayana|           186|  1|
|  Kannada|         Kurukshetra|           185|  2|
|  Kannada|             Kaatera|           183|  3|
|Malayalam|Marakkar: Lion of...|           181|  1|
|Malayalam| Ayyappanum Koshiyum|           177|  2|
|Malayalam|             Lucifer|           175|  3|
|    Tamil|                   I|           188|  1|
|    Tamil|               Jilla|           185|  2|
|    Tamil|               Cobra|           183|  3|
|   Telugu|RRR (Rise Roar Re...|           187|  1|
|   Telugu|         Arjun Reddy|           182|  2|
|   Telugu|       Kalki 2898-AD|           180|  3|
+---------+--------------------+--------------+---+



#### 44 Write a query to get language wise bottom 5 shortest runtime moves? 

In [189]:
# With Pandas
p_boxf.merge(p_lan,on = "LanguageID", how= "inner")[["Title","Language","Runtime (mins)"]]\
    .sort_values(["Language","Runtime (mins)"],ascending = [True,True])\
    .assign(rnk =lambda x : x.groupby("Language").cumcount()+1)\
    .query('rnk <=3')

,Title,Language,Runtime (mins),rnk
599,Kill,Hindi,105,1
497,Raksha Bandhan,Hindi,108,2
560,Bhoot: Part One - The Haunted Ship,Hindi,114,3
185,Salaga,Kannada,124,1
36,Tagaru,Kannada,129,2
13,Ugramm,Kannada,132,3
87,Malikappuram,Malayalam,121,1
42,Sudani from Nigeria,Malayalam,123,2
143,Vaazha: Biopic of a Billion Boys,Malayalam,125,3
279,Kochadaiiyaan,Tamil,118,1


In [190]:
# With PySpark
window_spec = Window.partitionBy("Language").orderBy("Runtime (mins)")
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .withColumn("rnk",row_number().over(window_spec))\
    .filter(col("rnk")<=3).select("Title","Language","Runtime (mins)","rnk")\
    .sort("Language",asc("Runtime (mins)")).show()

+--------------------+---------+--------------+---+
|               Title| Language|Runtime (mins)|rnk|
+--------------------+---------+--------------+---+
|                Kill|    Hindi|           105|  1|
|      Raksha Bandhan|    Hindi|           108|  2|
|Bhoot: Part One -...|    Hindi|           114|  3|
|              Salaga|  Kannada|           124|  1|
|              Tagaru|  Kannada|           129|  2|
|              Ugramm|  Kannada|           132|  3|
|        Malikappuram|Malayalam|           121|  1|
| Sudani from Nigeria|Malayalam|           123|  2|
|Vaazha: Biopic of...|Malayalam|           125|  3|
|       Kochadaiiyaan|    Tamil|           118|  1|
|   Dhilluku Dhuddu 2|    Tamil|           119|  2|
|      Kadaram Kondan|    Tamil|           121|  3|
|    The Ghazi Attack|   Telugu|           116|  1|
|                Guru|   Telugu|           116|  2|
|   Raju Gari Gadhi 2|   Telugu|           117|  3|
+--------------------+---------+--------------+---+



In [191]:
# With Spark SQL
spark.sql("select * from (select Title, Language,`Runtime (mins)`,\
row_number() over(partition by Language order by `Runtime (mins)`)as rnk \
from boxf_tb boxf \
inner join lan_tb lan on lan.LanguageID = boxf.LanguageID) a \
where rnk <= 3 \
order by Language, `Runtime (mins)`").show()

+--------------------+---------+--------------+---+
|               Title| Language|Runtime (mins)|rnk|
+--------------------+---------+--------------+---+
|                Kill|    Hindi|           105|  1|
|      Raksha Bandhan|    Hindi|           108|  2|
|Bhoot: Part One -...|    Hindi|           114|  3|
|              Salaga|  Kannada|           124|  1|
|              Tagaru|  Kannada|           129|  2|
|              Ugramm|  Kannada|           132|  3|
|        Malikappuram|Malayalam|           121|  1|
| Sudani from Nigeria|Malayalam|           123|  2|
|Vaazha: Biopic of...|Malayalam|           125|  3|
|       Kochadaiiyaan|    Tamil|           118|  1|
|   Dhilluku Dhuddu 2|    Tamil|           119|  2|
|      Kadaram Kondan|    Tamil|           121|  3|
|    The Ghazi Attack|   Telugu|           116|  1|
|                Guru|   Telugu|           116|  2|
|   Raju Gari Gadhi 2|   Telugu|           117|  3|
+--------------------+---------+--------------+---+



#### 45 Write a query to get language wise top 5 films based first day collections? 

In [192]:
p_boxf.merge(p_lan, on = "LanguageID",how = "inner")\
    [["Title","Language","First Day Collection Worldwide  in Crores"]]\
    .sort_values(["Language","First Day Collection Worldwide  in Crores"],ascending = [True,False])\
    .assign(rnk=lambda x: x.groupby("Language").cumcount()+1)\
    .query("rnk <=3")

,Title,Language,First Day Collection Worldwide in Crores,rnk
572,Jawan,Hindi,129.10,1
114,Animal,Hindi,116.00,2
573,Pathaan,Hindi,104.80,3
149,K.G.F: Chapter 2,Kannada,159.00,1
187,VR (Vikrant Rona),Kannada,25.50,2
148,K.G.F: Chapter 1,Kannada,25.00,3
178,Marakkar: Lion of the Arabian Sea,Malayalam,20.30,1
159,Odiyan,Malayalam,18.30,2
180,Kurup,Malayalam,18.30,3
116,Leo,Tamil,142.70,1


In [193]:
# With PySpark
window_spec = Window.partitionBy("Language").orderBy(desc("First Day Collection Worldwide  in Crores"))
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .withColumn("rnk",row_number().over(window_spec))\
    .filter(col("rnk")<=3).select("Title","Language","First Day Collection Worldwide  in Crores","rnk").show()

+--------------------+---------+-----------------------------------------+---+
|               Title| Language|First Day Collection Worldwide  in Crores|rnk|
+--------------------+---------+-----------------------------------------+---+
|               Jawan|    Hindi|                                    129.1|  1|
|              Animal|    Hindi|                                    116.0|  2|
|             Pathaan|    Hindi|                                    104.8|  3|
|    K.G.F: Chapter 2|  Kannada|                                    159.0|  1|
|   VR (Vikrant Rona)|  Kannada|                                     25.5|  2|
|    K.G.F: Chapter 1|  Kannada|                                     25.0|  3|
|Marakkar: Lion of...|Malayalam|                                     20.3|  1|
|              Odiyan|Malayalam|                                     18.3|  2|
|               Kurup|Malayalam|                                     18.3|  3|
|                 Leo|    Tamil|                    

In [194]:
# With Spark SQL
spark.sql("select * from (select Title,Language,\
`First Day Collection Worldwide  in Crores`,row_number() \
over(partition by Language order by `First Day Collection Worldwide  in Crores` desc) as rnk \
from boxf_tb boxf inner join lan_tb lan on lan.LanguageID = boxf.LanguageID) a \
where rnk <= 3 \
order by Language, `First Day Collection Worldwide  in Crores` desc").show()

+--------------------+---------+-----------------------------------------+---+
|               Title| Language|First Day Collection Worldwide  in Crores|rnk|
+--------------------+---------+-----------------------------------------+---+
|               Jawan|    Hindi|                                    129.1|  1|
|              Animal|    Hindi|                                    116.0|  2|
|             Pathaan|    Hindi|                                    104.8|  3|
|    K.G.F: Chapter 2|  Kannada|                                    159.0|  1|
|   VR (Vikrant Rona)|  Kannada|                                     25.5|  2|
|    K.G.F: Chapter 1|  Kannada|                                     25.0|  3|
|Marakkar: Lion of...|Malayalam|                                     20.3|  1|
|              Odiyan|Malayalam|                                     18.3|  2|
|               Kurup|Malayalam|                                     18.3|  3|
|                 Leo|    Tamil|                    

#### 46 Write a query to get language wise top 5 films based India gross collections?

In [195]:
# with Pandas
p_boxf.merge(p_lan,on = "LanguageID", how="inner")[["Title","Language","First Day Collection Worldwide  in Crores"]]\
    .sort_values(["Language","First Day Collection Worldwide  in Crores"],ascending = [True,True])\
    .assign(rnk = lambda x: x.groupby("Language").cumcount()+1)\
    .query("rnk <= 3") 

,Title,Language,First Day Collection Worldwide in Crores,rnk
602,Laapataa Ladies,Hindi,1.40,1
121,12th Fail,Hindi,1.50,2
599,Kill,Hindi,1.50,3
13,Ugramm,Kannada,2.00,1
237,Sapta Sagaradaache Ello: Side A,Kannada,2.30,2
207,Odeya,Kannada,2.50,3
140,Kishkindha Kaandam,Malayalam,0.50,1
227,Romancham,Malayalam,0.50,2
87,Malikappuram,Malayalam,0.52,3
29,Ratsasan,Tamil,0.50,1


In [196]:
# With PySpark
window_spec = Window.partitionBy("Language").orderBy(asc("First Day Collection Worldwide  in Crores"))
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .withColumn("rnk",row_number().over(window_spec)).filter(col("rnk")<=3)\
    .select("Title","Language","First Day Collection Worldwide  in Crores","rnk").show()

+--------------------+---------+-----------------------------------------+---+
|               Title| Language|First Day Collection Worldwide  in Crores|rnk|
+--------------------+---------+-----------------------------------------+---+
|     Laapataa Ladies|    Hindi|                                      1.4|  1|
|           12th Fail|    Hindi|                                      1.5|  2|
|                Kill|    Hindi|                                      1.5|  3|
|              Ugramm|  Kannada|                                      2.0|  1|
|Sapta Sagaradaach...|  Kannada|                                      2.3|  2|
|               Odeya|  Kannada|                                      2.5|  3|
|  Kishkindha Kaandam|Malayalam|                                      0.5|  1|
|           Romancham|Malayalam|                                      0.5|  2|
|        Malikappuram|Malayalam|                                     0.52|  3|
|            Ratsasan|    Tamil|                    

In [197]:
# With Spark SQL
spark.sql("select * from (select Title,Language,`First Day Collection Worldwide  in Crores`,\
row_number() over(partition by Language order by `First Day Collection Worldwide  in Crores`) as rnk \
from boxf_tb boxf inner join lan_tb lan on lan.LanguageID = boxf.LanguageID) a \
where rnk <= 3 \
order by Language,`First Day Collection Worldwide  in Crores`").show()

+--------------------+---------+-----------------------------------------+---+
|               Title| Language|First Day Collection Worldwide  in Crores|rnk|
+--------------------+---------+-----------------------------------------+---+
|     Laapataa Ladies|    Hindi|                                      1.4|  1|
|           12th Fail|    Hindi|                                      1.5|  2|
|                Kill|    Hindi|                                      1.5|  3|
|              Ugramm|  Kannada|                                      2.0|  1|
|Sapta Sagaradaach...|  Kannada|                                      2.3|  2|
|               Odeya|  Kannada|                                      2.5|  3|
|  Kishkindha Kaandam|Malayalam|                                      0.5|  1|
|           Romancham|Malayalam|                                      0.5|  2|
|        Malikappuram|Malayalam|                                     0.52|  3|
|            Ratsasan|    Tamil|                    

#### 47 Write a query to get language, Director wise films count? 

In [198]:
# With Pandas
p_boxf.merge(p_lan,on="LanguageID",how="inner")\
    .merge(p_dir,left_on = "DirectorID",\
           right_on = "Director ID",how="inner")\
    .groupby(["Language","Director"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False)\
    .reset_index().head(10)

,Language,Director,Total_Films
0,Hindi,Rohit Shetty,6
1,Telugu,Trivikram Srinivas,6
2,Telugu,Anil Ravipudi,5
3,Tamil,Siva,5
4,Telugu,Boyapati Srinu,5
5,Telugu,Maruthi Dasari,4
6,Telugu,Koratala Siva,4
7,Telugu,K.S. Ravindra,4
8,Telugu,Sukumar,4
9,Telugu,Kalyan Krishna,4


In [199]:
# With PySpark
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .join(s_dir, s_boxf["DirectorID"]==s_dir["Director ID"],"inner")\
    .groupby("Language","Director").agg(count("FilmID").alias("Total_Films"))\
    .sort(desc("Total_Films")).show(10)

+--------+------------------+-----------+
|Language|          Director|Total_Films|
+--------+------------------+-----------+
|  Telugu|Trivikram Srinivas|          6|
|   Hindi|      Rohit Shetty|          6|
|   Tamil|              Siva|          5|
|  Telugu|     Anil Ravipudi|          5|
|  Telugu|    Boyapati Srinu|          5|
|   Tamil|         Sundar C.|          4|
|   Hindi|   Siddharth Anand|          4|
|  Telugu|     K.S. Ravindra|          4|
|  Telugu|     Koratala Siva|          4|
|  Telugu|    Kalyan Krishna|          4|
+--------+------------------+-----------+
only showing top 10 rows


In [200]:
# With Spark SQL
spark.sql("select Language,Director, count(FilmID) \
as Total_Films from boxf_tb boxf \
inner join lan_tb lan on lan.LanguageID = boxf.LanguageID \
inner join dir_tb dir on dir.`Director ID` = boxf.DirectorID \
group by Language,Director \
order by Total_Films desc limit 10").show()

+--------+------------------+-----------+
|Language|          Director|Total_Films|
+--------+------------------+-----------+
|   Hindi|      Rohit Shetty|          6|
|  Telugu|Trivikram Srinivas|          6|
|   Tamil|              Siva|          5|
|  Telugu|     Anil Ravipudi|          5|
|  Telugu|    Boyapati Srinu|          5|
|   Hindi|   Ali Abbas Zafar|          4|
|   Hindi|        Mohit Suri|          4|
|  Telugu|    Maruthi Dasari|          4|
|   Tamil|         H. Vinoth|          4|
|   Hindi|      Remo D'Souza|          4|
+--------+------------------+-----------+



#### 48 Write a query to get language wise OTT platofrm wise fims count? 

In [201]:
# With Pandas
p_boxf.merge(p_lan,on="LanguageID",how="inner")\
    .groupby(["Language","OTT Platform"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False)\
    .reset_index().head(10)

,Language,OTT Platform,Total_Films
0,Telugu,Amazon Prime Video,119
1,Tamil,Amazon Prime Video,100
2,Hindi,Amazon Prime Video,78
3,Malayalam,Amazon Prime Video,60
4,Hindi,Netflix,52
5,Hindi,Disney+ Hotstar,29
6,Kannada,Amazon Prime Video,27
7,Telugu,Netflix,23
8,Tamil,Netflix,19
9,Hindi,ZEE5,18


In [202]:
# With PySpark
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .groupby("Language","OTT Platform").agg(count("FilmID").alias("Total_Films"))\
    .sort(desc("Total_Films")).show(10)

+---------+------------------+-----------+
| Language|      OTT Platform|Total_Films|
+---------+------------------+-----------+
|   Telugu|Amazon Prime Video|        119|
|    Tamil|Amazon Prime Video|        100|
|    Hindi|Amazon Prime Video|         78|
|Malayalam|Amazon Prime Video|         60|
|    Hindi|           Netflix|         52|
|    Hindi|   Disney+ Hotstar|         29|
|  Kannada|Amazon Prime Video|         27|
|   Telugu|           Netflix|         23|
|    Tamil|           Netflix|         19|
|    Hindi|              Zee5|         18|
+---------+------------------+-----------+
only showing top 10 rows


In [203]:
# With Spark SQL
spark.sql("select Language,`OTT Platform`, count(FilmID) \
as Total_Films from boxf_tb boxf \
inner join lan_tb lan on lan.LanguageID = boxf.LanguageID \
group by 1,2 \
order by 3 desc limit 10").show()

+---------+------------------+-----------+
| Language|      OTT Platform|Total_Films|
+---------+------------------+-----------+
|   Telugu|Amazon Prime Video|        119|
|    Tamil|Amazon Prime Video|        100|
|    Hindi|Amazon Prime Video|         78|
|Malayalam|Amazon Prime Video|         60|
|    Hindi|           Netflix|         52|
|    Hindi|   Disney+ Hotstar|         29|
|  Kannada|Amazon Prime Video|         27|
|   Telugu|           Netflix|         23|
|    Tamil|           Netflix|         19|
|    Hindi|              Zee5|         18|
+---------+------------------+-----------+



#### 49 What are top 10 films based language and first day collection? 

In [204]:
# With Pandas
p_boxf.merge(p_lan,on = "LanguageID",how= 'inner')[['Title','Language','First Day Collection Worldwide  in Crores']]\
    .sort_values(['Language','First Day Collection Worldwide  in Crores'],ascending = [True,False])\
    .assign(rnk = lambda x:x.groupby("Language").cumcount()+1).query("rnk <=10")

,Title,Language,First Day Collection Worldwide in Crores,rnk
572,Jawan,Hindi,129.10,1
114,Animal,Hindi,116.00,2
573,Pathaan,Hindi,104.80,3
574,Tiger 3,Hindi,94.00,4
127,Stree 2: Sarkate Ka Aatank,Hindi,80.00,5
484,Thugs of Hindostan,Hindi,76.00,6
199,War,Hindi,74.30,7
27,Sultan,Hindi,71.30,8
544,Brahmastra Part One: Shiva,Hindi,70.00,9
519,Prem Ratan Dhan Payo,Hindi,66.20,10


In [205]:
# With PySpark
window_spec = Window.partitionBy("Language").orderBy(desc("First Day Collection Worldwide  in Crores"))
s_boxf.join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .withColumn("rnk",row_number().over(window_spec)).filter(col("rnk")<=10)\
    .select('Title','Language','First Day Collection Worldwide  in Crores','rnk').show(50)

+--------------------+---------+-----------------------------------------+---+
|               Title| Language|First Day Collection Worldwide  in Crores|rnk|
+--------------------+---------+-----------------------------------------+---+
|               Jawan|    Hindi|                                    129.1|  1|
|              Animal|    Hindi|                                    116.0|  2|
|             Pathaan|    Hindi|                                    104.8|  3|
|             Tiger 3|    Hindi|                                     94.0|  4|
|Stree 2: Sarkate ...|    Hindi|                                     80.0|  5|
|  Thugs of Hindostan|    Hindi|                                     76.0|  6|
|                 War|    Hindi|                                     74.3|  7|
|              Sultan|    Hindi|                                     71.3|  8|
|Brahmastra Part O...|    Hindi|                                     70.0|  9|
|Prem Ratan Dhan Payo|    Hindi|                    

In [206]:
# With Spark SQL
spark.sql("select * from (select Title, Language,`First Day Collection Worldwide  in Crores`,\
row_number() over(Partition by Language order by `First Day Collection Worldwide  in Crores` desc) as rnk \
from boxf_tb boxf  inner join lan_tb lan on lan.LanguageID = boxf.LanguageID)a \
where rnk <= 10 \
order by Language,`First Day Collection Worldwide  in Crores` desc").show()

+--------------------+--------+-----------------------------------------+---+
|               Title|Language|First Day Collection Worldwide  in Crores|rnk|
+--------------------+--------+-----------------------------------------+---+
|               Jawan|   Hindi|                                    129.1|  1|
|              Animal|   Hindi|                                    116.0|  2|
|             Pathaan|   Hindi|                                    104.8|  3|
|             Tiger 3|   Hindi|                                     94.0|  4|
|Stree 2: Sarkate ...|   Hindi|                                     80.0|  5|
|  Thugs of Hindostan|   Hindi|                                     76.0|  6|
|                 War|   Hindi|                                     74.3|  7|
|              Sultan|   Hindi|                                     71.3|  8|
|Brahmastra Part O...|   Hindi|                                     70.0|  9|
|Prem Ratan Dhan Payo|   Hindi|                                 

#### 50 Write a query to get director wise number of fims released in from year 2017 to 2019

In [207]:
# With Pandas 
p_boxf.merge(p_dir,left_on ="DirectorID",right_on="Director ID",how="right")\
    .query("Year in [2017,2018,2019]").groupby(["Director","Year"])\
    .agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False)\
    .reset_index().head(10)

,Director,Year,Total_Films
0,Trivikram Srinivas,2018,2
1,A.R. Murugadoss,2017,1
2,S Krishna,2019,1
3,Rajkumar Hirani,2018,1
4,Ram Kumar,2018,1
5,Rambhala,2019,1
6,Ramesh Varma,2019,1
7,Ravi Udyawar,2017,1
8,Reema Kagti,2018,1
9,Remo D'Souza,2018,1


In [208]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"inner")\
    .filter(col("Year").isin(2017,2018,2019)).groupby("Director","Year")\
    .agg(count("FilmID").alias("Total_Films")).sort(desc("Total_Films"))\
    .show(10)

+--------------------+----+-----------+
|            Director|Year|Total_Films|
+--------------------+----+-----------+
|  Trivikram Srinivas|2018|          2|
|Puri Jagannadh, A...|2019|          1|
|     Abhishek Varman|2019|          1|
|       Milan Luthria|2017|          1|
|          K.V. Anand|2017|          1|
|          Milind Rau|2017|          1|
|     Rajkumar Hirani|2018|          1|
|    Shashank Khaitan|2017|          1|
|     Ali Abbas Zafar|2019|          1|
|      Suresh Triveni|2017|          1|
+--------------------+----+-----------+
only showing top 10 rows


In [209]:
# With Spark SQL
spark.sql("""select Director, YEAR(try_to_timestamp(`Release Date`,\
'dd-MMM-yy')) as YR,count(FilmID) as Total_Films from boxf_tb boxf \
right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
group by Director, YR \
having YEAR(try_to_timestamp(`Release Date`,'dd-MMM-yy'))\
between 2017 and 2019 \
order by 3 desc limit 10""").show()

+--------------------+----+-----------+
|            Director|  YR|Total_Films|
+--------------------+----+-----------+
|  Trivikram Srinivas|2018|          2|
|Puri Jagannadh, A...|2019|          1|
|     Abhishek Varman|2019|          1|
|       Milan Luthria|2017|          1|
|          K.V. Anand|2017|          1|
|          Milind Rau|2017|          1|
|     Rajkumar Hirani|2018|          1|
|    Shashank Khaitan|2017|          1|
|     Ali Abbas Zafar|2019|          1|
|      Suresh Triveni|2017|          1|
+--------------------+----+-----------+



#### 51 Write a query to get director wise world wide collections? 

In [210]:
# With Pandas
p_boxf.merge(p_dir,left_on = "DirectorID",right_on="Director ID",how="right")\
    .groupby("Director").agg(Total_WWC=("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index().head(10)

,Director,Total_WWC
0,S.S. Rajamouli,3668.00
1,Siddharth Anand,2229.45
2,Nitesh Tiwari,2122.30
3,Prashanth Neel,2101.75
4,Atlee,1865.35
5,Rajkumar Hirani,1834.50
6,Rohit Shetty,1656.60
7,Lokesh Kanagaraj,1350.13
8,Sandeep Reddy Vanga,1342.00
9,Kabir Khan,1220.55


In [211]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director").agg(sum("Worldwide Collection  in Crores")\
        .alias("Total_WWC")).sort(desc("Total_WWC")).show(10)

+-------------------+------------------+
|           Director|         Total_WWC|
+-------------------+------------------+
|     S.S. Rajamouli|            3668.0|
|    Siddharth Anand|           2229.45|
|      Nitesh Tiwari|            2122.3|
|     Prashanth Neel|           2101.75|
|              Atlee|           1865.35|
|    Rajkumar Hirani|            1834.5|
|       Rohit Shetty|            1656.6|
|   Lokesh Kanagaraj|1350.1299999999999|
|Sandeep Reddy Vanga|            1342.0|
|         Kabir Khan|           1220.55|
+-------------------+------------------+
only showing top 10 rows


In [212]:
# With Spark SQL
spark.sql("Select Director, sum(`Worldwide Collection  in Crores`) as Total_WWC \
from boxf_tb boxf right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
group by Director \
order by Total_WWC desc limit 10").show()

+-------------------+------------------+
|           Director|         Total_WWC|
+-------------------+------------------+
|     S.S. Rajamouli|            3668.0|
|    Siddharth Anand|           2229.45|
|      Nitesh Tiwari|            2122.3|
|     Prashanth Neel|           2101.75|
|              Atlee|           1865.35|
|    Rajkumar Hirani|            1834.5|
|       Rohit Shetty|            1656.6|
|   Lokesh Kanagaraj|1350.1299999999999|
|Sandeep Reddy Vanga|            1342.0|
|         Kabir Khan|           1220.55|
+-------------------+------------------+



#### 52 Write a query to get director wise first day world wide collections? 

In [213]:
# With Pandas
p_boxf.merge(p_dir,left_on = "DirectorID",right_on="Director ID",how="right")\
    .groupby("Director").agg(Total_WWC=("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending = False).reset_index().head(10)

,Director,Total_WWC
0,S.S. Rajamouli,3668.00
1,Siddharth Anand,2229.45
2,Nitesh Tiwari,2122.30
3,Prashanth Neel,2101.75
4,Atlee,1865.35
5,Rajkumar Hirani,1834.50
6,Rohit Shetty,1656.60
7,Lokesh Kanagaraj,1350.13
8,Sandeep Reddy Vanga,1342.00
9,Kabir Khan,1220.55


In [214]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director").agg(sum("Worldwide Collection  in Crores")\
        .alias("Total_WWC")).sort(desc("Total_WWC")).show(10)

+-------------------+------------------+
|           Director|         Total_WWC|
+-------------------+------------------+
|     S.S. Rajamouli|            3668.0|
|    Siddharth Anand|           2229.45|
|      Nitesh Tiwari|            2122.3|
|     Prashanth Neel|           2101.75|
|              Atlee|           1865.35|
|    Rajkumar Hirani|            1834.5|
|       Rohit Shetty|            1656.6|
|   Lokesh Kanagaraj|1350.1299999999999|
|Sandeep Reddy Vanga|            1342.0|
|         Kabir Khan|           1220.55|
+-------------------+------------------+
only showing top 10 rows


In [215]:
# With Spark SQL
spark.sql("Select Director, sum(`Worldwide Collection  in Crores`) as Total_WWC \
from boxf_tb boxf right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
group by Director \
order by Total_WWC desc limit 10").show()

+-------------------+------------------+
|           Director|         Total_WWC|
+-------------------+------------------+
|     S.S. Rajamouli|            3668.0|
|    Siddharth Anand|           2229.45|
|      Nitesh Tiwari|            2122.3|
|     Prashanth Neel|           2101.75|
|              Atlee|           1865.35|
|    Rajkumar Hirani|            1834.5|
|       Rohit Shetty|            1656.6|
|   Lokesh Kanagaraj|1350.1299999999999|
|Sandeep Reddy Vanga|            1342.0|
|         Kabir Khan|           1220.55|
+-------------------+------------------+



#### 53 Write a query to get director wise India gross collections? 

In [216]:
# With Pandas
p_boxf.merge(p_dir,left_on = "DirectorID",right_on="Director ID",how="right")\
    .groupby("Director").agg(Total_IGC=("India Gross Collection in Crores","sum"))\
    .sort_values("Total_IGC",ascending = False).reset_index().head(10)

,Director,Total_IGC
0,S.S. Rajamouli,2848.85
1,Prashanth Neel,1747.60
2,Siddharth Anand,1560.91
3,Atlee,1256.05
4,Rohit Shetty,1246.20
5,Rajkumar Hirani,1199.50
6,Sandeep Reddy Vanga,1028.40
7,Amar Kaushik,961.48
8,Lokesh Kanagaraj,956.88
9,Ali Abbas Zafar,856.79


In [217]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director").agg(sum("India Gross Collection in Crores")\
        .alias("Total_IGC")).sort(desc("Total_IGC")).show(10)

+-------------------+------------------+
|           Director|         Total_IGC|
+-------------------+------------------+
|     S.S. Rajamouli|           2848.85|
|     Prashanth Neel|            1747.6|
|    Siddharth Anand|1560.9099999999999|
|              Atlee|           1256.05|
|       Rohit Shetty|            1246.2|
|    Rajkumar Hirani|            1199.5|
|Sandeep Reddy Vanga|            1028.4|
|       Amar Kaushik|            961.48|
|   Lokesh Kanagaraj| 956.8799999999999|
|    Ali Abbas Zafar|            856.79|
+-------------------+------------------+
only showing top 10 rows


In [218]:
# With Spark SQL
spark.sql("Select Director, sum(`India Gross Collection in Crores`) as Total_IGC \
from boxf_tb boxf right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
group by Director \
order by Total_IGC desc limit 10").show()

+-------------------+------------------+
|           Director|         Total_IGC|
+-------------------+------------------+
|     S.S. Rajamouli|           2848.85|
|     Prashanth Neel|            1747.6|
|    Siddharth Anand|1560.9099999999999|
|              Atlee|           1256.05|
|       Rohit Shetty|            1246.2|
|    Rajkumar Hirani|            1199.5|
|Sandeep Reddy Vanga|            1028.4|
|       Amar Kaushik|            961.48|
|   Lokesh Kanagaraj| 956.8799999999999|
|    Ali Abbas Zafar|            856.79|
+-------------------+------------------+



#### 54 Write a query to get director wise overseas collections? 

In [219]:
# With Pandas
p_boxf.merge(p_dir,left_on = "DirectorID",right_on="Director ID",how="right")\
    .groupby("Director").agg(Total_OSC=("Overseas Collection in Crores","sum"))\
    .sort_values("Total_OSC",ascending = False).reset_index().head(10)

,Director,Total_OSC
0,Nitesh Tiwari,1535.30
1,Advait Chandan,892.40
2,Siddharth Anand,819.37
3,S.S. Rajamouli,819.15
4,Rajkumar Hirani,635.00
5,Atlee,609.30
6,Kabir Khan,549.05
7,Rohit Shetty,410.40
8,Lokesh Kanagaraj,393.25
9,Prashanth Neel,354.15


In [220]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director").agg(sum("Overseas Collection in Crores")\
        .alias("Total_OSC")).sort(desc("Total_OSC")).show(10)

+----------------+---------+
|        Director|Total_OSC|
+----------------+---------+
|   Nitesh Tiwari|   1535.3|
|  Advait Chandan|    892.4|
| Siddharth Anand|   819.37|
|  S.S. Rajamouli|   819.15|
| Rajkumar Hirani|    635.0|
|           Atlee|    609.3|
|      Kabir Khan|   549.05|
|    Rohit Shetty|    410.4|
|Lokesh Kanagaraj|   393.25|
|  Prashanth Neel|   354.15|
+----------------+---------+
only showing top 10 rows


In [221]:
# With Spark SQL
spark.sql("Select Director, sum(`Overseas Collection in Crores`) as Total_OSC \
from boxf_tb boxf right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
group by Director \
order by Total_OSC desc limit 10").show()

+----------------+---------+
|        Director|Total_OSC|
+----------------+---------+
|   Nitesh Tiwari|   1535.3|
|  Advait Chandan|    892.4|
| Siddharth Anand|   819.37|
|  S.S. Rajamouli|   819.15|
| Rajkumar Hirani|    635.0|
|           Atlee|    609.3|
|      Kabir Khan|   549.05|
|    Rohit Shetty|    410.4|
|Lokesh Kanagaraj|   393.25|
|  Prashanth Neel|   354.15|
+----------------+---------+



#### 55 Write a query to get director, lead actor/actress and number of films? 

In [222]:
# With Pandas
p_boxf.merge(p_dir,left_on="DirectorID",right_on="Director ID",how="right")\
    .groupby(["Director","Lead Actor/Actress"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Director,Lead Actor/Actress,Total_Films
0,Siva,Ajith Kumar,4
1,H. Vinoth,Ajith Kumar,3
2,Sabbir Khan,Tiger Shroff,3
3,Siddharth Anand,Hrithik Roshan,3
4,Atlee,Joseph Vijay,3
5,Ajay Devgn,Ajay Devgn,3
6,Prashanth Neel,Yash,2
7,"Mani Ratnam, Sruti Harihara Subramanian",Vikram,2
8,Priyadarshan,Mohanlal,2
9,Koratala Siva,N.T. Rama Rao Jr.,2


In [223]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director","Lead Actor/Actress").agg(count("FilmID")\
        .alias("Total_Films")).sort(desc("Total_Films")).show(10)

+-----------------+------------------+-----------+
|         Director|Lead Actor/Actress|Total_Films|
+-----------------+------------------+-----------+
|             Siva|       Ajith Kumar|          4|
|  Siddharth Anand|    Hrithik Roshan|          3|
|       Ajay Devgn|        Ajay Devgn|          3|
|      Sabbir Khan|      Tiger Shroff|          3|
|            Atlee|      Joseph Vijay|          3|
|        H. Vinoth|       Ajith Kumar|          3|
|Tinu Suresh Desai|      Akshay Kumar|          2|
|           Vysakh|         Mammootty|          2|
|  A.R. Murugadoss|      Joseph Vijay|          2|
|    Vivek Athreya|              Nani|          2|
+-----------------+------------------+-----------+
only showing top 10 rows


In [224]:
# With Spark SQL
spark.sql("select Director,`Lead Actor/Actress`, \
count(FilmID) as Total_Films from boxf_tb boxf \
right join dir_tb dir on dir.`Director ID` = boxf.DirectorID \
group by Director, `Lead Actor/Actress` \
order by Total_Films desc limit 10").show()

+-----------------+------------------+-----------+
|         Director|Lead Actor/Actress|Total_Films|
+-----------------+------------------+-----------+
|             Siva|       Ajith Kumar|          4|
|       Ajay Devgn|        Ajay Devgn|          3|
|      Sabbir Khan|      Tiger Shroff|          3|
|  Siddharth Anand|    Hrithik Roshan|          3|
|            Atlee|      Joseph Vijay|          3|
|        H. Vinoth|       Ajith Kumar|          3|
|      Vetrimaaran|           Dhanush|          2|
|Tinu Suresh Desai|      Akshay Kumar|          2|
|    Koratala Siva| N.T. Rama Rao Jr.|          2|
|  A.R. Murugadoss|      Joseph Vijay|          2|
+-----------------+------------------+-----------+



#### 56 Write a query to get films which is having budget on between 150 crores and 277 crores? 

In [225]:
# With Pandas
p_boxf[p_boxf["Budget in Crores"].between(150,277)][["Title","Budget in Crores"]]\
    .sort_values("Budget in Crores",ascending = False).reset_index().head(10)

,index,Title,Budget in Crores
0,484,Thugs of Hindostan,275.0
1,591,Bade Miyan Chote Miyan,250.0
2,77,Ponniyin Selvan: Part One,250.0
3,586,Fighter,250.0
4,574,Tiger 3,250.0
5,116,Leo,250.0
6,396,Bãhubali 2: The Conclusion,250.0
7,573,Pathaan,250.0
8,345,Indian 2,250.0
9,455,Devara: Part 1,250.0


In [226]:
# With PySpark
s_boxf[s_boxf["Budget in Crores"].between(150,277)]\
    [["Title","Budget in Crores"]].sort(desc("Budget in Crores"))\
    .show(10)

+--------------------+----------------+
|               Title|Budget in Crores|
+--------------------+----------------+
|  Thugs of Hindostan|           275.0|
|Ponniyin Selvan: ...|           250.0|
|                 Leo|           250.0|
|Ponniyin Selvan: ...|           250.0|
|            Indian 2|           250.0|
|Bãhubali 2: The C...|           250.0|
|      Devara: Part 1|           250.0|
|             Pathaan|           250.0|
|             Tiger 3|           250.0|
|             Fighter|           250.0|
+--------------------+----------------+
only showing top 10 rows


In [227]:
# With Spark SQL
spark.sql("select Title, `Budget in Crores` from boxf_tb \
where `Budget in Crores` between 150 and 277 \
order by `Budget in Crores` desc limit 10").show()

+--------------------+----------------+
|               Title|Budget in Crores|
+--------------------+----------------+
|  Thugs of Hindostan|           275.0|
|Ponniyin Selvan: ...|           250.0|
|                 Leo|           250.0|
|Ponniyin Selvan: ...|           250.0|
|            Indian 2|           250.0|
|Bãhubali 2: The C...|           250.0|
|      Devara: Part 1|           250.0|
|             Pathaan|           250.0|
|             Tiger 3|           250.0|
|             Fighter|           250.0|
+--------------------+----------------+



#### 57 Write a query to get director, week name wise films released? 

In [228]:
# With Pandas 
p_boxf.merge(p_dir, left_on ="DirectorID", right_on = "Director ID",how="right")\
    .groupby(["WeekDay","Director"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False).reset_index().head(10)

,WeekDay,Director,Total_Films
0,Friday,Rohit Shetty,6
1,Friday,Siva,4
2,Friday,Mohit Suri,4
3,Friday,Farhad Samji,3
4,Friday,Anil Ravipudi,3
5,Friday,Anees Bazmee,3
6,Friday,Meghna Gulzar,3
7,Friday,Remo D'Souza,3
8,Friday,Boyapati Srinu,3
9,Wednesday,Ali Abbas Zafar,3


In [229]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("WeekDay","Director").agg(count("FilmID").alias("Total_Films"))\
    .sort(desc("Total_Films")).show(10)

+-------+--------------+-----------+
|WeekDay|      Director|Total_Films|
+-------+--------------+-----------+
| Friday|  Rohit Shetty|          6|
| Friday|          Siva|          4|
| Friday|    Mohit Suri|          4|
| Friday| Anil Ravipudi|          3|
| Friday|   Sabbir Khan|          3|
| Friday|     H. Vinoth|          3|
| Friday|  P.S. Mithran|          3|
| Friday|    K.V. Anand|          3|
| Friday|Surender Reddy|          3|
| Friday|Boyapati Srinu|          3|
+-------+--------------+-----------+
only showing top 10 rows


In [230]:
s_boxf.createOrReplaceTempView("boxf_tb")

In [231]:
# With Spark SQL
spark.sql("select WeekDay, Director,count(FilmID)as Total_Films from \
boxf_tb boxf right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
group by WeekDay, Director \
order by Total_Films desc limit 10").show()

+-------+--------------+-----------+
|WeekDay|      Director|Total_Films|
+-------+--------------+-----------+
| Friday|  Rohit Shetty|          6|
| Friday|          Siva|          4|
| Friday|    Mohit Suri|          4|
| Friday|Surender Reddy|          3|
| Friday|   Sabbir Khan|          3|
| Friday|     H. Vinoth|          3|
| Friday|  P.S. Mithran|          3|
| Friday|Prashanth Neel|          3|
| Friday|    K.V. Anand|          3|
| Friday|  Anees Bazmee|          3|
+-------+--------------+-----------+



#### 58 Write a query to get OTT platofrm and director wise films released?

In [232]:
# With Pandas
p_boxf.merge(p_dir,left_on="DirectorID",right_on="Director ID",how="right")\
    .groupby(["OTT Platform","Director"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False).reset_index().head(10)

,OTT Platform,Director,Total_Films
0,Amazon Prime Video,Anil Ravipudi,5
1,Amazon Prime Video,Trivikram Srinivas,5
2,Amazon Prime Video,A.R. Murugadoss,4
3,Amazon Prime Video,K.S. Ravindra,4
4,Netflix,Rohit Shetty,4
5,Amazon Prime Video,Prashanth Neel,4
6,Amazon Prime Video,Siva,4
7,Amazon Prime Video,Sukumar,4
8,Amazon Prime Video,Vikram K. Kumar,4
9,Amazon Prime Video,Sundar C.,4


In [233]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("OTT Platform","Director").agg(count("FilmID").alias("Total_Films"))\
    .sort(desc("Total_Films")).show(10)

+------------------+------------------+-----------+
|      OTT Platform|          Director|Total_Films|
+------------------+------------------+-----------+
|Amazon Prime Video|     Anil Ravipudi|          5|
|Amazon Prime Video|Trivikram Srinivas|          5|
|           Netflix|      Rohit Shetty|          4|
|Amazon Prime Video|     K.S. Ravindra|          4|
|Amazon Prime Video|   Vikram K. Kumar|          4|
|Amazon Prime Video| Vamshi Paidipally|          4|
|Amazon Prime Video|    Prashanth Neel|          4|
|Amazon Prime Video|           Sukumar|          4|
|Amazon Prime Video|   A.R. Murugadoss|          4|
|Amazon Prime Video|              Siva|          4|
+------------------+------------------+-----------+
only showing top 10 rows


In [234]:
# with Spark SQL
spark.sql("select `OTT Platform`, Director , count(FilmID) as \
Total_Films from boxf_tb boxf right join dir_tb dir on \
dir.`Director ID` = boxf.DirectorID \
group by 1,2 \
order by 3 desc limit 10").show()

+------------------+------------------+-----------+
|      OTT Platform|          Director|Total_Films|
+------------------+------------------+-----------+
|Amazon Prime Video|     Anil Ravipudi|          5|
|Amazon Prime Video|Trivikram Srinivas|          5|
|Amazon Prime Video|     K.S. Ravindra|          4|
|Amazon Prime Video|    Prashanth Neel|          4|
|Amazon Prime Video| Vamshi Paidipally|          4|
|           Netflix|      Rohit Shetty|          4|
|Amazon Prime Video|   Vikram K. Kumar|          4|
|Amazon Prime Video|           Sukumar|          4|
|Amazon Prime Video|   A.R. Murugadoss|          4|
|Amazon Prime Video|              Siva|          4|
+------------------+------------------+-----------+



#### 59 Write a query to get director wise films released on Friday only?

In [235]:
# With Pandas 
p_boxf.merge(p_dir, left_on ="DirectorID", right_on = "Director ID",how="right")\
    .query("WeekDay == 'Friday'").groupby(["WeekDay","Director"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False).reset_index().head(10)

,WeekDay,Director,Total_Films
0,Friday,Rohit Shetty,6
1,Friday,Siva,4
2,Friday,Mohit Suri,4
3,Friday,Farhad Samji,3
4,Friday,Surender Reddy,3
5,Friday,Meghna Gulzar,3
6,Friday,Anil Ravipudi,3
7,Friday,S.S. Rajamouli,3
8,Friday,Boyapati Srinu,3
9,Friday,Remo D'Souza,3


In [236]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .filter(col("Weekday")=="Friday")\
    .groupby("WeekDay","Director").agg(count("FilmID").alias("Total_Films"))\
    .sort(desc("Total_Films")).show(10)

+-------+--------------+-----------+
|WeekDay|      Director|Total_Films|
+-------+--------------+-----------+
| Friday|  Rohit Shetty|          6|
| Friday|          Siva|          4|
| Friday|    Mohit Suri|          4|
| Friday| Anil Ravipudi|          3|
| Friday|   Sabbir Khan|          3|
| Friday|     H. Vinoth|          3|
| Friday|  P.S. Mithran|          3|
| Friday|    K.V. Anand|          3|
| Friday|  Anees Bazmee|          3|
| Friday|Boyapati Srinu|          3|
+-------+--------------+-----------+
only showing top 10 rows


In [237]:
# With Spark SQL
spark.sql("select WeekDay, Director,count(FilmID)as Total_Films from \
boxf_tb boxf right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
where Weekday ='Friday' \
group by WeekDay, Director \
order by Total_Films desc limit 10").show()

+-------+--------------+-----------+
|WeekDay|      Director|Total_Films|
+-------+--------------+-----------+
| Friday|  Rohit Shetty|          6|
| Friday|          Siva|          4|
| Friday|    Mohit Suri|          4|
| Friday|Surender Reddy|          3|
| Friday|   Sabbir Khan|          3|
| Friday|     H. Vinoth|          3|
| Friday|  P.S. Mithran|          3|
| Friday|Prashanth Neel|          3|
| Friday|    K.V. Anand|          3|
| Friday|  Anees Bazmee|          3|
+-------+--------------+-----------+



#### 60 Write a query to get films based on IMDb reating between 6.5 and 7.7?

In [238]:
# With Pandas
p_boxf[p_boxf["IMDb Rating"].between(6.5,7.7)][["Title","IMDb Rating"]]\
    .sort_values("Title",ascending = False).head(10)

,Title,IMDb Rating
307,Yennai Arindhaal,7.3
356,Yashoda,6.5
199,War,6.5
321,Viswasam,6.5
211,Virus,7.7
221,Virupaksha,7.2
546,Vikram Vedha,7.1
308,Vendhu Thanindhathu Kaadu,7.3
299,Velaikkaran,7.2
9,Veeram,6.5


In [239]:
# With PySpark
s_boxf.filter(col("IMDb Rating").isin(6.5,7.7))\
    .select("Title","IMDb Rating")\
    .sort(desc("Title")).show(10)

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|             Yashoda|        6.5|
|                 War|        6.5|
|            Viswasam|        6.5|
|               Virus|        7.7|
|              Veeram|        6.5|
|              Uppena|        6.5|
|                Unda|        7.7|
|     Thamizh Padam 2|        6.5|
|Swatantrya Veer S...|        7.7|
|                Star|        6.5|
+--------------------+-----------+
only showing top 10 rows


In [240]:
# With Spark SQL
spark.sql("select Title, `IMDb Rating` from boxf_tb \
where `IMDb Rating` between 6.5 and 7.7 \
order by Title desc").show()

+--------------------+-----------+
|               Title|IMDb Rating|
+--------------------+-----------+
|    Yennai Arindhaal|        7.3|
|             Yashoda|        6.5|
|                 War|        6.5|
|            Viswasam|        6.5|
|               Virus|        7.7|
|          Virupaksha|        7.2|
|        Vikram Vedha|        7.1|
|Vendhu Thanindhat...|        7.3|
|         Velaikkaran|        7.2|
|              Veeram|        6.5|
|Varshangalkku She...|        6.6|
|            Varathan|        7.5|
|  Varane Avashyamund|        6.9|
|         Vakeel Saab|        6.9|
|Vaazha: Biopic of...|        7.3|
|              Vaathi|        7.3|
|   VR (Vikrant Rona)|        6.9|
|             Uunchai|        7.0|
|              Uppena|        6.5|
|                Unda|        7.7|
+--------------------+-----------+
only showing top 20 rows


#### 61 Write a query to get director,films and IMDb ratings?

In [241]:
# With Pandas
p_boxf.merge(p_dir,left_on="DirectorID",right_on="Director ID",how="right")\
    [["Title","Director","IMDb Rating"]].sort_values("IMDb Rating",ascending = False)\
    .head(10)

,Title,Director,IMDb Rating
539,12th Fail,Vidhu Vinod Chopra,8.8
442,Rocketry: The Nambi Effect,Madhavan,8.7
423,777 Charlie,Kiranraj K,8.7
414,The Kashmir Files,Vivek Agnihotri,8.6
583,Kishkindha Kaandam,Dinjith Ayyathan,8.6
579,Maharaja,Nithilan Saminathan,8.5
425,Sita Ramam,Hanu Raghavapudi,8.5
458,Jersey,Gowtam Tinnanuri,8.5
265,Sachin,James Erskine,8.5
505,Kumbalangi Nights,Madhu C. Narayanan,8.5


In [242]:
# With PySpark
s_boxf.join(s_dir, s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .select("Title","Director","IMDb Rating").sort(desc("IMDb Rating"))\
    .show(10)

+--------------------+------------------+-----------+
|               Title|          Director|IMDb Rating|
+--------------------+------------------+-----------+
|           12th Fail|Vidhu Vinod Chopra|        8.8|
|Rocketry: The Nam...|          Madhavan|        8.7|
|         777 Charlie|        Kiranraj K|        8.7|
|   The Kashmir Files|   Vivek Agnihotri|        8.6|
|  Kishkindha Kaandam|  Dinjith Ayyathan|        8.6|
|         Meiyazhagan|     C. Prem Kumar|        8.5|
|                  96|     C. Prem Kumar|        8.5|
|              Sachin|     James Erskine|        8.5|
|          Sita Ramam|  Hanu Raghavapudi|        8.5|
|              Jersey|  Gowtam Tinnanuri|        8.5|
+--------------------+------------------+-----------+
only showing top 10 rows


In [243]:
# With Spark SQL
spark.sql("select Title, Director, `IMDb Rating` from boxf_tb boxf \
right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
order by `IMDb Rating` desc limit 10").show()

+--------------------+------------------+-----------+
|               Title|          Director|IMDb Rating|
+--------------------+------------------+-----------+
|           12th Fail|Vidhu Vinod Chopra|        8.8|
|         777 Charlie|        Kiranraj K|        8.7|
|Rocketry: The Nam...|          Madhavan|        8.7|
|  Kishkindha Kaandam|  Dinjith Ayyathan|        8.6|
|   The Kashmir Files|   Vivek Agnihotri|        8.6|
|              Jersey|  Gowtam Tinnanuri|        8.5|
|   Kumbalangi Nights|Madhu C. Narayanan|        8.5|
|          Sita Ramam|  Hanu Raghavapudi|        8.5|
|         Meiyazhagan|     C. Prem Kumar|        8.5|
|              Sachin|     James Erskine|        8.5|
+--------------------+------------------+-----------+



#### 62 Write a query to get films with highest budget based flop verdict?

In [244]:
# With Pandas 
p_boxf.query("Verdict =='Flop'")[["Title","Verdict","Budget in Crores"]]\
    .sort_values("Budget in Crores",ascending = False).head(10)

,Title,Verdict,Budget in Crores
484,Thugs of Hindostan,Flop,275.0
541,83,Flop,200.0
277,Annaatthe,Flop,180.0
301,Valimai,Flop,150.0
546,Vikram Vedha,Flop,150.0
468,Race 3,Flop,150.0
366,Spyder,Flop,120.0
280,Vivegam,Flop,120.0
491,Baaghi 3,Flop,100.0
403,Godfather,Flop,100.0


In [245]:
# With PySpark
s_boxf.filter(col("Verdict")=="Flop")\
    .select("Title","Verdict","Budget in Crores")\
    .sort(desc("Budget in Crores")).show(10)

+------------------+-------+----------------+
|             Title|Verdict|Budget in Crores|
+------------------+-------+----------------+
|Thugs of Hindostan|   Flop|           275.0|
|                83|   Flop|           200.0|
|         Annaatthe|   Flop|           180.0|
|           Valimai|   Flop|           150.0|
|      Vikram Vedha|   Flop|           150.0|
|            Race 3|   Flop|           150.0|
|           Vivegam|   Flop|           120.0|
|            Spyder|   Flop|           120.0|
|               Bro|   Flop|           100.0|
|          Baaghi 3|   Flop|           100.0|
+------------------+-------+----------------+
only showing top 10 rows


In [246]:
# With Spark SQL
spark.sql("select Title,Verdict,`Budget in Crores` from boxf_tb \
where Verdict ='Flop' \
order by `Budget in Crores` desc limit 10").show()

+------------------+-------+----------------+
|             Title|Verdict|Budget in Crores|
+------------------+-------+----------------+
|Thugs of Hindostan|   Flop|           275.0|
|                83|   Flop|           200.0|
|         Annaatthe|   Flop|           180.0|
|           Valimai|   Flop|           150.0|
|            Race 3|   Flop|           150.0|
|      Vikram Vedha|   Flop|           150.0|
|            Spyder|   Flop|           120.0|
|           Vivegam|   Flop|           120.0|
|               Bro|   Flop|           100.0|
|         Godfather|   Flop|           100.0|
+------------------+-------+----------------+



#### 63 Write a query to get total number of directors?

In [247]:
# With Pandas
Total_directors = p_dir["Director"].nunique()
print("Total Directors: ",Total_directors)

Total Directors:  384


In [248]:
# With PySpark
s_dir.select("Director").distinct().count()

384

In [249]:
# With Spark SQL
spark.sql("select distinct(count(Director)) as Total_Directors from dir_tb").show()

+---------------+
|Total_Directors|
+---------------+
|            384|
+---------------+



#### 64 Write a query to get vedridct wise total films released? 

In [250]:
# With Pandas
p_boxf.groupby("Verdict").agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False).reset_index()

,Verdict,Total_Films
0,Blockbuster,166
1,Hit,96
2,SuperHit,89
3,Disaster,68
4,Flop,57
5,BelowAverage,47
6,Average,42
7,AboveAverage,23
8,AllTimeBlockbuster,16


In [251]:
# With PySpark
s_boxf.groupby("Verdict").agg(count("FilmID")\
    .alias("Total_Films")).sort(desc("Total_Films")).show()

+--------------------+-----------+
|             Verdict|Total_Films|
+--------------------+-----------+
|         Blockbuster|        166|
|                 Hit|         96|
|           Super Hit|         89|
|            Disaster|         68|
|                Flop|         57|
|       Below Average|         47|
|             Average|         42|
|       Above Average|         23|
|All Time Blockbuster|         16|
+--------------------+-----------+



In [252]:
# With Spark SQL
spark.sql("select Verdict, count(FilmID) as Total_Films \
from boxf_tb \
group by Verdict \
order by Total_Films desc").show()

+--------------------+-----------+
|             Verdict|Total_Films|
+--------------------+-----------+
|         Blockbuster|        166|
|                 Hit|         96|
|           Super Hit|         89|
|            Disaster|         68|
|                Flop|         57|
|       Below Average|         47|
|             Average|         42|
|       Above Average|         23|
|All Time Blockbuster|         16|
+--------------------+-----------+



#### 65 Write a query to get top 10 directors based number of films?

In [253]:
# With Pandas
p_boxf.merge(p_dir,left_on = "DirectorID",right_on = "Director ID", how = "right")\
    .groupby("Director").agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending = False)\
    .reset_index().head(10)

,Director,Total_Films
0,Trivikram Srinivas,6
1,Rohit Shetty,6
2,A.R. Murugadoss,5
3,Siva,5
4,Anil Ravipudi,5
5,Boyapati Srinu,5
6,Koratala Siva,5
7,Remo D'Souza,4
8,Atlee,4
9,Siddharth Anand,4


In [254]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director").agg(count("FilmID").alias("Total_Films"))\
    .sort(desc("Total_Films")).show(10)

+------------------+-----------+
|          Director|Total_Films|
+------------------+-----------+
|Trivikram Srinivas|          6|
|      Rohit Shetty|          6|
|     Anil Ravipudi|          5|
|     Koratala Siva|          5|
|   A.R. Murugadoss|          5|
|    Boyapati Srinu|          5|
|              Siva|          5|
|    Maruthi Dasari|          4|
|         Sundar C.|          4|
|   Siddharth Anand|          4|
+------------------+-----------+
only showing top 10 rows


In [255]:
# With PySpark
spark.sql("select Director, count(FilmID) as Total_Films \
from boxf_tb boxf \
right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
group by Director \
order by Total_Films desc limit 10").show()

+------------------+-----------+
|          Director|Total_Films|
+------------------+-----------+
|Trivikram Srinivas|          6|
|      Rohit Shetty|          6|
|     Anil Ravipudi|          5|
|     Koratala Siva|          5|
|   A.R. Murugadoss|          5|
|    Boyapati Srinu|          5|
|              Siva|          5|
|    Maruthi Dasari|          4|
|  Lokesh Kanagaraj|          4|
|   Siddharth Anand|          4|
+------------------+-----------+



#### 66 Write a query to get top 5 directors based on world wide collections and also industry wise? 

In [256]:
# With Pandas
p_boxf.merge(p_dir,left_on= "DirectorID",right_on = "Director ID",how="inner")\
    .groupby(["Director","Industry"]).agg(Total_WWC=("Worldwide Collection  in Crores","sum"))\
    .sort_values('Total_WWC',ascending = False).assign(rnk=lambda x:x\
        .groupby("Industry").cumcount()+1).query("rnk == 1").reset_index().head(5)

,Director,Industry,Total_WWC,rnk
0,S.S. Rajamouli,Tollywood,3668.00,1
1,Siddharth Anand,Bollywood,2229.45,1
2,Prashanth Neel,Sandalwood,1484.00,1
3,Lokesh Kanagaraj,Kollywood,1350.13,1
4,Vysakh,Mollywood,300.65,1


In [257]:
# With PySpark
window_spec = Window.partitionBy("Industry").orderBy(desc("Total_WWC"))
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .groupby("Director","Industry").agg(sum("Worldwide Collection  in Crores")\
        .alias("Total_WWC")).withColumn("rnk",row_number().over(window_spec))\
    .sort(desc("Total_WWC")).filter(col("rnk")==1).show()

+----------------+----------+------------------+---+
|        Director|  Industry|         Total_WWC|rnk|
+----------------+----------+------------------+---+
|  S.S. Rajamouli| Tollywood|            3668.0|  1|
| Siddharth Anand| Bollywood|           2229.45|  1|
|  Prashanth Neel|Sandalwood|            1484.0|  1|
|Lokesh Kanagaraj| Kollywood|1350.1299999999999|  1|
|          Vysakh| Mollywood|            300.65|  1|
+----------------+----------+------------------+---+



In [258]:
# With Spark SQL
spark.sql("select * from (select Director,Industry,Total_WWC \
,row_number() over(partition by Industry order by Total_WWC desc) as rnk \
from (select Director, Industry, sum(`Worldwide Collection  in Crores`) as Total_WWC \
from boxf_tb boxf right join dir_tb dir on dir.`Director Id`=boxf.DirectorID \
group by Director, Industry)a )b \
where rnk = 1 ").show()

+----------------+----------+------------------+---+
|        Director|  Industry|         Total_WWC|rnk|
+----------------+----------+------------------+---+
| Siddharth Anand| Bollywood|           2229.45|  1|
|Lokesh Kanagaraj| Kollywood|1350.1299999999999|  1|
|          Vysakh| Mollywood|            300.65|  1|
|  Prashanth Neel|Sandalwood|            1484.0|  1|
|  S.S. Rajamouli| Tollywood|            3668.0|  1|
+----------------+----------+------------------+---+



#### 67 Write a query to get top 3 directors based on India gross collections in Bollowood industry? 

In [259]:
# With Pandas 
p_boxf.merge(p_dir, left_on = "DirectorID",right_on = "Director ID", how="right")\
    .query("Industry=='Bollywood'").groupby(["Director","Industry"])\
    .agg(Total_IGC= ("India Gross Collection in Crores","sum"))\
    .sort_values("Total_IGC",ascending = False)\
    .assign(rnk=lambda x:x.groupby("Industry").cumcount()+1)\
    .query("rnk <=3").reset_index()

,Director,Industry,Total_IGC,rnk
0,Siddharth Anand,Bollywood,1560.91,1
1,Rohit Shetty,Bollywood,1246.20,2
2,Rajkumar Hirani,Bollywood,1199.50,3


In [260]:
# PySpark
window_spec = Window.partitionBy("Industry").orderBy(desc("Total_IGC"))
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .filter(col("Industry")=="Bollywood").groupby("Director","Industry")\
    .agg(sum("India Gross Collection in Crores").alias("Total_IGC"))\
    .withColumn("rnk",row_number().over(window_spec)).sort(desc("Total_IGC"))\
    .filter(col("rnk")<=3).show()

+---------------+---------+------------------+---+
|       Director| Industry|         Total_IGC|rnk|
+---------------+---------+------------------+---+
|Siddharth Anand|Bollywood|1560.9099999999999|  1|
|   Rohit Shetty|Bollywood|            1246.2|  2|
|Rajkumar Hirani|Bollywood|            1199.5|  3|
+---------------+---------+------------------+---+



In [261]:
# With Spark SQL
spark.sql("select * from (select Director, Industry, Total_IGC, \
row_number() over(partition by Industry order by Total_IGC desc) as rnk from \
(select Director,Industry,sum(`India Gross Collection in Crores`) as Total_IGC from \
boxf_tb boxf right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
where Industry = 'Bollywood' \
group by Director, Industry)a)b where rnk <=3").show()

+---------------+---------+------------------+---+
|       Director| Industry|         Total_IGC|rnk|
+---------------+---------+------------------+---+
|Siddharth Anand|Bollywood|1560.9099999999999|  1|
|   Rohit Shetty|Bollywood|            1246.2|  2|
|Rajkumar Hirani|Bollywood|            1199.5|  3|
+---------------+---------+------------------+---+



#### 68 Write a query to get top 3 directors based on India gross collections in Tollywood industry?

In [262]:
# Pandas
p_boxf.merge(p_dir,left_on="DirectorID",right_on="Director ID",how="right")\
    .query("Industry =='Tollywood'").groupby(["Director","Industry"])\
    .agg(Total_IGC=("India Gross Collection in Crores","sum"))\
    .sort_values("Total_IGC",ascending = False)\
    .assign(rnk = lambda x:x.groupby("Industry").cumcount()+1)\
    .query("rnk <=3").reset_index()

,Director,Industry,Total_IGC,rnk
0,S.S. Rajamouli,Tollywood,2848.85,1
1,Nag Ashwin,Tollywood,840.25,2
2,Trivikram Srinivas,Tollywood,711.65,3


In [263]:
# With PySpark
window_spec = Window.partitionBy("Industry").orderBy(desc("Total_IGC"))
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .filter(col("Industry")=="Tollywood").groupby("Director","Industry")\
    .agg(sum("India Gross Collection in Crores").alias("Total_IGC"))\
    .sort(desc("Total_IGC")).withColumn("rnk",row_number().over(window_spec))\
    .filter(col("rnk")<=3).show()

+------------------+---------+-----------------+---+
|          Director| Industry|        Total_IGC|rnk|
+------------------+---------+-----------------+---+
|    S.S. Rajamouli|Tollywood|          2848.85|  1|
|        Nag Ashwin|Tollywood|           840.25|  2|
|Trivikram Srinivas|Tollywood|711.6500000000001|  3|
+------------------+---------+-----------------+---+



In [264]:
# With Spark SQL  
spark.sql("Select * from (select Director, Industry, Total_IGC,\
row_number() over(partition by Industry order by Total_IGC desc) as rnk from \
(select Director, Industry, sum(`India Gross Collection in Crores`) as Total_IGC \
from boxf_tb boxf right join dir_tb dir on dir.`Director ID`==boxf.DirectorID \
where Industry = 'Tollywood' \
group by Industry, Director)a) b \
where rnk <= 3").show()


+------------------+---------+-----------------+---+
|          Director| Industry|        Total_IGC|rnk|
+------------------+---------+-----------------+---+
|    S.S. Rajamouli|Tollywood|          2848.85|  1|
|        Nag Ashwin|Tollywood|           840.25|  2|
|Trivikram Srinivas|Tollywood|711.6500000000001|  3|
+------------------+---------+-----------------+---+



#### 69 Write a query to get top 3 directors based on India gross collections in Kollowood industry? 

In [265]:
# With Pandas
p_boxf.merge(p_dir,left_on="DirectorID",right_on="Director ID",how="right")\
    .query("Industry == 'Kollywood'").groupby(["Director","Industry"])\
    .agg(Total_IGC=("India Gross Collection in Crores","sum"))\
    .sort_values("Total_IGC",ascending = False)\
    .assign(rnk= lambda x:x.groupby("Industry").cumcount()+1)\
    .query("rnk <=3").reset_index()

,Director,Industry,Total_IGC,rnk
0,Lokesh Kanagaraj,Kollywood,956.88,1
1,S. Shankar,Kollywood,826.83,2
2,Nelson Dilipkumar,Kollywood,669.88,3


In [266]:
# With PySpark
window_spec = Window.partitionBy("Industry").orderBy(desc("Total_IGC"))
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .filter(col("Industry")=="Kollywood").groupby("Director","Industry")\
    .agg(sum("India Gross Collection in Crores").alias("Total_IGC"))\
    .sort(desc("Total_IGC")).withColumn("rnk",row_number().over(window_spec))\
    .filter(col("rnk")<=3).show()

+-----------------+---------+---------+---+
|         Director| Industry|Total_IGC|rnk|
+-----------------+---------+---------+---+
| Lokesh Kanagaraj|Kollywood|   956.88|  1|
|       S. Shankar|Kollywood|   826.83|  2|
|Nelson Dilipkumar|Kollywood|   669.88|  3|
+-----------------+---------+---------+---+



In [267]:
# With Spark SQL
spark.sql("select * from (select Director, Industry, Total_IGC,\
row_number() over(partition by Industry order by Total_IGC desc) as rnk from \
(select Director, Industry, sum(`India Gross Collection in Crores`) as Total_IGC \
from boxf_tb boxf right join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
where Industry = 'Kollywood' \
group by Director, Industry)a) b \
where rnk <=3").show()

+-----------------+---------+---------+---+
|         Director| Industry|Total_IGC|rnk|
+-----------------+---------+---------+---+
| Lokesh Kanagaraj|Kollywood|   956.88|  1|
|       S. Shankar|Kollywood|   826.83|  2|
|Nelson Dilipkumar|Kollywood|   669.88|  3|
+-----------------+---------+---------+---+



#### 70 Write a query to get top 3 directors based on India gross collections in Mollowood industry? 

In [268]:
# With Pandas
p_boxf.merge(p_dir,left_on = "DirectorID", right_on = "Director ID", how="right")\
    .query("Industry == 'Mollywood'").groupby(["Director","Industry"])\
    .agg(Total_IGC=("India Gross Collection in Crores","sum"))\
    .sort_values("Total_IGC", ascending = False)\
    .assign(rnk=lambda x:x.groupby("Industry").cumcount()+1)\
    .query("rnk <=3").reset_index()

,Director,Industry,Total_IGC,rnk
0,Vysakh,Mollywood,208.55,1
1,Chidambaram,Mollywood,167.65,2
2,Jithu Madhavan,Mollywood,144.49,3


In [269]:
# With PySpark
window_spec = Window.partitionBy("Industry").orderBy(desc("Total_IGC"))
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"right")\
    .filter(col("Industry")=="Mollywood").groupby("Director","Industry")\
    .agg(sum("India Gross Collection in Crores").alias("Total_IGC"))\
    .sort(desc("Total_IGC")).withColumn("rnk",row_number().over(window_spec))\
    .filter(col("rnk")<=3).show()

+--------------+---------+---------+---+
|      Director| Industry|Total_IGC|rnk|
+--------------+---------+---------+---+
|        Vysakh|Mollywood|   208.55|  1|
|   Chidambaram|Mollywood|   167.65|  2|
|Jithu Madhavan|Mollywood|   144.49|  3|
+--------------+---------+---------+---+



In [270]:
# With Spark SQL
spark.sql("select * from (select Director, Industry, Total_IGC, \
row_number()over(partition by Industry order by Total_IGC desc) as rnk from \
(select Director, Industry,sum(`India Gross Collection in Crores`)as Total_IGC \
from boxf_tb boxf right join dir_tb dir on dir.`Director Id`=boxf.DirectorID \
where Industry = 'Mollywood' \
group by Director, Industry)a)b \
where rnk <=3").show()

+--------------+---------+---------+---+
|      Director| Industry|Total_IGC|rnk|
+--------------+---------+---------+---+
|        Vysakh|Mollywood|   208.55|  1|
|   Chidambaram|Mollywood|   167.65|  2|
|Jithu Madhavan|Mollywood|   144.49|  3|
+--------------+---------+---------+---+



#### 71 Write a query to get top 3 directors based on India gross collections in Sandalwood industry? 

In [271]:
# With Pandas
p_boxf.merge(p_dir,left_on = "DirectorID",right_on = "Director ID",how="right")\
    .query("Industry == 'Sandalwood'").groupby(["Director","Industry"])\
    .agg(Total_IGC = ("India Gross Collection in Crores","sum"))\
    .sort_values("Total_IGC",ascending = False)\
    .assign(rnk=lambda x:x.groupby("Industry").cumcount()+1)\
    .query("rnk<=3").reset_index()

,Director,Industry,Total_IGC,rnk
0,Prashanth Neel,Sandalwood,1259.85,1
1,Rishab Shetty,Sandalwood,363.82,2
2,Santhosh Ananddram,Sandalwood,164.40,3


In [272]:
# With PySpark
window_spec = Window.partitionBy("Industry").orderBy(desc("Total_IGC"))
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"inner")\
    .filter(col("Industry")=="Sandalwood").groupby("Director","Industry")\
    .agg(sum("India Gross Collection in Crores").alias("Total_IGC"))\
    .withColumn("rnk",row_number().over(window_spec))\
    .sort(desc("Total_IGC")).filter(col("rnk")<=3).show()

+------------------+----------+---------+---+
|          Director|  Industry|Total_IGC|rnk|
+------------------+----------+---------+---+
|    Prashanth Neel|Sandalwood|  1259.85|  1|
|     Rishab Shetty|Sandalwood|   363.82|  2|
|Santhosh Ananddram|Sandalwood|    164.4|  3|
+------------------+----------+---------+---+



In [273]:
# with Spark SQL
spark.sql("select * from (select Director,Industry,Total_IGC,\
row_number() over(partition by Industry order by Total_IGC desc)as rnk from \
(select Director, Industry, sum(`India Gross Collection in Crores`) as Total_IGC \
from boxf_tb boxf right join dir_tb dir on dir.`Director ID` = boxf.DirectorID \
where Industry ='Sandalwood' \
group by Industry, Director )a )b \
where rnk <=3").show()

+------------------+----------+---------+---+
|          Director|  Industry|Total_IGC|rnk|
+------------------+----------+---------+---+
|    Prashanth Neel|Sandalwood|  1259.85|  1|
|     Rishab Shetty|Sandalwood|   363.82|  2|
|Santhosh Ananddram|Sandalwood|    164.4|  3|
+------------------+----------+---------+---+



In [274]:
p_boxf.columns

Index(['FilmID', 'Title', 'Release Date', 'DirectorID', 'Lead Actor/Actress',
       'LanguageID', 'Industry', 'GenreID', 'Budget in Crores',
       'First Day Collection Worldwide  in Crores',
       'Worldwide Collection  in Crores', 'Overseas Collection in Crores',
       'India Gross Collection in Crores', 'Verdict', 'IMDb Rating',
       'Runtime (mins)', 'OTT Platform', 'Year', 'WeekDay'],
      dtype='object')

#### 72 Write to get total number of generes?

In [275]:
# With Pandas
p_gen["Genre"].nunique()

159

In [276]:
# With PySpark
s_gen.select("Genre").distinct().count()

159

In [277]:
# With Spark SQL
spark.sql("select Distinct(count(Genre)) Total_Genres from gen_tb ").show()

+------------+
|Total_Genres|
+------------+
|         159|
+------------+



#### 73 Write query to get director,language,genere films count? 

In [278]:
# With Pandas
p_boxf.merge(p_dir, left_on="DirectorID",right_on="Director ID",how="inner")\
    .merge(p_lan,on ="LanguageID",how="inner")\
    .merge(p_gen,on="GenreID",how="inner")\
    .groupby(["Director","Language","Genre"]).agg(Total_Films = ("FilmID","count"))\
    .sort_values("Total_Films",ascending = False).reset_index().head(10)

,Director,Language,Genre,Total_Films
0,Lokesh Kanagaraj,Tamil,"Action, Crime, Drama, Thriller",4
1,Trivikram Srinivas,Telugu,"Action, Drama",4
2,Prashanth Neel,Kannada,"Action, Crime, Drama, Thriller",3
3,Boyapati Srinu,Telugu,"Action, Drama",3
4,Santhosh Ananddram,Kannada,"Action, Drama",2
5,Sreenu Vaitla,Telugu,"Action, Comedy",2
6,Ajay Devgn,Hindi,"Action, Adventure, Crime, Drama, Thriller",2
7,Siddharth Anand,Hindi,"Action, Adventure, Thriller",2
8,Vetrimaaran,Tamil,"Action, Crime, Drama, Thriller",2
9,Amar Kaushik,Hindi,"Comedy, Horror",2


In [279]:
# With PySpark
s_boxf.join(s_dir,s_boxf["DirectorID"]==s_dir["Director ID"],"inner")\
    .join(s_lan,s_boxf["LanguageID"]==s_lan["LanguageID"],"inner")\
    .join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"inner")\
    .groupby("Director","Language","Genre").agg(count("FilmID")\
        .alias("Total_Films")).sort(desc("Total_Films")).show(10)

+------------------+--------+--------------------+-----------+
|          Director|Language|               Genre|Total_Films|
+------------------+--------+--------------------+-----------+
|  Lokesh Kanagaraj|   Tamil|Action, Crime, Dr...|          4|
|Trivikram Srinivas|  Telugu|       Action, Drama|          4|
|    Boyapati Srinu|  Telugu|       Action, Drama|          3|
|    Prashanth Neel| Kannada|Action, Crime, Dr...|          3|
|     Aanand L. Rai|   Hindi|Comedy, Drama, Ro...|          2|
|      Venky Atluri|  Telugu|             Romance|          2|
|Gopichand Malineni|  Telugu|       Action, Drama|          2|
|        Luv Ranjan|   Hindi|     Comedy, Romance|          2|
|   Siddharth Anand|   Hindi|Action, Adventure...|          2|
|        Ajay Devgn|   Hindi|Action, Adventure...|          2|
+------------------+--------+--------------------+-----------+
only showing top 10 rows


In [280]:
# With Spark SQL
spark.sql("select Director,Language,Genre, \
count(FilmID)as Total_Films from boxf_tb boxf \
inner join dir_tb dir on dir.`Director ID`=boxf.DirectorID \
inner join lan_tb lan on lan.LanguageID =boxf.LanguageID \
inner join gen_tb gen on gen.GenreID = boxf.GenreID \
group by Director, Language, Genre \
order by Total_Films desc limit 10").show()

+------------------+--------+--------------------+-----------+
|          Director|Language|               Genre|Total_Films|
+------------------+--------+--------------------+-----------+
|  Lokesh Kanagaraj|   Tamil|Action, Crime, Dr...|          4|
|Trivikram Srinivas|  Telugu|       Action, Drama|          4|
|    Boyapati Srinu|  Telugu|       Action, Drama|          3|
|    Prashanth Neel| Kannada|Action, Crime, Dr...|          3|
|    S.S. Rajamouli|  Telugu|       Action, Drama|          2|
|      Venky Atluri|  Telugu|             Romance|          2|
|Gopichand Malineni|  Telugu|       Action, Drama|          2|
|        Luv Ranjan|   Hindi|     Comedy, Romance|          2|
|   Siddharth Anand|   Hindi|Action, Adventure...|          2|
|              Siva|   Tamil|       Action, Drama|          2|
+------------------+--------+--------------------+-----------+



#### 74 Write a query to genere wise budget? 

In [281]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    .groupby("Genre").agg(Total_Budget=("Budget in Crores","sum"))\
    .sort_values("Total_Budget",ascending=False).reset_index().head(10)

,Genre,Total_Budget
0,"Action, Drama",3113.0
1,"Action, Crime, Drama, Thriller",2937.0
2,"Action, Crime, Thriller",1560.0
3,"Action, Drama, Thriller",1457.5
4,"Comedy, Drama, Romance",1308.5
5,"Action, Adventure, Drama",1299.0
6,"Action, Thriller",1277.0
7,"Action, Adventure, Thriller",1100.0
8,"Action, Comedy, Drama",915.0
9,"Action, Crime, Drama",880.0


In [282]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .groupby("Genre").agg(sum("Budget in Crores").alias("Total_Budget"))\
    .sort(desc("Total_Budget")).show(10)

+--------------------+------------+
|               Genre|Total_Budget|
+--------------------+------------+
|       Action, Drama|      3113.0|
|Action, Crime, Dr...|      2937.0|
|Action, Crime, Th...|      1560.0|
|Action, Drama, Th...|      1457.5|
|Comedy, Drama, Ro...|      1308.5|
|Action, Adventure...|      1299.0|
|    Action, Thriller|      1277.0|
|Action, Adventure...|      1100.0|
|Action, Comedy, D...|       915.0|
|Action, Crime, Drama|       880.0|
+--------------------+------------+
only showing top 10 rows


In [283]:
# With Spark SQL
spark.sql("Select Genre, sum(`Budget in Crores`)as Total_Budget \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
group by Genre \
order by 2 desc limit 10").show()

+--------------------+------------+
|               Genre|Total_Budget|
+--------------------+------------+
|       Action, Drama|      3113.0|
|Action, Crime, Dr...|      2937.0|
|Action, Crime, Th...|      1560.0|
|Action, Drama, Th...|      1457.5|
|Comedy, Drama, Ro...|      1308.5|
|Action, Adventure...|      1299.0|
|    Action, Thriller|      1277.0|
|Action, Adventure...|      1100.0|
|Action, Comedy, D...|       915.0|
|Action, Crime, Drama|       880.0|
+--------------------+------------+



#### 75 Write a query to get genere wise first day worldwide collections?

In [284]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    .groupby("Genre").agg(Total_WWC=("Worldwide Collection  in Crores","sum"))\
    .sort_values("Total_WWC",ascending=False).reset_index().head(10)

,Genre,Total_WWC
0,"Action, Crime, Drama, Thriller",8155.23
1,"Action, Drama",7298.85
2,"Comedy, Drama, Romance",3057.85
3,"Action, Adventure, Thriller",2871.72
4,"Action, Drama, Thriller",2676.40
5,"Action, Crime, Thriller",2499.66
6,"Action, Adventure, Drama",2331.95
7,"Comedy, Drama",2134.81
8,"Action, Biography, Drama, Sport",2122.30
9,"Action, Thriller",1958.75


In [285]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .groupby("Genre").agg(sum("Worldwide Collection  in Crores")\
        .alias("Total_WWC")).sort(desc("Total_WWC")).show(10)

+--------------------+------------------+
|               Genre|         Total_WWC|
+--------------------+------------------+
|Action, Crime, Dr...| 8155.229999999999|
|       Action, Drama| 7298.849999999999|
|Comedy, Drama, Ro...|           3057.85|
|Action, Adventure...|2871.7200000000003|
|Action, Drama, Th...|2676.3999999999996|
|Action, Crime, Th...|           2499.66|
|Action, Adventure...|           2331.95|
|       Comedy, Drama|           2134.81|
|Action, Biography...|            2122.3|
|    Action, Thriller|1958.7500000000002|
+--------------------+------------------+
only showing top 10 rows


In [286]:
# With Spark SQL
spark.sql("Select Genre, sum(`Worldwide Collection  in Crores`)as Total_WWC \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
group by Genre \
order by 2 desc limit 10").show()

+--------------------+------------------+
|               Genre|         Total_WWC|
+--------------------+------------------+
|Action, Crime, Dr...| 8155.229999999999|
|       Action, Drama| 7298.849999999999|
|Comedy, Drama, Ro...|           3057.85|
|Action, Adventure...|2871.7200000000003|
|Action, Drama, Th...|2676.3999999999996|
|Action, Crime, Th...|           2499.66|
|Action, Adventure...|           2331.95|
|       Comedy, Drama|           2134.81|
|Action, Biography...|            2122.3|
|    Action, Thriller|1958.7500000000002|
+--------------------+------------------+



#### 76 Write a query to get genere wise overseas collections?

In [287]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    .groupby("Genre").agg(Total_OSC=("Overseas Collection in Crores","sum"))\
    .sort_values("Total_OSC",ascending=False).reset_index().head(10)

,Genre,Total_OSC
0,"Action, Crime, Drama, Thriller",2043.13
1,"Action, Drama",1598.18
2,"Action, Biography, Drama, Sport",1535.30
3,"Comedy, Drama",872.96
4,"Drama, Music",846.40
5,"Action, Adventure, Thriller",791.74
6,"Comedy, Drama, Romance",723.85
7,"Action, Adventure, Drama",700.46
8,"Action, Drama, Thriller",667.55
9,"Action, Crime, Thriller",541.90


In [288]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .groupby("Genre").agg(sum("Overseas Collection in Crores")\
        .alias("Total_OSC")).sort(desc("Total_OSC")).show(10)

+--------------------+------------------+
|               Genre|         Total_OSC|
+--------------------+------------------+
|Action, Crime, Dr...|           2043.13|
|       Action, Drama|1598.1799999999998|
|Action, Biography...|            1535.3|
|       Comedy, Drama|            872.96|
|        Drama, Music|             846.4|
|Action, Adventure...|            791.74|
|Comedy, Drama, Ro...|            723.85|
|Action, Adventure...|            700.46|
|Action, Drama, Th...|            667.55|
|Action, Crime, Th...| 541.9000000000001|
+--------------------+------------------+
only showing top 10 rows


In [289]:
# With Spark SQL
spark.sql("Select Genre, sum(`Overseas Collection in Crores`)as Total_OSC \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
group by Genre \
order by 2 desc limit 10").show()

+--------------------+------------------+
|               Genre|         Total_OSC|
+--------------------+------------------+
|Action, Crime, Dr...|           2043.13|
|       Action, Drama|1598.1799999999998|
|Action, Biography...|            1535.3|
|       Comedy, Drama|            872.96|
|        Drama, Music|             846.4|
|Action, Adventure...|            791.74|
|Comedy, Drama, Ro...|            723.85|
|Action, Adventure...|            700.46|
|Action, Drama, Th...|            667.55|
|Action, Crime, Th...| 541.9000000000001|
+--------------------+------------------+



#### 77 Write a query to get genere wise India gross collections?

In [290]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    .groupby("Genre").agg(Total_IGC=("India Gross Collection in Crores","sum"))\
    .sort_values("Total_IGC",ascending=False).reset_index().head(10)

,Genre,Total_IGC
0,"Action, Crime, Drama, Thriller",6212.32
1,"Action, Drama",5851.50
2,"Comedy, Drama, Romance",2334.30
3,"Action, Adventure, Thriller",2079.98
4,"Action, Drama, Thriller",2044.35
5,"Action, Crime, Thriller",1957.76
6,"Action, Adventure, Drama",1665.80
7,"Action, Comedy, Drama",1498.85
8,"Action, Thriller",1494.40
9,"Action, Crime, Drama",1466.80


In [291]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .groupby("Genre").agg(sum("India Gross Collection in Crores")\
        .alias("Total_IGC")).sort(desc("Total_IGC")).show(10)

+--------------------+------------------+
|               Genre|         Total_IGC|
+--------------------+------------------+
|Action, Crime, Dr...|           6212.32|
|       Action, Drama|            5851.5|
|Comedy, Drama, Ro...|            2334.3|
|Action, Adventure...|           2079.98|
|Action, Drama, Th...|2044.3500000000001|
|Action, Crime, Th...|1957.7599999999998|
|Action, Adventure...|1665.8000000000002|
|Action, Comedy, D...|1498.8500000000001|
|    Action, Thriller|            1494.4|
|Action, Crime, Drama|            1466.8|
+--------------------+------------------+
only showing top 10 rows


In [292]:
# With Spark SQL
spark.sql("Select Genre, sum(`India Gross Collection in Crores`)as Total_IGC \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
group by Genre \
order by 2 desc limit 10").show()

+--------------------+------------------+
|               Genre|         Total_IGC|
+--------------------+------------------+
|Action, Crime, Dr...|           6212.32|
|       Action, Drama|            5851.5|
|Comedy, Drama, Ro...|            2334.3|
|Action, Adventure...|           2079.98|
|Action, Drama, Th...|2044.3500000000001|
|Action, Crime, Th...|1957.7599999999998|
|Action, Adventure...|1665.8000000000002|
|Action, Comedy, D...|1498.8500000000001|
|    Action, Thriller|            1494.4|
|Action, Crime, Drama|            1466.8|
+--------------------+------------------+



#### 78 Write a query to get genere wise top 2 longest run time movies?

In [293]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    [["Title","Genre","Runtime (mins)"]]\
    .sort_values("Runtime (mins)",ascending=False)\
    .reset_index().head(2)

,index,Title,Genre,Runtime (mins)
0,230,Animal,"Action, Crime, Drama, Thriller",204
1,463,I,"Action, Drama, Romance, Sci-Fi, Thriller",188


In [294]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .select("Title","Genre","Runtime (mins)")\
    .sort(desc("Runtime (mins)")).show(2)

+------+--------------------+--------------+
| Title|               Genre|Runtime (mins)|
+------+--------------------+--------------+
|Animal|Action, Crime, Dr...|           204|
|     I|Action, Drama, Ro...|           188|
+------+--------------------+--------------+
only showing top 2 rows


In [295]:
# With Spark SQL
spark.sql("Select Title,Genre,`Runtime (mins)` \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
order by 3 desc limit 2").show()

+------+--------------------+--------------+
| Title|               Genre|Runtime (mins)|
+------+--------------------+--------------+
|Animal|Action, Crime, Dr...|           204|
|     I|Action, Drama, Ro...|           188|
+------+--------------------+--------------+



#### 79 Write a query to get genere wise bottom shortest runtime movies? 

In [296]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    [["Title","Genre","Runtime (mins)"]]\
    .sort_values("Runtime (mins)",ascending=True)\
    .reset_index().head(2)

,index,Title,Genre,Runtime (mins)
0,257,Kill,"Action, Crime, Drama, Thriller",105
1,535,Raksha Bandhan,"Comedy, Drama, Family",108


In [297]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .select("Title","Genre","Runtime (mins)")\
    .sort(asc("Runtime (mins)")).show(2)

+--------------+--------------------+--------------+
|         Title|               Genre|Runtime (mins)|
+--------------+--------------------+--------------+
|          Kill|Action, Crime, Dr...|           105|
|Raksha Bandhan|Comedy, Drama, Fa...|           108|
+--------------+--------------------+--------------+
only showing top 2 rows


In [298]:
# With Spark SQL
spark.sql("Select Title,Genre,`Runtime (mins)` \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
order by 3 asc limit 2").show()

+--------------+--------------------+--------------+
|         Title|               Genre|Runtime (mins)|
+--------------+--------------------+--------------+
|          Kill|Action, Crime, Dr...|           105|
|Raksha Bandhan|Comedy, Drama, Fa...|           108|
+--------------+--------------------+--------------+



#### 80 Write a query to get verdict, genere wise films released? 

In [299]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    .groupby(["Verdict","Genre"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Verdict,Genre,Total_Films
0,Blockbuster,"Action, Crime, Drama, Thriller",13
1,Blockbuster,"Comedy, Drama, Romance",9
2,Blockbuster,"Comedy, Romance",9
3,Hit,"Action, Drama",8
4,Blockbuster,"Action, Drama",8
5,Blockbuster,Comedy,8
6,SuperHit,"Comedy, Drama",7
7,SuperHit,"Action, Drama, Thriller",7
8,BelowAverage,"Action, Drama",7
9,SuperHit,"Action, Drama",6


In [300]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .groupby("Verdict","Genre").agg(count("FilmID")\
        .alias("Total_Films")).sort("Total_Films",ascending = False).show(10)

+-------------+--------------------+-----------+
|      Verdict|               Genre|Total_Films|
+-------------+--------------------+-----------+
|  Blockbuster|Action, Crime, Dr...|         13|
|  Blockbuster|Comedy, Drama, Ro...|          9|
|  Blockbuster|     Comedy, Romance|          9|
|          Hit|       Action, Drama|          8|
|  Blockbuster|       Action, Drama|          8|
|  Blockbuster|              Comedy|          8|
|    Super Hit|       Comedy, Drama|          7|
|    Super Hit|Action, Drama, Th...|          7|
|Below Average|       Action, Drama|          7|
|  Blockbuster|Action, Drama, Th...|          6|
+-------------+--------------------+-----------+
only showing top 10 rows


In [301]:
# With Spark SQL
spark.sql("Select Verdict,Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
group by Genre,Verdict \
order by 3 desc limit 10").show()

+-------------+--------------------+-----------+
|      Verdict|               Genre|Total_Films|
+-------------+--------------------+-----------+
|  Blockbuster|Action, Crime, Dr...|         13|
|  Blockbuster|     Comedy, Romance|          9|
|  Blockbuster|Comedy, Drama, Ro...|          9|
|  Blockbuster|              Comedy|          8|
|          Hit|       Action, Drama|          8|
|  Blockbuster|       Action, Drama|          8|
|    Super Hit|Action, Drama, Th...|          7|
|    Super Hit|       Comedy, Drama|          7|
|Below Average|       Action, Drama|          7|
|          Hit|Action, Crime, Dr...|          6|
+-------------+--------------------+-----------+



#### 81 Write a query to get genere, OTT platform wise films count?

In [302]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    .groupby(["OTT Platform","Genre"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,OTT Platform,Genre,Total_Films
0,Amazon Prime Video,"Action, Drama",36
1,Amazon Prime Video,"Action, Drama, Thriller",23
2,Amazon Prime Video,"Action, Crime, Drama, Thriller",20
3,Amazon Prime Video,"Action, Thriller",17
4,Amazon Prime Video,"Action, Crime, Thriller",16
5,Amazon Prime Video,"Comedy, Drama",13
6,Amazon Prime Video,"Comedy, Drama, Romance",12
7,Amazon Prime Video,"Action, Comedy",11
8,Netflix,"Comedy, Drama, Romance",11
9,Amazon Prime Video,Comedy,10


In [303]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .groupby("OTT Platform","Genre").agg(count("FilmID")\
        .alias("Total_Films")).sort("Total_Films",ascending = False).show(10)

+------------------+--------------------+-----------+
|      OTT Platform|               Genre|Total_Films|
+------------------+--------------------+-----------+
|Amazon Prime Video|       Action, Drama|         36|
|Amazon Prime Video|Action, Drama, Th...|         23|
|Amazon Prime Video|Action, Crime, Dr...|         20|
|Amazon Prime Video|    Action, Thriller|         17|
|Amazon Prime Video|Action, Crime, Th...|         16|
|Amazon Prime Video|       Comedy, Drama|         13|
|Amazon Prime Video|Comedy, Drama, Ro...|         12|
|           Netflix|Comedy, Drama, Ro...|         11|
|Amazon Prime Video|      Action, Comedy|         11|
|Amazon Prime Video|              Comedy|         10|
+------------------+--------------------+-----------+
only showing top 10 rows


In [304]:
# With Spark SQL
spark.sql("Select `OTT Platform`,Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
group by Genre,`OTT platform` \
order by 3 desc limit 10").show()

+------------------+--------------------+-----------+
|      OTT Platform|               Genre|Total_Films|
+------------------+--------------------+-----------+
|Amazon Prime Video|       Action, Drama|         36|
|Amazon Prime Video|Action, Drama, Th...|         23|
|Amazon Prime Video|Action, Crime, Dr...|         20|
|Amazon Prime Video|    Action, Thriller|         17|
|Amazon Prime Video|Action, Crime, Th...|         16|
|Amazon Prime Video|       Comedy, Drama|         13|
|Amazon Prime Video|Comedy, Drama, Ro...|         12|
|           Netflix|Comedy, Drama, Ro...|         11|
|Amazon Prime Video|      Action, Comedy|         11|
|Amazon Prime Video|              Comedy|         10|
+------------------+--------------------+-----------+



#### 82 Write a query to get genere wise films count? 

In [305]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    .groupby("Genre").agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Genre,Total_Films
0,"Action, Drama",44
1,"Action, Crime, Drama, Thriller",36
2,"Comedy, Drama, Romance",29
3,"Action, Drama, Thriller",28
4,"Action, Crime, Thriller",25
5,"Comedy, Drama",22
6,"Action, Thriller",19
7,"Comedy, Romance",17
8,"Action, Comedy, Drama",16
9,"Action, Comedy",12


In [306]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .groupby("Genre").agg(count("FilmID")\
        .alias("Total_Films")).sort("Total_Films",ascending = False).show(10)

+--------------------+-----------+
|               Genre|Total_Films|
+--------------------+-----------+
|       Action, Drama|         44|
|Action, Crime, Dr...|         36|
|Comedy, Drama, Ro...|         29|
|Action, Drama, Th...|         28|
|Action, Crime, Th...|         25|
|       Comedy, Drama|         22|
|    Action, Thriller|         19|
|     Comedy, Romance|         17|
|Action, Comedy, D...|         16|
|      Drama, Romance|         12|
+--------------------+-----------+
only showing top 10 rows


In [307]:
# With Spark SQL
spark.sql("Select Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
group by Genre \
order by 2 desc limit 10").show()

+--------------------+-----------+
|               Genre|Total_Films|
+--------------------+-----------+
|       Action, Drama|         44|
|Action, Crime, Dr...|         36|
|Comedy, Drama, Ro...|         29|
|Action, Drama, Th...|         28|
|Action, Crime, Th...|         25|
|       Comedy, Drama|         22|
|    Action, Thriller|         19|
|     Comedy, Romance|         17|
|Action, Comedy, D...|         16|
|      Drama, Romance|         12|
+--------------------+-----------+



#### 83 Write a query to get genere wise films count in Tollowood Industry? 

In [308]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right").query("Industry =='Tollywood'")\
    .groupby(["Industry","Genre"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Industry,Genre,Total_Films
0,Tollywood,"Action, Drama",19
1,Tollywood,"Action, Crime, Drama, Thriller",9
2,Tollywood,"Comedy, Drama, Romance",8
3,Tollywood,"Action, Comedy",8
4,Tollywood,"Action, Drama, Thriller",8
5,Tollywood,"Action, Thriller",7
6,Tollywood,"Drama, Romance",6
7,Tollywood,"Action, Comedy, Drama",6
8,Tollywood,"Action, Drama, Romance",5
9,Tollywood,"Action, Crime, Thriller",4


In [309]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .filter(col("Industry")=="Tollywood").groupby("Industry","Genre")\
    .agg(count("FilmID").alias("Total_Films"))\
    .sort("Total_Films",ascending = False).show(10)

+---------+--------------------+-----------+
| Industry|               Genre|Total_Films|
+---------+--------------------+-----------+
|Tollywood|       Action, Drama|         19|
|Tollywood|Action, Crime, Dr...|          9|
|Tollywood|Action, Drama, Th...|          8|
|Tollywood|      Action, Comedy|          8|
|Tollywood|Comedy, Drama, Ro...|          8|
|Tollywood|    Action, Thriller|          7|
|Tollywood|Action, Comedy, D...|          6|
|Tollywood|      Drama, Romance|          6|
|Tollywood|Action, Drama, Ro...|          5|
|Tollywood|Action, Crime, Th...|          4|
+---------+--------------------+-----------+
only showing top 10 rows


In [310]:
# With Spark SQL
spark.sql("Select Industry,Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
where Industry = 'Tollywood'\
group by Industry,Genre \
order by 3 desc limit 10").show()

+---------+--------------------+-----------+
| Industry|               Genre|Total_Films|
+---------+--------------------+-----------+
|Tollywood|       Action, Drama|         19|
|Tollywood|Action, Crime, Dr...|          9|
|Tollywood|Action, Drama, Th...|          8|
|Tollywood|      Action, Comedy|          8|
|Tollywood|Comedy, Drama, Ro...|          8|
|Tollywood|    Action, Thriller|          7|
|Tollywood|Action, Comedy, D...|          6|
|Tollywood|      Drama, Romance|          6|
|Tollywood|Action, Drama, Ro...|          5|
|Tollywood|               Drama|          4|
+---------+--------------------+-----------+



#### 84 Write a query to get genere wise films count in Kollowood Industry?

In [311]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right").query("Industry =='Kollywood'")\
    .groupby(["Industry","Genre"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Industry,Genre,Total_Films
0,Kollywood,"Action, Drama",17
1,Kollywood,"Action, Crime, Drama, Thriller",14
2,Kollywood,"Action, Crime, Thriller",13
3,Kollywood,"Action, Drama, Thriller",8
4,Kollywood,"Action, Thriller",7
5,Kollywood,"Action, Crime, Drama",6
6,Kollywood,"Action, Comedy, Drama",5
7,Kollywood,"Comedy, Horror",4
8,Kollywood,Action,3
9,Kollywood,"Comedy, Drama",3


In [312]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .filter(col("Industry")=="Kollywood").groupby("Industry","Genre")\
    .agg(count("FilmID").alias("Total_Films"))\
    .sort("Total_Films",ascending = False).show(10)

+---------+--------------------+-----------+
| Industry|               Genre|Total_Films|
+---------+--------------------+-----------+
|Kollywood|       Action, Drama|         17|
|Kollywood|Action, Crime, Dr...|         14|
|Kollywood|Action, Crime, Th...|         13|
|Kollywood|Action, Drama, Th...|          8|
|Kollywood|    Action, Thriller|          7|
|Kollywood|Action, Crime, Drama|          6|
|Kollywood|Action, Comedy, D...|          5|
|Kollywood|      Comedy, Horror|          4|
|Kollywood|              Action|          3|
|Kollywood|              Comedy|          3|
+---------+--------------------+-----------+
only showing top 10 rows


In [313]:
# With Spark SQL
spark.sql("Select Industry,Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
where Industry = 'Kollywood'\
group by Industry,Genre \
order by 3 desc limit 10").show()

+---------+--------------------+-----------+
| Industry|               Genre|Total_Films|
+---------+--------------------+-----------+
|Kollywood|       Action, Drama|         17|
|Kollywood|Action, Crime, Dr...|         14|
|Kollywood|Action, Crime, Th...|         13|
|Kollywood|Action, Drama, Th...|          8|
|Kollywood|    Action, Thriller|          7|
|Kollywood|Action, Crime, Drama|          6|
|Kollywood|Action, Comedy, D...|          5|
|Kollywood|      Comedy, Horror|          4|
|Kollywood|              Action|          3|
|Kollywood|       Comedy, Drama|          3|
+---------+--------------------+-----------+



#### 85 Write a query to get genere wise films count in Mollowood Industry? 

In [314]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right").query("Industry =='Mollywood'")\
    .groupby(["Industry","Genre"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Industry,Genre,Total_Films
0,Mollywood,"Action, Drama, Thriller",6
1,Mollywood,"Comedy, Drama, Romance",6
2,Mollywood,"Comedy, Drama",5
3,Mollywood,"Drama, Thriller",4
4,Mollywood,"Comedy, Romance",4
5,Mollywood,"Action, Crime, Drama, Thriller",4
6,Mollywood,Comedy,3
7,Mollywood,Drama,2
8,Mollywood,"Crime, Drama, Mystery, Thriller",2
9,Mollywood,"Action, Thriller",2


In [315]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .filter(col("Industry")=="Mollywood").groupby("Industry","Genre")\
    .agg(count("FilmID").alias("Total_Films"))\
    .sort("Total_Films",ascending = False).show(10)

+---------+--------------------+-----------+
| Industry|               Genre|Total_Films|
+---------+--------------------+-----------+
|Mollywood|Action, Drama, Th...|          6|
|Mollywood|Comedy, Drama, Ro...|          6|
|Mollywood|       Comedy, Drama|          5|
|Mollywood|     Comedy, Romance|          4|
|Mollywood|     Drama, Thriller|          4|
|Mollywood|Action, Crime, Dr...|          4|
|Mollywood|              Comedy|          3|
|Mollywood|               Drama|          2|
|Mollywood|Crime, Drama, Mys...|          2|
|Mollywood|Action, Comedy, T...|          2|
+---------+--------------------+-----------+
only showing top 10 rows


In [316]:
# With Spark SQL
spark.sql("Select Industry,Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
where Industry = 'Mollywood'\
group by Industry,Genre \
order by 3 desc limit 10").show()

+---------+--------------------+-----------+
| Industry|               Genre|Total_Films|
+---------+--------------------+-----------+
|Mollywood|Action, Drama, Th...|          6|
|Mollywood|Comedy, Drama, Ro...|          6|
|Mollywood|       Comedy, Drama|          5|
|Mollywood|     Comedy, Romance|          4|
|Mollywood|     Drama, Thriller|          4|
|Mollywood|Action, Crime, Dr...|          4|
|Mollywood|              Comedy|          3|
|Mollywood|Action, Drama, Hi...|          2|
|Mollywood|Horror, Mystery, ...|          2|
|Mollywood|Action, Comedy, T...|          2|
+---------+--------------------+-----------+



#### 86 Write a query to get genere wise films count in Bollowood Industry? 

In [317]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right").query("Industry =='Bollywood'")\
    .groupby(["Industry","Genre"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Industry,Genre,Total_Films
0,Bollywood,"Comedy, Drama",12
1,Bollywood,"Comedy, Drama, Romance",12
2,Bollywood,"Comedy, Romance",8
3,Bollywood,"Action, Adventure, Thriller",7
4,Bollywood,"Biography, Drama",6
5,Bollywood,"Action, Crime, Drama, Thriller",6
6,Bollywood,"Action, Drama, Thriller",5
7,Bollywood,"Action, Crime, Thriller",4
8,Bollywood,"Action, Crime, Drama",4
9,Bollywood,"Drama, Romance",4


In [318]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .filter(col("Industry")=="Bollywood").groupby("Industry","Genre")\
    .agg(count("FilmID").alias("Total_Films"))\
    .sort("Total_Films",ascending = False).show(10)

+---------+--------------------+-----------+
| Industry|               Genre|Total_Films|
+---------+--------------------+-----------+
|Bollywood|Comedy, Drama, Ro...|         12|
|Bollywood|       Comedy, Drama|         12|
|Bollywood|     Comedy, Romance|          8|
|Bollywood|Action, Adventure...|          7|
|Bollywood|    Biography, Drama|          6|
|Bollywood|Action, Crime, Dr...|          6|
|Bollywood|Action, Drama, Th...|          5|
|Bollywood|      Drama, Romance|          4|
|Bollywood|Action, Crime, Th...|          4|
|Bollywood|      Comedy, Horror|          4|
+---------+--------------------+-----------+
only showing top 10 rows


In [319]:
# With Spark SQL
spark.sql("Select Industry,Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
where Industry = 'Bollywood'\
group by Industry,Genre \
order by 3 desc limit 10").show()

+---------+--------------------+-----------+
| Industry|               Genre|Total_Films|
+---------+--------------------+-----------+
|Bollywood|Comedy, Drama, Ro...|         12|
|Bollywood|       Comedy, Drama|         12|
|Bollywood|     Comedy, Romance|          8|
|Bollywood|Action, Adventure...|          7|
|Bollywood|    Biography, Drama|          6|
|Bollywood|Action, Crime, Dr...|          6|
|Bollywood|Action, Drama, Th...|          5|
|Bollywood|Action, Crime, Th...|          4|
|Bollywood|      Drama, Romance|          4|
|Bollywood|      Comedy, Horror|          4|
+---------+--------------------+-----------+



#### 87 Write a query to get genere wise films count in Sandalwood Industry? 

In [320]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right").query("Industry =='Sandalwood'")\
    .groupby(["Industry","Genre"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Industry,Genre,Total_Films
0,Sandalwood,"Action, Drama",5
1,Sandalwood,Action,4
2,Sandalwood,"Action, Crime",3
3,Sandalwood,"Action, Crime, Drama, Thriller",3
4,Sandalwood,"Action, Crime, Thriller",2
5,Sandalwood,"Action, Drama, Romance",2
6,Sandalwood,"Action, Thriller",2
7,Sandalwood,"Action, Adventure, Drama, Thriller",1
8,Sandalwood,"Action, Comedy, Drama",1
9,Sandalwood,"Action, Comedy, Musical",1


In [321]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .filter(col("Industry")=="Sandalwood").groupby("Industry","Genre")\
    .agg(count("FilmID").alias("Total_Films"))\
    .sort("Total_Films",ascending = False).show(10)

+----------+--------------------+-----------+
|  Industry|               Genre|Total_Films|
+----------+--------------------+-----------+
|Sandalwood|       Action, Drama|          5|
|Sandalwood|              Action|          4|
|Sandalwood|Action, Crime, Dr...|          3|
|Sandalwood|       Action, Crime|          3|
|Sandalwood|Action, Crime, Th...|          2|
|Sandalwood|    Action, Thriller|          2|
|Sandalwood|Action, Drama, Ro...|          2|
|Sandalwood|Adventure, Comedy...|          1|
|Sandalwood|Action, Adventure...|          1|
|Sandalwood|Action, Drama, Sport|          1|
+----------+--------------------+-----------+
only showing top 10 rows


In [322]:
# With Spark SQL
spark.sql("Select Industry,Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
where Industry = 'Sandalwood'\
group by Industry,Genre \
order by 3 desc limit 10").show()

+----------+--------------------+-----------+
|  Industry|               Genre|Total_Films|
+----------+--------------------+-----------+
|Sandalwood|       Action, Drama|          5|
|Sandalwood|              Action|          4|
|Sandalwood|Action, Crime, Dr...|          3|
|Sandalwood|       Action, Crime|          3|
|Sandalwood|Action, Crime, Th...|          2|
|Sandalwood|    Action, Thriller|          2|
|Sandalwood|Action, Drama, Ro...|          2|
|Sandalwood|Adventure, Comedy...|          1|
|Sandalwood|Action, Adventure...|          1|
|Sandalwood|Action, Drama, Sport|          1|
+----------+--------------------+-----------+



#### 88 Write a query to get lead actors/actress wise,genere and films count? 

In [323]:
# With Pandas
p_boxf.merge(p_gen,on = "GenreID",how="right")\
    .groupby(["Lead Actor/Actress","Genre"]).agg(Total_Films=("FilmID","count"))\
    .sort_values("Total_Films",ascending=False).reset_index().head(10)

,Lead Actor/Actress,Genre,Total_Films
0,Dhanush,"Action, Drama",5
1,Mahesh Babu,"Action, Drama",4
2,N.T. Rama Rao Jr.,"Action, Drama, Thriller",3
3,Vijay Deverakonda,"Comedy, Drama, Romance",3
4,Ravi Teja,"Action, Crime, Drama, Thriller",3
5,Nani,"Comedy, Romance",3
6,Ajith Kumar,"Action, Crime, Drama, Thriller",3
7,Allu Arjun,"Action, Drama",3
8,Ayushmann Khurrana,"Comedy, Drama",2
9,Sivakarthikeyan,"Action, Drama, Thriller",2


In [324]:
# With PySpark
s_boxf.join(s_gen,s_boxf["GenreID"]==s_gen["GenreID"],"right")\
    .groupby("Lead Actor/Actress","Genre")\
    .agg(count("FilmID").alias("Total_Films"))\
    .sort("Total_Films",ascending = False).show(10)

+------------------+--------------------+-----------+
|Lead Actor/Actress|               Genre|Total_Films|
+------------------+--------------------+-----------+
|           Dhanush|       Action, Drama|          5|
|       Mahesh Babu|       Action, Drama|          4|
|        Allu Arjun|       Action, Drama|          3|
|              Nani|     Comedy, Romance|          3|
| N.T. Rama Rao Jr.|Action, Drama, Th...|          3|
|       Ajith Kumar|Action, Crime, Dr...|          3|
| Vijay Deverakonda|Comedy, Drama, Ro...|          3|
|         Ravi Teja|Action, Crime, Dr...|          3|
|           Prabhas|       Action, Drama|          2|
|         Mammootty|Action, Crime, Dr...|          2|
+------------------+--------------------+-----------+
only showing top 10 rows


In [325]:
# With Spark SQL
spark.sql("Select `Lead Actor/Actress`,Genre,count(FilmID)as Total_Films \
from boxf_tb boxf right join gen_tb gen on gen.GenreID = boxf.GenreID \
group by `Lead Actor/Actress`,Genre \
order by 3 desc limit 10").show()

+------------------+--------------------+-----------+
|Lead Actor/Actress|               Genre|Total_Films|
+------------------+--------------------+-----------+
|           Dhanush|       Action, Drama|          5|
|       Mahesh Babu|       Action, Drama|          4|
|        Allu Arjun|       Action, Drama|          3|
|              Nani|     Comedy, Romance|          3|
| N.T. Rama Rao Jr.|Action, Drama, Th...|          3|
|       Ajith Kumar|Action, Crime, Dr...|          3|
| Vijay Deverakonda|Comedy, Drama, Ro...|          3|
|         Ravi Teja|Action, Crime, Dr...|          3|
|           Prabhas|       Action, Drama|          2|
|            Vikram|Action, Adventure...|          2|
+------------------+--------------------+-----------+



#### 89 Write a query to get 5th rank movie based on Worldwide total collections? 

In [326]:
# With Pandas
p_boxf["rnk"]=p_boxf[["Worldwide Collection  in Crores"]].rank(method="dense",ascending=False)
p_boxf.query("rnk == 5")[["Title","Worldwide Collection  in Crores","rnk"]]

,Title,Worldwide Collection in Crores,rnk
572,Jawan,1160.0,5.0


In [327]:
# With PySpark
s_boxf = s_boxf.withColumn("rnk",dense_rank().over(Window.orderBy(desc("Worldwide Collection  in Crores"))))
s_boxf.filter(col("rnk")==5).select("Title","Worldwide Collection  in Crores","rnk").show()

+-----+-------------------------------+---+
|Title|Worldwide Collection  in Crores|rnk|
+-----+-------------------------------+---+
|Jawan|                         1160.0|  5|
+-----+-------------------------------+---+



In [328]:


# With Spark SQL
spark.sql("select * from(select Title,`Worldwide Collection  in Crores`, \
dense_rank() over(Order by `Worldwide Collection  in Crores` desc)as rnk from boxf_tb) \
where rnk = 5").show()

+-----+-------------------------------+---+
|Title|Worldwide Collection  in Crores|rnk|
+-----+-------------------------------+---+
|Jawan|                         1160.0|  5|
+-----+-------------------------------+---+



#### 90 Write a query to get 5th rank movie by industry wise based on First day worldwide collections?

In [329]:
p_boxf["rnk"]=p_boxf[["First Day Collection Worldwide  in Crores"]].rank(method="dense",ascending=False)
p_boxf.query("rnk==5")[["Title","First Day Collection Worldwide  in Crores","rnk"]]

,Title,First Day Collection Worldwide in Crores,rnk
212,Salaar: Part 1 - Ceasefire,158.1,5.0


In [330]:
# With PySpark
s_boxf=s_boxf.withColumn("rnk",dense_rank().over(Window.orderBy(desc("First Day Collection Worldwide  in Crores"))))
s_boxf.filter(col("rnk")==5).select("Title","First Day Collection Worldwide  in Crores","rnk").show()

+--------------------+-----------------------------------------+---+
|               Title|First Day Collection Worldwide  in Crores|rnk|
+--------------------+-----------------------------------------+---+
|Salaar: Part 1 - ...|                                    158.1|  5|
+--------------------+-----------------------------------------+---+



In [331]:
#With Spark SQL
spark.sql("select * from(select Title,`First Day Collection Worldwide  in Crores`,\
dense_rank() over(order by `First Day Collection Worldwide  in Crores` desc) as rnk \
from boxf_tb) \
where rnk = 5").show()

+--------------------+-----------------------------------------+---+
|               Title|First Day Collection Worldwide  in Crores|rnk|
+--------------------+-----------------------------------------+---+
|Salaar: Part 1 - ...|                                    158.1|  5|
+--------------------+-----------------------------------------+---+



#### 91 Write a query to get 3rd rank movie by industry wise based on IMDb Ratings? 

In [332]:
# With Pandas 
p_boxf[["Industry","Title","IMDb Rating"]]\
    .sort_values(["Industry","IMDb Rating"],ascending=[True,False])\
    .assign(rnk= lambda x:x.groupby("Industry").cumcount()+1)\
    .query("rnk ==3")

,Industry,Title,IMDb Rating,rnk
512,Bollywood,Sachin,8.5,3
135,Kollywood,Maharaja,8.5,3
147,Mollywood,Bangalore Days,8.3,3
149,Sandalwood,K.G.F: Chapter 2,8.2,3
35,Tollywood,Mahanati,8.4,3


In [333]:
# With PySpark
window_spec = Window.partitionBy("Industry").orderBy(desc("IMDb Rating"))
s_boxf.withColumn("rnk",row_number().over(window_spec))\
    .filter(col("rnk")==3).sort("Industry",desc("IMDb Rating"))\
    .select("Industry","Title","IMDb Rating","rnk").show()

+----------+----------------+-----------+---+
|  Industry|           Title|IMDb Rating|rnk|
+----------+----------------+-----------+---+
| Bollywood|          Sachin|        8.5|  3|
| Kollywood|        Maharaja|        8.5|  3|
| Mollywood|  Bangalore Days|        8.3|  3|
|Sandalwood|K.G.F: Chapter 2|        8.2|  3|
| Tollywood|        Mahanati|        8.4|  3|
+----------+----------------+-----------+---+



In [334]:
#with Spark SQL
spark.sql("select * from(select Industry,Title,`IMDb Rating`,\
row_number() over(partition by Industry order by `IMDb Rating` desc)as rnk \
from boxf_tb ) where rnk =3 ").show()

+----------+----------------+-----------+---+
|  Industry|           Title|IMDb Rating|rnk|
+----------+----------------+-----------+---+
| Bollywood|          Sachin|        8.5|  3|
| Kollywood|        Maharaja|        8.5|  3|
| Mollywood|  Bangalore Days|        8.3|  3|
|Sandalwood|K.G.F: Chapter 2|        8.2|  3|
| Tollywood|        Mahanati|        8.4|  3|
+----------+----------------+-----------+---+



#### 92 Calculate YoY% Budget growth? 

#### With Pandas

In [385]:
Yearly_Bud=p_boxf.groupby("Year")[["Budget in Crores"]].sum()\
    .sort_values("Year",ascending=True).reset_index()

In [386]:
Yearly_Bud["YoY_Growth"]=Yearly_Bud["Budget in Crores"].pct_change()*100

In [387]:
Yearly_Bud

,Year,Budget in Crores,YoY_Growth
0,2014,1820.0,NaN
1,2015,1906.0,4.725275
2,2016,2142.0,12.381952
3,2017,3291.0,53.641457
4,2018,3679.0,11.789730
5,2019,2544.0,-30.850775
6,2020,1387.0,-45.479560
7,2021,1855.0,33.741889
8,2022,5722.0,208.463612
9,2023,5877.0,2.708843


#### With PySpark

In [388]:
Yearly_Bud = s_boxf.groupby("Year").agg(sum("Budget in Crores").alias("Total_Budget"))

In [389]:
Yearly_Bud = Yearly_Bud.withColumn("PY_Budget",lag("Total_Budget")\
    .over(Window.orderBy("Year"))).withColumn("YoY%",round(((col("Total_Budget")-col("PY_Budget"))/col("PY_Budget")*100),2))

In [390]:
Yearly_Bud.show()

+----+------------+---------+------+
|Year|Total_Budget|PY_Budget|  YoY%|
+----+------------+---------+------+
|2014|      1820.0|     NULL|  NULL|
|2015|      1906.0|   1820.0|  4.73|
|2016|      2142.0|   1906.0| 12.38|
|2017|      3291.0|   2142.0| 53.64|
|2018|      3679.0|   3291.0| 11.79|
|2019|      2544.0|   3679.0|-30.85|
|2020|      1387.0|   2544.0|-45.48|
|2021|      1855.0|   1387.0| 33.74|
|2022|      5722.0|   1855.0|208.46|
|2023|      5877.0|   5722.0|  2.71|
|2024|      4430.0|   5877.0|-24.62|
+----+------------+---------+------+



#### With Spark SQL

In [391]:
spark.sql("select Year,Total_Bud,PY_Bud, round(((Total_Bud-PY_Bud)/PY_Bud)*100,2) YoY from \
(select Year, Total_Bud,lag(Total_Bud) over(order by Year) PY_Bud from \
(select Year, sum(`Budget in Crores`) as Total_Bud from boxf_tb \
group by Year)a)b").show()

+----+---------+------+------+
|Year|Total_Bud|PY_Bud|   YoY|
+----+---------+------+------+
|2014|   1820.0|  NULL|  NULL|
|2015|   1906.0|1820.0|  4.73|
|2016|   2142.0|1906.0| 12.38|
|2017|   3291.0|2142.0| 53.64|
|2018|   3679.0|3291.0| 11.79|
|2019|   2544.0|3679.0|-30.85|
|2020|   1387.0|2544.0|-45.48|
|2021|   1855.0|1387.0| 33.74|
|2022|   5722.0|1855.0|208.46|
|2023|   5877.0|5722.0|  2.71|
|2024|   4430.0|5877.0|-24.62|
+----+---------+------+------+



#### 93 Calculate YoY% Worldwide total collelctions growth? 

In [392]:
Yearly_WWC = p_boxf.groupby("Year")[["Worldwide Collection  in Crores"]].sum()\
    .sort_values("Year",ascending = True).reset_index()

In [393]:
Yearly_WWC["YoY%_Growth"]=Yearly_WWC["Worldwide Collection  in Crores"].pct_change()*100

In [394]:
Yearly_WWC

,Year,Worldwide Collection in Crores,YoY%_Growth
0,2014,4645.00,NaN
1,2015,5470.20,17.765339
2,2016,7088.80,29.589412
3,2017,9659.20,36.260016
4,2018,8275.30,-14.327273
5,2019,8242.72,-0.393702
6,2020,2348.75,-71.505158
7,2021,2711.65,15.450772
8,2022,9853.16,263.364003
9,2023,13461.72,36.623378


In [395]:
Yearly_WWC = s_boxf.groupby("Year").agg(sum("Worldwide Collection  in Crores")\
    .alias("Total_WWC"))

In [396]:
Yearly_WWC = Yearly_WWC.withColumn("PY_WWC",lag("Total_WWC")\
    .over(Window.orderBy("Year")))\
    .withColumn("YoY%",round(((col("Total_WWC")-col("PY_WWC"))/col("PY_WWC")*100),2))

In [397]:
Yearly_WWC.show()

+----+------------------+------------------+------+
|Year|         Total_WWC|            PY_WWC|  YoY%|
+----+------------------+------------------+------+
|2014| 4645.000000000001|              NULL|  NULL|
|2015| 5470.200000000001| 4645.000000000001| 17.77|
|2016| 7088.799999999999| 5470.200000000001| 29.59|
|2017|            9659.2| 7088.799999999999| 36.26|
|2018| 8275.300000000001|            9659.2|-14.33|
|2019|           8242.72| 8275.300000000001| -0.39|
|2020|           2348.75|           8242.72|-71.51|
|2021|2711.6499999999996|           2348.75| 15.45|
|2022| 9853.160000000002|2711.6499999999996|263.36|
|2023|13461.720000000001| 9853.160000000002| 36.62|
|2024|           8122.29|13461.720000000001|-39.66|
+----+------------------+------------------+------+



### With Spark SQL

In [398]:
spark.sql("select Year, Total_WWC, PY_WWC,round(((Total_WWC-PY_WWC)/PY_WWC)*100,2) YoY from \
(select Year, Total_WWC, lag(Total_WWC) over(order by Year) as PY_WWC from \
(select Year, sum(`Worldwide Collection  in Crores`) as Total_WWC from boxf_tb \
group by Year)a)b").show()

+----+------------------+------------------+------+
|Year|         Total_WWC|            PY_WWC|   YoY|
+----+------------------+------------------+------+
|2014| 4645.000000000001|              NULL|  NULL|
|2015| 5470.200000000001| 4645.000000000001| 17.77|
|2016| 7088.799999999999| 5470.200000000001| 29.59|
|2017|            9659.2| 7088.799999999999| 36.26|
|2018| 8275.300000000001|            9659.2|-14.33|
|2019|           8242.72| 8275.300000000001| -0.39|
|2020|           2348.75|           8242.72|-71.51|
|2021|2711.6499999999996|           2348.75| 15.45|
|2022| 9853.160000000002|2711.6499999999996|263.36|
|2023|13461.720000000001| 9853.160000000002| 36.62|
|2024|           8122.29|13461.720000000001|-39.66|
+----+------------------+------------------+------+



#### 94 Calculate YoY% Indian Gross colelctions growth?

In [399]:
Yearly_IGC = p_boxf.groupby("Year")[["India Gross Collection in Crores"]].sum()\
    .sort_values("Year",ascending = True).reset_index()

In [400]:
Yearly_IGC["YoY%_Growth"]=Yearly_IGC["India Gross Collection in Crores"].pct_change()*100

In [401]:
Yearly_IGC

,Year,India Gross Collection in Crores,YoY%_Growth
0,2014,3711.10,NaN
1,2015,3935.30,6.041335
2,2016,4459.90,13.330623
3,2017,6852.90,53.655912
4,2018,6327.40,-7.668286
5,2019,6655.58,5.186649
6,2020,1966.35,-70.455618
7,2021,2238.40,13.835279
8,2022,7716.21,244.719889
9,2023,10080.17,30.636284


#### With PySpark

In [402]:
Yearly_IGC = s_boxf.groupby("Year")\
    .agg(sum("India Gross Collection in Crores")\
        .alias("Total_IGC"))

In [403]:
Yearly_IGC = Yearly_IGC.withColumn("PY_IGC",lag("Total_IGC")\
    .over(Window.orderBy("Year")))\
    .withColumn("YoY%",round(((col("Total_IGC")-col("PY_IGC"))/col("PY_IGC")*100),2))

In [404]:
Yearly_IGC.show()

+----+------------------+------------------+------+
|Year|         Total_IGC|            PY_IGC|  YoY%|
+----+------------------+------------------+------+
|2014| 3711.099999999999|              NULL|  NULL|
|2015|3935.2999999999997| 3711.099999999999|  6.04|
|2016| 4459.900000000001|3935.2999999999997| 13.33|
|2017|6852.9000000000015| 4459.900000000001| 53.66|
|2018| 6327.400000000001|6852.9000000000015| -7.67|
|2019| 6655.580000000001| 6327.400000000001|  5.19|
|2020|           1966.35| 6655.580000000001|-70.46|
|2021|            2238.4|           1966.35| 13.84|
|2022| 7716.210000000003|            2238.4|244.72|
|2023|          10080.17| 7716.210000000003| 30.64|
|2024|           6120.79|          10080.17|-39.28|
+----+------------------+------------------+------+



#### With Spark SQL

In [405]:
spark.sql("select Year, Total_IGC, PY_IGC, round(((Total_IGC-PY_IGC)/PY_IGC)*100,2) YoY from \
(select Year, Total_IGC, lag(Total_IGC) over(order by Year) as PY_IGC from \
(select Year, sum(`India Gross Collection in Crores`) as Total_IGC from boxf_tb \
group by Year)a)b").show()

+----+------------------+------------------+------+
|Year|         Total_IGC|            PY_IGC|   YoY|
+----+------------------+------------------+------+
|2014| 3711.099999999999|              NULL|  NULL|
|2015|3935.2999999999997| 3711.099999999999|  6.04|
|2016| 4459.900000000001|3935.2999999999997| 13.33|
|2017|6852.9000000000015| 4459.900000000001| 53.66|
|2018| 6327.400000000001|6852.9000000000015| -7.67|
|2019| 6655.580000000001| 6327.400000000001|  5.19|
|2020|           1966.35| 6655.580000000001|-70.46|
|2021|            2238.4|           1966.35| 13.84|
|2022| 7716.210000000003|            2238.4|244.72|
|2023|          10080.17| 7716.210000000003| 30.64|
|2024|           6120.79|          10080.17|-39.28|
+----+------------------+------------------+------+

